# 04B_policy_field_reference_classification_bumsu

이 노트북은 기존 `04_policy_field_analysis_bumsu.ipynb`를 대체하거나 덮어쓰는 것이 아니라, 은세가 공유한 `사업 내용 분류.txt` 기준을 반영하여 최종 분석용 `사업내용분류_*` 변수를 새로 생성하기 위한 보조 노트북이다.

기존 04번 v3c 결과는 사업명 토큰과 반복 패턴을 기반으로 한 탐색적 자동분류 초안으로 보존한다.  
04B에서는 05번 분석에 투입할 최종 정책분야 변수로 `사업내용분류_*` 열을 생성한다.

작업 중 원본 열은 덮어쓰지 않고, 새 열만 생성한다.  
사업명만으로 판단이 어려운 사례는 `기타`, `검토필요`, `복합`으로 남긴다.

04B-1. 라이브러리 불러오기 및 경로 설정
1. 이 단계의 목적

04B 노트북에서 사용할 라이브러리를 불러오고, 입력 파일·참고 기준 파일·출력 폴더 경로를 설정합니다.

특히 이번 04B는 기존 04번 출력 폴더와 섞이지 않게 아래 새 폴더를 사용합니다.

04_v2_사업내용분류분석

In [1]:
# 04B-1. 라이브러리 불러오기 및 경로 설정

# ------------------------------------------------------------
# 0) Google Drive 마운트
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")


# ------------------------------------------------------------
# 1) 기본 라이브러리 불러오기
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter, defaultdict

import os
import re
import json
import ast
import shutil

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2) 실행 timestamp 설정
# ------------------------------------------------------------
# 저장 파일명이 덮어써지지 않도록 현재 시각을 파일명에 사용한다.

RUN_TS = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M")

print(f"RUN_TS: {RUN_TS}")


# ------------------------------------------------------------
# 3) 프로젝트 기본 경로 설정
# ------------------------------------------------------------
# 필요하면 BASE_DIR만 본인 Drive 구조에 맞게 수정하면 된다.

BASE_DIR = Path("/content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA")

GUIDE_OUTPUT_DIR = BASE_DIR / "[guide & output] 분석 협업 매뉴얼 및 Figure 후보"


# ------------------------------------------------------------
# 4) 04B 입력 파일 경로 설정
# ------------------------------------------------------------
# 03번 보강 완료 후 새로 생성된 최신 master parquet 파일을 사용한다.

INPUT_PATH = (
    GUIDE_OUTPUT_DIR
    / "03_중간결과표"
    / "03_v1_사업명텍스트분석"
    / "tables"
    / "final"
    / "03_project_name_text_analysis_v1_20260529_0135.parquet"
)


# ------------------------------------------------------------
# 5) 사업 내용 분류 기준 파일 경로 설정
# ------------------------------------------------------------
# 은세가 공유한 기준 파일이다.
# 실제 Drive 위치가 다르면 이 경로만 수정하면 된다.

REFERENCE_PATH = (
    BASE_DIR
    / "Reference Resources"
    / "[텍스트 분석] 참고 자료"
    / "04 Notebook - 사업 내용 유형"
    / "사업 내용 분류.txt"
)


# ------------------------------------------------------------
# 6) 04B 출력 폴더 설정
# ------------------------------------------------------------
# 기존 04번 결과와 섞이지 않도록 별도 폴더를 사용한다.

OUTPUT_DIR = (
    GUIDE_OUTPUT_DIR
    / "03_중간결과표"
    / "04_v2_사업내용분류분석"
)

TABLE_DIR = OUTPUT_DIR / "tables"
FINAL_DIR = TABLE_DIR / "final"
DIAGNOSTICS_DIR = TABLE_DIR / "diagnostics"
ARCHIVE_DIR = TABLE_DIR / "archive"
FIGURE_DIR = OUTPUT_DIR / "figures"

for folder in [OUTPUT_DIR, TABLE_DIR, FINAL_DIR, DIAGNOSTICS_DIR, ARCHIVE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 7) 경로 존재 여부 확인
# ------------------------------------------------------------

print("\n[경로 확인]")
print(f"BASE_DIR: {BASE_DIR}")
print(f"GUIDE_OUTPUT_DIR: {GUIDE_OUTPUT_DIR}")

print("\n[입력 파일]")
print(f"INPUT_PATH: {INPUT_PATH}")
print(f"입력 파일 존재 여부: {INPUT_PATH.exists()}")

print("\n[참고 기준 파일]")
print(f"REFERENCE_PATH: {REFERENCE_PATH}")
print(f"참고 기준 파일 존재 여부: {REFERENCE_PATH.exists()}")

print("\n[출력 폴더]")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"TABLE_DIR: {TABLE_DIR}")
print(f"FINAL_DIR: {FINAL_DIR}")
print(f"DIAGNOSTICS_DIR: {DIAGNOSTICS_DIR}")
print(f"ARCHIVE_DIR: {ARCHIVE_DIR}")
print(f"FIGURE_DIR: {FIGURE_DIR}")

Mounted at /content/drive
RUN_TS: 20260606_1912

[경로 확인]
BASE_DIR: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA
GUIDE_OUTPUT_DIR: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보

[입력 파일]
INPUT_PATH: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/03_v1_사업명텍스트분석/tables/final/03_project_name_text_analysis_v1_20260529_0135.parquet
입력 파일 존재 여부: True

[참고 기준 파일]
REFERENCE_PATH: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/Reference Resources/[텍스트 분석] 참고 자료/04 Notebook - 사업 내용 유형/사업 내용 분류.txt
참고 기준 파일 존재 여부: True

[출력 폴더]
OUTPUT_DIR: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석
TABLE_DIR: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables
FINAL_DIR: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류

### *추가적인 파일 생성 방지 코드.

출력 결과 상 '저장 완료'표시는 실제로 저장되지 않음.

코드 작업 과정만을 표시하기 위함.

04B-2. 최신 03번 master 데이터 불러오기 및 필수 열 확인
1. 이 단계의 목적

식별자 보강이 완료된 최신 03번 master parquet 파일을 불러오고, 04B handoff에 필요한 필수 열이 모두 존재하는지 확인합니다.

특히 아래 두 열이 중요합니다.

v1_row_id
원파일명

이 두 열은 05번 병합 안정성을 위해 반드시 확인해야 합니다.

In [3]:
# 04B-2. 최신 03번 master 데이터 불러오기 및 필수 열 확인

# ------------------------------------------------------------
# 0) 입력 파일 존재 여부 재확인
# ------------------------------------------------------------

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"입력 파일을 찾을 수 없습니다.\n"
        f"현재 INPUT_PATH: {INPUT_PATH}\n"
        "04B-1에서 INPUT_PATH 경로를 다시 확인하세요."
    )

print("✅ 입력 파일 존재 확인 완료")


# ------------------------------------------------------------
# 1) parquet 파일 불러오기
# ------------------------------------------------------------

df_04b = pd.read_parquet(INPUT_PATH)

print("✅ 최신 03번 master parquet 불러오기 완료")
print(f"df_04b shape: {df_04b.shape}")


# ------------------------------------------------------------
# 2) 전체 열 목록 확인
# ------------------------------------------------------------

print("\n[전체 열 목록]")
for i, col in enumerate(df_04b.columns, start=1):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 3) 04B 필수 열 존재 여부 확인
# ------------------------------------------------------------

required_cols_04b = [
    "v1_row_id",
    "시도",
    "지자체명",
    "원파일명",
    "사업명",
    "사업명_clean",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "tokens_analysis_second_filter",
    "보조사업자",
    "평가결과",
    "평가종류",
]

missing_required_cols_04b = [
    col for col in required_cols_04b
    if col not in df_04b.columns
]

print("\n[04B 필수 열 확인]")
if len(missing_required_cols_04b) == 0:
    print("✅ 04B 필수 열이 모두 존재합니다.")
else:
    print("❌ 누락된 필수 열이 있습니다.")
    print(missing_required_cols_04b)

    # 자주 쓰이는 대체 후보 열을 함께 출력한다.
    possible_id_cols = [
        col for col in df_04b.columns
        if ("row" in col.lower()) or ("id" in col.lower()) or ("파일" in col) or ("source" in col.lower())
    ]

    print("\n[식별자/파일명 관련 후보 열]")
    print(possible_id_cols)

    raise KeyError(
        "04B 필수 열이 누락되어 다음 단계로 진행할 수 없습니다. "
        "03번 보강 결과 또는 열 이름을 다시 확인하세요."
    )


# ------------------------------------------------------------
# 4) 식별자 품질 점검
# ------------------------------------------------------------

id_check_summary = pd.DataFrame({
    "항목": [
        "전체 행 수",
        "v1_row_id 결측 수",
        "v1_row_id 중복 수",
        "원파일명 결측 수",
        "사업명 결측 수",
        "보조사업자 결측 수",
        "평가결과 결측 수",
        "평가종류 결측 수",
    ],
    "값": [
        len(df_04b),
        df_04b["v1_row_id"].isna().sum(),
        df_04b["v1_row_id"].duplicated().sum(),
        df_04b["원파일명"].isna().sum(),
        df_04b["사업명"].isna().sum(),
        df_04b["보조사업자"].isna().sum(),
        df_04b["평가결과"].isna().sum(),
        df_04b["평가종류"].isna().sum(),
    ]
})

print("\n[식별자 및 주요 열 품질 점검]")
display(id_check_summary)


# ------------------------------------------------------------
# 5) 기본 샘플 확인
# ------------------------------------------------------------

preview_cols = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
]

preview_cols = [col for col in preview_cols if col in df_04b.columns]

print("\n[기본 샘플 10행]")
display(df_04b[preview_cols].head(10))


# ------------------------------------------------------------
# 6) 토큰 열 타입 확인
# ------------------------------------------------------------

token_type_rows = []

for col in ["tokens_analysis_primary", "tokens_analysis_second_filter"]:
    sample_non_null = df_04b[col].dropna().head(20)
    observed_types = sorted(set([type(x).__name__ for x in sample_non_null]))

    token_type_rows.append({
        "토큰열": col,
        "결측수": df_04b[col].isna().sum(),
        "관찰된_타입_상위샘플": ", ".join(observed_types),
    })

token_type_summary = pd.DataFrame(token_type_rows)

print("\n[토큰 열 타입 점검]")
display(token_type_summary)

✅ 입력 파일 존재 확인 완료
✅ 최신 03번 master parquet 불러오기 완료
df_04b shape: (79262, 43)

[전체 열 목록]
01. 시도
02. 지자체명
03. 사업명
04. 보조사업자
05. 사업비합계
06. 보조금
07. 사업자부담
08. 평가결과
09. 평가종류
10. 사업명_is_missing
11. 사업명_clean
12. 사업명_key
13. 사업명_clean_is_empty
14. 사업명_clean_changed
15. 사업명_clean_length
16. 사업명_clean_spacefix
17. char_space_issue_flag
18. char_space_run_text
19. char_space_run_count
20. char_space_max_run_len
21. spacefix_applied
22. spacefix_note
23. char_space_run_text_after_fix2
24. tokens_raw
25. tokens_raw_count
26. tokens_raw_empty
27. tokens_filtered_keep_region
28. tokens_filtered_keep_region_count
29. tokens_filtered_keep_region_empty
30. tokens_filtered_no_region
31. tokens_filtered_no_region_count
32. tokens_filtered_no_region_empty
33. tokens_filtered_no_region_strict
34. tokens_filtered_no_region_strict_count
35. tokens_filtered_no_region_strict_empty
36. tokens_analysis_primary
37. tokens_analysis_primary_count
38. tokens_analysis_primary_empty
39. tokens_analysis_second_filter
40. 

,항목,값
0,전체 행 수,79262
1,v1_row_id 결측 수,0
2,v1_row_id 중복 수,0
3,원파일명 결측 수,0
4,사업명 결측 수,0
5,보조사업자 결측 수,182
6,평가결과 결측 수,78
7,평가종류 결측 수,1466



[기본 샘플 10행]


,v1_row_id,원파일명,시도,지자체명,사업명,사업명_clean_spacefix,tokens_analysis_primary,보조사업자,평가결과,평가종류
0,0,경남_남해군.xlsx,경남,남해군,마늘재배 영농지원단 지원,마늘재배 영농지원단 지원,"[마늘, 재배, 영농, 지원단]",농협중앙회 남해군지부,매우우수,유지필요성평가
1,1,경남_남해군.xlsx,경남,남해군,2023년 사)진주지역범죄피해자지원센터‘등불 ’남해지부 운영,2023년 사)진주지역범죄피해자지원센터'등불 '남해지부 운영,"[지역, 범죄, 피해자, 센터, 등불, 지부]",사단법인진주지역범죄피 해자지원센터남해지부,미흡,유지필요성평가
2,2,경남_남해군.xlsx,경남,남해군,자유민주주의신장 및 자유선진의식 고취,자유민주주의신장 및 자유선진의식 고취,"[자유, 민주주의, 신장, 자유선진, 의식, 고취]",한국자유총연맹남해군지 회,미흡,유지필요성평가
3,3,경남_남해군.xlsx,경남,남해군,보물섬 독서학교 지원,보물섬 독서학교 지원,"[보물섬, 독서, 학교]",보물섬남해독서학교,미흡,유지필요성평가
4,4,경남_남해군.xlsx,경남,남해군,참전경찰유공자 보훈활동,참전경찰유공자 보훈활동,"[참전, 경찰, 유공자, 보훈, 활동]",한국참전경찰유공자회,미흡,유지필요성평가
5,5,경남_남해군.xlsx,경남,남해군,무공수훈자회 운영비 보조,무공수훈자회 운영비 보조,"[무공, 수훈자]",대한민국무공수훈자회남 해 군 지 회,미흡,유지필요성평가
6,6,경남_남해군.xlsx,경남,남해군,행복나눔센터 운영,행복나눔센터 운영,"[행복, 센터]",남해군지역사회보장협의 체,미흡,유지필요성평가
7,7,경남_남해군.xlsx,경남,남해군,경로당 양곡구입비 지원,경로당 양곡구입비 지원,"[경로당, 양곡, 구입비]",봉내경로당 외 10개소,미흡,유지필요성평가
8,8,경남_남해군.xlsx,경남,남해군,장애인직업재활 자활교육,장애인직업재활 자활교육,"[장애인, 직업, 재활, 자활, 교육]",남해장애인종합복지관,미흡,유지필요성평가
9,9,경남_남해군.xlsx,경남,남해군,어린이집 운전기사 인건비 (미조지역영유아수송),어린이집 운전기사 인건비 (미조지역영유아수송),"[어린이집, 운전기사, 미조, 지역, 영유아, 수송]",모 모 어 린 이 집,미흡,유지필요성평가



[토큰 열 타입 점검]


,토큰열,결측수,관찰된_타입_상위샘플
0,tokens_analysis_primary,0,ndarray
1,tokens_analysis_second_filter,0,ndarray


04B-3. 사업 내용 분류 기준 파일 불러오기 및 원문 구조 확인
1. 이 단계의 목적

은세가 공유한 사업 내용 분류.txt 파일을 읽고, 기준 파일의 구조를 확인합니다.

이 단계에서는 아직 분류 규칙을 확정하지 않습니다.
먼저 파일 안에 어떤 대분류, 세분류, 키워드, 예시가 있는지 사람이 확인할 수 있게 출력합니다.

In [4]:
# 04B-3. 사업 내용 분류 기준 파일 불러오기 및 원문 구조 확인

# ------------------------------------------------------------
# 0) 기준 파일 존재 여부 확인
# ------------------------------------------------------------

if not REFERENCE_PATH.exists():
    raise FileNotFoundError(
        f"사업 내용 분류 기준 파일을 찾을 수 없습니다.\n"
        f"현재 REFERENCE_PATH: {REFERENCE_PATH}\n"
        "04B-1에서 REFERENCE_PATH 경로를 다시 확인하세요."
    )

print("✅ 사업 내용 분류 기준 파일 존재 확인 완료")


# ------------------------------------------------------------
# 1) 여러 인코딩을 시도해서 txt 파일 읽기
# ------------------------------------------------------------
# txt 파일은 utf-8, cp949, euc-kr 중 하나일 가능성이 있다.

candidate_encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

reference_text_raw = None
used_encoding = None

for enc in candidate_encodings:
    try:
        reference_text_raw = REFERENCE_PATH.read_text(encoding=enc)
        used_encoding = enc
        break
    except UnicodeDecodeError:
        continue

if reference_text_raw is None:
    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        "utf-8, cp949, euc-kr 인코딩으로 모두 읽지 못했습니다."
    )

print(f"✅ 기준 파일 읽기 완료")
print(f"사용 인코딩: {used_encoding}")
print(f"전체 글자 수: {len(reference_text_raw):,}")


# ------------------------------------------------------------
# 2) 줄 단위로 분해
# ------------------------------------------------------------

reference_lines = reference_text_raw.splitlines()

reference_nonempty_lines = [
    line.strip()
    for line in reference_lines
    if str(line).strip() != ""
]

print(f"\n전체 줄 수: {len(reference_lines):,}")
print(f"빈 줄 제외 줄 수: {len(reference_nonempty_lines):,}")


# ------------------------------------------------------------
# 3) 원문 앞부분 출력
# ------------------------------------------------------------
# 기준 파일의 구조를 확인하기 위해 앞부분 120줄을 출력한다.

print("\n[사업 내용 분류.txt 원문 미리보기 - 빈 줄 제외 앞 120줄]")
for i, line in enumerate(reference_nonempty_lines[:120], start=1):
    print(f"{i:03d}: {line}")


# ------------------------------------------------------------
# 4) 줄 단위 미리보기 DataFrame 생성
# ------------------------------------------------------------

reference_preview_df = pd.DataFrame({
    "line_no": range(1, len(reference_nonempty_lines) + 1),
    "text": reference_nonempty_lines
})

print("\n[기준 파일 줄 단위 미리보기]")
display(reference_preview_df.head(120))


# ------------------------------------------------------------
# 5) 기준 파일 구조 추정용 간단 탐색
# ------------------------------------------------------------
# 대분류/세분류/키워드/예시 같은 표현이 들어간 줄을 우선 찾아본다.
# 이 결과는 자동 분류가 아니라 구조 파악용이다.

structure_keywords = [
    "대분류",
    "중분류",
    "세분류",
    "하위",
    "유형",
    "키워드",
    "예시",
    "판단",
    "분류",
]

structure_hint_rows = []

for idx, line in enumerate(reference_nonempty_lines, start=1):
    hit_keywords = [kw for kw in structure_keywords if kw in line]

    if hit_keywords:
        structure_hint_rows.append({
            "line_no": idx,
            "hit_keywords": ", ".join(hit_keywords),
            "text": line
        })

reference_structure_hint_df = pd.DataFrame(structure_hint_rows)

print("\n[기준 파일 구조 힌트 줄]")
if len(reference_structure_hint_df) > 0:
    display(reference_structure_hint_df.head(100))
else:
    print("구조 힌트 키워드가 포함된 줄이 발견되지 않았습니다.")
    print("원문 미리보기를 보고 다음 셀에서 직접 파싱 기준을 정해야 합니다.")


# ------------------------------------------------------------
# 6) 원문 백업 저장
# ------------------------------------------------------------
# 04B 기준 파일을 그대로 diagnostics에 백업해둔다.
# 나중에 어떤 기준 파일을 사용했는지 추적하기 위한 용도다.

reference_backup_path = DIAGNOSTICS_DIR / f"04B_reference_business_content_raw_{RUN_TS}.txt"

reference_backup_path.write_text(reference_text_raw, encoding="utf-8")

print("\n[기준 파일 원문 백업 저장 완료]")
print(reference_backup_path)

✅ 사업 내용 분류 기준 파일 존재 확인 완료
✅ 기준 파일 읽기 완료
사용 인코딩: utf-8-sig
전체 글자 수: 230

전체 줄 수: 22
빈 줄 제외 줄 수: 21

[사업 내용 분류.txt 원문 미리보기 - 빈 줄 제외 앞 120줄]
001: 출처: https://www.bojo.go.kr/bojo.do
002: 출처 내 경로: 공모사업찾기>공모사업 목록
003: 검색 필터: 사업연도 - 2023, 사업구분 - 지방보조사업, [사업주제] 검색 화면 참고
004: 행정/통일/외교
005: 안전 보장
006: 교육 보장
007: 문화활동
008: 관광/휴양활동
009: 종교활동
010: 환경향상
011: 사회복지향상
012: 보훈향상
013: 고용안정
014: 주거안정
015: 보건/의료
016: 1차 산업지원
017: 산업/에너지지원
018: 국토개발지원
019: 교통/물류진흥
020: 방송/통신진흥
021: 과학기술진흥

[기준 파일 줄 단위 미리보기]


,line_no,text
0,1,출처: https://www.bojo.go.kr/bojo.do
1,2,출처 내 경로: 공모사업찾기>공모사업 목록
2,3,"검색 필터: 사업연도 - 2023, 사업구분 - 지방보조사업, [사업주제] 검색 화..."
3,4,행정/통일/외교
4,5,안전 보장
5,6,교육 보장
6,7,문화활동
7,8,관광/휴양활동
8,9,종교활동
9,10,환경향상



[기준 파일 구조 힌트 줄]
구조 힌트 키워드가 포함된 줄이 발견되지 않았습니다.
원문 미리보기를 보고 다음 셀에서 직접 파싱 기준을 정해야 합니다.

[기준 파일 원문 백업 저장 완료]
/content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_reference_business_content_raw_20260606_1912.txt


**차단 확인

In [5]:
# 04B-3 저장 차단 여부 확인

from pathlib import Path
import pandas as pd

# 04B-3 출력에 나온 파일 경로를 그대로 넣습니다.
check_path = Path("/content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_reference_business_content_raw_20260606_1912.txt")

print("04B-3 백업 txt 실제 파일 존재 여부:", check_path.exists())

if "DRY_RUN_SAVE_LOG" in globals():
    dry_log_df = pd.DataFrame(DRY_RUN_SAVE_LOG)
    print("\n현재까지 저장 차단 로그 수:", len(dry_log_df))
    display(dry_log_df.tail(20))
else:
    print("DRY_RUN_SAVE_LOG가 없습니다. 04B-1B 저장 차단 셀이 실행되지 않았을 수 있습니다.")

04B-3 백업 txt 실제 파일 존재 여부: False

현재까지 저장 차단 로그 수: 1


,저장종류,저장대상
0,Path.write_text,/content/drive/MyDrive/파공장 (PA Factory)/2026 K...


04B-4. 사업 내용 분류 기준 구조화 및 규칙 초안 생성
1. 이 단계의 목적

사업 내용 분류.txt에서 확인된 18개 사업주제명을 04B의 공식 기준 분류명으로 정리합니다.

이 단계에서는 아직 df_04b 전체에 분류를 적용하지 않습니다.
먼저 기준 분류표와 규칙 초안을 만든 뒤, 다음 단계에서 실제 분류를 적용합니다.

In [6]:
# 04B-4. 사업 내용 분류 기준 구조화 및 규칙 초안 생성

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 확인
# ------------------------------------------------------------

required_objects = [
    "df_04b",
    "reference_nonempty_lines",
    "reference_text_raw",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1 → 04B-2 → 04B-3을 먼저 실행하세요."
    )

print("✅ 04B-4 실행 전 필수 객체 확인 완료")


# ------------------------------------------------------------
# 1) 기준 파일에서 사업내용분류명 추출
# ------------------------------------------------------------
# 기준 파일 앞 3줄은 출처/경로/검색필터 설명이므로 제외한다.
# 4번째 줄부터 실제 사업주제명으로 사용한다.

business_content_categories = reference_nonempty_lines[3:]

print("[사업 내용 분류.txt 기준 분류명]")
for i, cat in enumerate(business_content_categories, start=1):
    print(f"{i:02d}. {cat}")


# ------------------------------------------------------------
# 2) 기준 분류표 생성
# ------------------------------------------------------------

reference_category_df = pd.DataFrame({
    "source": "사업 내용 분류.txt",
    "category_order": range(1, len(business_content_categories) + 1),
    "사업내용분류_대분류": business_content_categories,
    "사업내용분류_세분류_존재여부": False,
    "비고": "기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음"
})

print("\n[기준 분류표]")
display(reference_category_df)


# ------------------------------------------------------------
# 3) 분류명 누락/중복 점검
# ------------------------------------------------------------

category_check_summary = pd.DataFrame({
    "항목": [
        "기준 분류명 수",
        "기준 분류명 결측 수",
        "기준 분류명 중복 수",
    ],
    "값": [
        len(reference_category_df),
        reference_category_df["사업내용분류_대분류"].isna().sum(),
        reference_category_df["사업내용분류_대분류"].duplicated().sum(),
    ]
})

print("\n[기준 분류명 점검]")
display(category_check_summary)


# ------------------------------------------------------------
# 4) 사업내용분류 규칙 초안 생성
# ------------------------------------------------------------
# 중요:
# - 분류명은 사업 내용 분류.txt의 명칭을 그대로 사용한다.
# - 기준 파일에는 키워드가 없으므로, 아래 키워드는 04B에서 자동분류를 위해 만든 운영 규칙 초안이다.
# - 사업명만으로 불안정한 경우는 검토필요로 남긴다.
# - 과도하게 분류하지 않기 위해 strong/weak를 구분한다.

BUSINESS_CONTENT_RULE_VERSION_V1 = "business_content_rules_v1_reference_category_operationalized"

business_content_rules_v1 = {
    "행정/통일/외교": {
        "strong": [
            "행정", "통일", "민주평통", "평화", "민주", "인권",
            "주민자치", "자치", "이장", "새마을", "바르게살기",
            "자유총연맹", "자원봉사", "봉사", "국제교류", "외교"
        ],
        "weak": [
            "협의회", "단체", "지회", "법인", "활동", "운영"
        ],
        "note": "행정·시민단체·통일·평화·외교 관련 사업"
    },

    "안전 보장": {
        "strong": [
            "안전", "재난", "방재", "소방", "방범", "범죄",
            "피해자", "구급", "응급", "교통질서", "모범운전자",
            "CCTV", "cctv"
        ],
        "weak": [
            "예방", "보호", "순찰", "감시", "질서"
        ],
        "note": "재난·방범·범죄피해·응급·안전 관련 사업"
    },

    "교육 보장": {
        "strong": [
            "교육", "학교", "대학", "인재", "장학", "평생교육",
            "학습", "진로", "독서", "도서관", "문해", "교실",
            "아카데미"
        ],
        "weak": [
            "역량", "연수", "프로그램", "강좌", "강사"
        ],
        "note": "교육·학습·장학·학교·도서관 관련 사업"
    },

    "문화활동": {
        "strong": [
            "문화", "예술", "공연", "전시", "국악", "농악",
            "미술", "음악", "문학", "콘텐츠", "문화재",
            "박물관", "향교", "제례", "석전대제", "법회",
            "전통", "민속", "가요제", "가요", "예총"
        ],
        "weak": [
            "행사", "기념", "축전", "한마당", "체험", "심포지엄"
        ],
        "note": "문화·예술·전통문화·공연·전시 관련 사업"
    },

    "관광/휴양활동": {
        "strong": [
            "관광", "휴양", "축제", "여행", "탐방", "투어",
            "체험관광", "관광객", "관광지", "둘레길", "캠핑",
            "해수욕장", "마을축제"
        ],
        "weak": [
            "체험", "홍보", "마케팅", "방문", "코스"
        ],
        "note": "관광·휴양·축제·방문객 유치 관련 사업"
    },

    "종교활동": {
        "strong": [
            "종교", "불교", "기독교", "천주교", "교회", "성당",
            "사찰", "절", "법회", "목사", "스님", "신앙"
        ],
        "weak": [
            "예배", "기도", "신도"
        ],
        "note": "종교 단체·종교행사 관련 사업"
    },

    "환경향상": {
        "strong": [
            "환경", "생태", "기후", "탄소", "쓰레기", "폐기물",
            "재활용", "하천", "수질", "숲", "녹지", "미세먼지",
            "자연환경", "환경보전", "정화", "자원순환"
        ],
        "weak": [
            "자연", "보전", "개선", "정비"
        ],
        "note": "환경보전·생태·폐기물·자원순환 관련 사업"
    },

    "사회복지향상": {
        "strong": [
            "복지", "복지관", "장애인", "노인", "노인회", "어르신",
            "아동", "청소년", "여성", "가족", "다문화", "보육",
            "어린이집", "경로당", "돌봄", "자활", "재활",
            "취약", "저소득", "영유아", "한부모"
        ],
        "weak": [
            "시설", "센터", "종사자", "위문", "나눔"
        ],
        "note": "사회복지 대상자·돌봄·보육·취약계층 관련 사업"
    },

    "보훈향상": {
        "strong": [
            "보훈", "유공자", "참전", "고엽제", "미망인",
            "수훈자", "군인회", "유족회", "전우회", "상이군경",
            "광복회", "재향군인회", "전몰군경", "무공"
        ],
        "weak": [
            "위문", "추모", "기념"
        ],
        "note": "보훈·참전·유공자·유족 관련 사업"
    },

    "고용안정": {
        "strong": [
            "고용", "일자리", "취업", "창업", "근로자", "노동",
            "청년창업", "직업", "훈련", "인력양성"
        ],
        "weak": [
            "청년", "역량", "교육", "지원"
        ],
        "note": "고용·취업·일자리·창업 관련 사업"
    },

    "주거안정": {
        "strong": [
            "주거", "주택", "공동주택", "빈집", "임대주택",
            "집수리", "주거환경", "마을공동시설"
        ],
        "weak": [
            "생활환경", "정비", "시설개선"
        ],
        "note": "주거·주택·공동주택·빈집 관련 사업"
    },

    "보건/의료": {
        "strong": [
            "보건", "의료", "병원", "건강", "치매", "감염",
            "질병", "예방접종", "정신건강", "위생", "의약",
            "진료", "환자"
        ],
        "weak": [
            "예방", "관리", "상담", "검진"
        ],
        "note": "보건·의료·건강·감염·질병 관련 사업"
    },

    "1차 산업지원": {
        "strong": [
            "농업", "농촌", "농가", "농산물", "농산", "농업인",
            "작물", "쌀", "마늘", "양파", "원예", "축산",
            "축산물", "가축", "한우", "수산", "수산물", "어업",
            "어촌", "임업", "산림", "귀농", "귀촌", "영농",
            "재배", "벼", "감자", "딸기", "양봉", "낙농",
            "병해충", "방제"
        ],
        "weak": [
            "생산", "품질", "공급", "포장재", "인증", "출하", "물류비"
        ],
        "note": "농림축산·수산·임업 등 1차 산업 관련 사업"
    },

    "산업/에너지지원": {
        "strong": [
            "산업", "기업", "소상공인", "상권", "전통시장",
            "마케팅", "수출", "제품", "브랜드", "사회적기업",
            "에너지", "태양광", "발전", "전기", "가스"
        ],
        "weak": [
            "시장", "할인", "직거래", "브랜딩", "커머스"
        ],
        "note": "산업·기업·상권·에너지 관련 사업"
    },

    "국토개발지원": {
        "strong": [
            "국토", "도시재생", "도시", "지역개발", "개발",
            "인프라", "거점", "생활권", "공간", "정비",
            "도로", "하천정비"
        ],
        "weak": [
            "마을", "지역", "주민", "활성화", "조성"
        ],
        "note": "국토·도시·지역개발·인프라 관련 사업"
    },

    "교통/물류진흥": {
        "strong": [
            "교통", "물류", "택시", "버스", "운수", "운송",
            "주차", "도로교통", "교통약자", "터미널", "카드결제",
            "단말기", "통신수수료"
        ],
        "weak": [
            "수송", "이동", "차량", "운행"
        ],
        "note": "교통·물류·운수·택시·버스 관련 사업"
    },

    "방송/통신진흥": {
        "strong": [
            "방송", "통신", "미디어", "인터넷", "정보통신",
            "ICT", "스마트", "디지털", "온라인", "플랫폼"
        ],
        "weak": [
            "홍보", "영상", "콘텐츠", "채널"
        ],
        "note": "방송·통신·디지털·미디어 관련 사업"
    },

    "과학기술진흥": {
        "strong": [
            "과학", "기술", "연구", "R&D", "실험", "과학기술",
            "혁신기술", "기술개발", "연구개발"
        ],
        "weak": [
            "혁신", "개발", "지원"
        ],
        "note": "과학기술·연구개발 관련 사업"
    },
}


# ------------------------------------------------------------
# 5) 규칙 요약표 생성
# ------------------------------------------------------------

rule_rows = []

for category, rule in business_content_rules_v1.items():
    rule_rows.append({
        "사업내용분류_대분류": category,
        "strong_keywords": ", ".join(rule.get("strong", [])),
        "weak_keywords": ", ".join(rule.get("weak", [])),
        "strong_keyword_count": len(rule.get("strong", [])),
        "weak_keyword_count": len(rule.get("weak", [])),
        "note": rule.get("note", ""),
        "rule_id": BUSINESS_CONTENT_RULE_VERSION_V1,
    })

business_content_rule_summary_v1 = pd.DataFrame(rule_rows)

print("\n[사업내용분류 규칙 요약표]")
display(business_content_rule_summary_v1)


# ------------------------------------------------------------
# 6) 기준 파일 분류명과 규칙 분류명 일치 여부 확인
# ------------------------------------------------------------

categories_from_reference = set(reference_category_df["사업내용분류_대분류"])
categories_from_rules = set(business_content_rule_summary_v1["사업내용분류_대분류"])

missing_in_rules = sorted(categories_from_reference - categories_from_rules)
extra_in_rules = sorted(categories_from_rules - categories_from_reference)

category_rule_match_summary = pd.DataFrame({
    "항목": [
        "기준 파일 분류명 수",
        "규칙 분류명 수",
        "기준에는 있으나 규칙에 없는 분류 수",
        "규칙에는 있으나 기준에 없는 분류 수",
    ],
    "값": [
        len(categories_from_reference),
        len(categories_from_rules),
        len(missing_in_rules),
        len(extra_in_rules),
    ],
    "상세": [
        "",
        "",
        ", ".join(missing_in_rules),
        ", ".join(extra_in_rules),
    ]
})

print("\n[기준 분류명 - 규칙 분류명 일치 여부]")
display(category_rule_match_summary)

if len(missing_in_rules) > 0 or len(extra_in_rules) > 0:
    raise ValueError(
        "기준 파일의 분류명과 규칙 dictionary의 분류명이 일치하지 않습니다. "
        "분류명은 사업 내용 분류.txt의 명칭을 그대로 사용해야 합니다."
    )

print("✅ 기준 파일 분류명과 규칙 분류명이 모두 일치합니다.")


# ------------------------------------------------------------
# 7) 저장
# ------------------------------------------------------------

reference_category_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_reference_categories_v1_{RUN_TS}.xlsx"
)

rule_summary_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_rule_summary_v1_{RUN_TS}.xlsx"
)

with pd.ExcelWriter(reference_category_path, engine="openpyxl") as writer:
    reference_category_df.to_excel(writer, sheet_name="reference_categories", index=False)
    category_check_summary.to_excel(writer, sheet_name="category_check", index=False)

with pd.ExcelWriter(rule_summary_path, engine="openpyxl") as writer:
    business_content_rule_summary_v1.to_excel(writer, sheet_name="rule_summary", index=False)
    category_rule_match_summary.to_excel(writer, sheet_name="category_rule_match", index=False)

print("\n[저장 완료]")
print(f"- 기준 분류표: {reference_category_path}")
print(f"- 사업내용분류 규칙 요약표: {rule_summary_path}")

print("\n✅ 04B-4 사업 내용 분류 기준 구조화 및 규칙 초안 생성 완료")

✅ 04B-4 실행 전 필수 객체 확인 완료
[사업 내용 분류.txt 기준 분류명]
01. 행정/통일/외교
02. 안전 보장
03. 교육 보장
04. 문화활동
05. 관광/휴양활동
06. 종교활동
07. 환경향상
08. 사회복지향상
09. 보훈향상
10. 고용안정
11. 주거안정
12. 보건/의료
13. 1차 산업지원
14. 산업/에너지지원
15. 국토개발지원
16. 교통/물류진흥
17. 방송/통신진흥
18. 과학기술진흥

[기준 분류표]


,source,category_order,사업내용분류_대분류,사업내용분류_세분류_존재여부,비고
0,사업 내용 분류.txt,1,행정/통일/외교,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
1,사업 내용 분류.txt,2,안전 보장,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
2,사업 내용 분류.txt,3,교육 보장,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
3,사업 내용 분류.txt,4,문화활동,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
4,사업 내용 분류.txt,5,관광/휴양활동,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
5,사업 내용 분류.txt,6,종교활동,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
6,사업 내용 분류.txt,7,환경향상,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
7,사업 내용 분류.txt,8,사회복지향상,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
8,사업 내용 분류.txt,9,보훈향상,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음
9,사업 내용 분류.txt,10,고용안정,False,기준 파일에는 세분류·키워드·예시 사업명이 별도로 제시되어 있지 않음



[기준 분류명 점검]


,항목,값
0,기준 분류명 수,18
1,기준 분류명 결측 수,0
2,기준 분류명 중복 수,0



[사업내용분류 규칙 요약표]


,사업내용분류_대분류,strong_keywords,weak_keywords,strong_keyword_count,weak_keyword_count,note,rule_id
0,행정/통일/외교,"행정, 통일, 민주평통, 평화, 민주, 인권, 주민자치, 자치, 이장, 새마을, 바...","협의회, 단체, 지회, 법인, 활동, 운영",16,6,행정·시민단체·통일·평화·외교 관련 사업,business_content_rules_v1_reference_category_o...
1,안전 보장,"안전, 재난, 방재, 소방, 방범, 범죄, 피해자, 구급, 응급, 교통질서, 모범운...","예방, 보호, 순찰, 감시, 질서",13,5,재난·방범·범죄피해·응급·안전 관련 사업,business_content_rules_v1_reference_category_o...
2,교육 보장,"교육, 학교, 대학, 인재, 장학, 평생교육, 학습, 진로, 독서, 도서관, 문해,...","역량, 연수, 프로그램, 강좌, 강사",13,5,교육·학습·장학·학교·도서관 관련 사업,business_content_rules_v1_reference_category_o...
3,문화활동,"문화, 예술, 공연, 전시, 국악, 농악, 미술, 음악, 문학, 콘텐츠, 문화재, ...","행사, 기념, 축전, 한마당, 체험, 심포지엄",21,6,문화·예술·전통문화·공연·전시 관련 사업,business_content_rules_v1_reference_category_o...
4,관광/휴양활동,"관광, 휴양, 축제, 여행, 탐방, 투어, 체험관광, 관광객, 관광지, 둘레길, 캠...","체험, 홍보, 마케팅, 방문, 코스",13,5,관광·휴양·축제·방문객 유치 관련 사업,business_content_rules_v1_reference_category_o...
5,종교활동,"종교, 불교, 기독교, 천주교, 교회, 성당, 사찰, 절, 법회, 목사, 스님, 신앙","예배, 기도, 신도",12,3,종교 단체·종교행사 관련 사업,business_content_rules_v1_reference_category_o...
6,환경향상,"환경, 생태, 기후, 탄소, 쓰레기, 폐기물, 재활용, 하천, 수질, 숲, 녹지, ...","자연, 보전, 개선, 정비",16,4,환경보전·생태·폐기물·자원순환 관련 사업,business_content_rules_v1_reference_category_o...
7,사회복지향상,"복지, 복지관, 장애인, 노인, 노인회, 어르신, 아동, 청소년, 여성, 가족, 다...","시설, 센터, 종사자, 위문, 나눔",21,5,사회복지 대상자·돌봄·보육·취약계층 관련 사업,business_content_rules_v1_reference_category_o...
8,보훈향상,"보훈, 유공자, 참전, 고엽제, 미망인, 수훈자, 군인회, 유족회, 전우회, 상이군...","위문, 추모, 기념",14,3,보훈·참전·유공자·유족 관련 사업,business_content_rules_v1_reference_category_o...
9,고용안정,"고용, 일자리, 취업, 창업, 근로자, 노동, 청년창업, 직업, 훈련, 인력양성","청년, 역량, 교육, 지원",10,4,고용·취업·일자리·창업 관련 사업,business_content_rules_v1_reference_category_o...



[기준 분류명 - 규칙 분류명 일치 여부]


,항목,값,상세
0,기준 파일 분류명 수,18,
1,규칙 분류명 수,18,
2,기준에는 있으나 규칙에 없는 분류 수,0,
3,규칙에는 있으나 기준에 없는 분류 수,0,


✅ 기준 파일 분류명과 규칙 분류명이 모두 일치합니다.

[저장 완료]
- 기준 분류표: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_reference_categories_v1_20260606_1912.xlsx
- 사업내용분류 규칙 요약표: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_rule_summary_v1_20260606_1912.xlsx

✅ 04B-4 사업 내용 분류 기준 구조화 및 규칙 초안 생성 완료


*참고
지금 만든 키워드는 기준 파일에 직접 적힌 키워드가 아니라 자동분류를 위해 우리가 구성한 운영 규칙이다.“보조금24 사업주제 기준의 18개 분류명을 사용하고, 사업명 텍스트 기반 키워드 규칙으로 운영화했다"

04B-5. 사업내용분류 v1 적용 및 기본 분포 확인
1. 이 단계의 목적

이 단계의 목적은 business_content_rules_v1을 사용해 전체 사업명에 사업내용분류 v1을 적용하는 것입니다.

이번 단계는 아직 최종 handoff 저장이 아니라, v1 적용 후 분포와 검토필요 사례를 진단하는 단계입니다.

저장 파일은 다음입니다.

---
04B_business_content_classification_summary_v1_YYYYMMDD_HHMM.xlsx
04B_business_content_review_cases_v1_YYYYMMDD_HHMM.xlsx
04B_business_content_rule_id_summary_v1_YYYYMMDD_HHMM.xlsx

In [7]:
# 04B-5. 사업내용분류 v1 적용 및 기본 분포 확인

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 확인
# ------------------------------------------------------------

required_objects = [
    "df_04b",
    "business_content_rules_v1",
    "BUSINESS_CONTENT_RULE_VERSION_V1",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1 → 04B-2 → 04B-3 → 04B-4를 먼저 실행하세요."
    )

required_cols_04b_5 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "tokens_analysis_second_filter",
    "보조사업자",
    "평가결과",
    "평가종류",
]

missing_cols_04b_5 = [
    col for col in required_cols_04b_5
    if col not in df_04b.columns
]

if missing_cols_04b_5:
    raise KeyError(
        f"04B-5 실행에 필요한 열이 없습니다: {missing_cols_04b_5}"
    )

print("✅ 04B-5 실행 전 필수 객체 및 열 확인 완료")


# ------------------------------------------------------------
# 1) 토큰 안전 변환 함수 정의
# ------------------------------------------------------------
# tokens_analysis_primary가 ndarray로 확인되었으므로,
# set 변환 전에 반드시 list 형태로 안전하게 바꾼다.

def safe_parse_tokens(x):
    """
    tokens 열을 안전하게 list[str]로 변환한다.
    - list, tuple, set, numpy.ndarray 처리
    - 문자열 형태의 리스트 처리
    - NaN/None 처리
    """
    if x is None:
        return []

    # numpy ndarray
    if isinstance(x, np.ndarray):
        return [str(v).strip() for v in x.tolist() if str(v).strip()]

    # list/tuple/set
    if isinstance(x, (list, tuple, set)):
        return [str(v).strip() for v in list(x) if str(v).strip()]

    # pandas NaN
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass

    # 문자열
    if isinstance(x, str):
        text = x.strip()

        if text == "":
            return []

        # "['복지', '노인']" 같은 문자열 리스트 처리
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    return [str(v).strip() for v in list(parsed) if str(v).strip()]
            except Exception:
                pass

        # 쉼표 구분 문자열 처리
        if "," in text:
            return [v.strip() for v in text.split(",") if v.strip()]

        return [text]

    return [str(x).strip()] if str(x).strip() else []


def normalize_text_for_match(x):
    """문자열 비교용 소문자 변환"""
    if x is None:
        return ""
    return str(x).lower()


def get_token_set_lower(tokens):
    """
    토큰을 소문자 set으로 변환.
    ndarray가 직접 set에 들어가면 TypeError가 날 수 있으므로
    safe_parse_tokens를 먼저 적용한다.
    """
    parsed = safe_parse_tokens(tokens)
    return set([str(t).lower().strip() for t in parsed if str(t).strip()])


def has_token_any(token_set_lower, words):
    """토큰 set에 words 중 하나라도 정확히 포함되면 True"""
    words_lower = [str(w).lower().strip() for w in words]
    return any(w in token_set_lower for w in words_lower)


def text_has_any(text_lower, words):
    """문자열에 words 중 하나라도 포함되면 True"""
    words_lower = [str(w).lower().strip() for w in words]
    return any(w in text_lower for w in words_lower)


def match_keywords(token_set_lower, text_lower, keywords):
    """
    키워드 매칭 함수.
    - 토큰 정확 매칭 우선
    - 2글자 이상 키워드는 사업명 문자열 포함도 허용
    - 1글자 키워드는 문자열 포함으로 잡으면 오분류 위험이 커서 토큰 기준만 사용
    """
    hits = []

    for kw in keywords:
        kw_str = str(kw).strip()
        kw_lower = kw_str.lower()

        if kw_lower in token_set_lower:
            hits.append(kw_str)
            continue

        if len(kw_str) >= 2 and kw_lower in text_lower:
            hits.append(kw_str)
            continue

    return sorted(set(hits))


print("✅ 토큰 안전 변환 및 매칭 함수 정의 완료")


# ------------------------------------------------------------
# 2) 분류 결과 생성 함수
# ------------------------------------------------------------

def make_business_content_result_v1(
    category,
    reason,
    need_review,
    strong_matches,
    weak_matches,
    strong_categories,
    weak_categories
):
    """사업내용분류 v1 결과를 pd.Series로 반환"""

    if need_review is False:
        analysis_flag = "주분석_포함"
    elif category == "복합":
        analysis_flag = "보조분석_검토"
    else:
        analysis_flag = "해석주의"

    return pd.Series({
        "사업내용분류_대분류_v1": category,
        "사업내용분류_세분류_v1": "",
        "사업내용분류_후보분야_v1": ", ".join(sorted(set(strong_categories + weak_categories))),
        "사업내용분류_분류근거_v1": reason,
        "사업내용분류_검토필요여부_v1": bool(need_review),
        "사업내용분류_분석포함여부_v1": analysis_flag,
        "사업내용분류_rule_id_v1": BUSINESS_CONTENT_RULE_VERSION_V1,
        "사업내용분류_강키워드_v1": json.dumps(strong_matches, ensure_ascii=False),
        "사업내용분류_약키워드_v1": json.dumps(weak_matches, ensure_ascii=False),
    })


# ------------------------------------------------------------
# 3) 사업내용분류 v1 분류 함수
# ------------------------------------------------------------

def classify_business_content_v1(row):
    """
    사업내용분류 v1 분류 함수.

    원칙:
    - 기준 파일의 18개 분류명을 사용한다.
    - strong 키워드 단일 매칭이면 해당 분류로 확정한다.
    - strong 키워드가 여러 분야에 걸치면 우선순위 규칙을 먼저 적용한다.
    - 그래도 애매하면 복합으로 남긴다.
    - weak 키워드만 잡히면 검토필요로 남긴다.
    """

    tokens = safe_parse_tokens(row.get("tokens_analysis_primary", []))
    token_set_lower = get_token_set_lower(tokens)
    text_lower = normalize_text_for_match(row.get("사업명_clean_spacefix", ""))

    strong_matches = {}
    weak_matches = {}

    for category, rule in business_content_rules_v1.items():
        strong_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("strong", [])
        )
        weak_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("weak", [])
        )

        if strong_hit:
            strong_matches[category] = strong_hit

        if weak_hit:
            weak_matches[category] = weak_hit

    strong_categories = list(strong_matches.keys())
    weak_categories = list(weak_matches.keys())

    # --------------------------------------------------------
    # 3-1) 우선순위 예외 규칙
    # --------------------------------------------------------

    # A. 보훈 관련은 보훈향상 우선
    bohun_terms = [
        "보훈", "유공자", "참전", "고엽제", "미망인", "수훈자",
        "군인회", "유족회", "전우회", "상이군경", "광복회",
        "재향군인회", "전몰군경", "무공"
    ]
    if has_token_any(token_set_lower, bohun_terms) or text_has_any(text_lower, bohun_terms):
        return make_business_content_result_v1(
            category="보훈향상",
            reason="우선규칙: 보훈·참전·유공자·유족 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # B. 사회복지 대상자 관련은 사회복지향상 우선
    welfare_terms = [
        "복지", "장애인", "노인", "노인회", "어르신", "아동", "청소년",
        "여성", "가족", "다문화", "보육", "어린이집", "경로당",
        "돌봄", "자활", "재활", "취약", "저소득", "영유아", "한부모"
    ]
    if has_token_any(token_set_lower, welfare_terms) or text_has_any(text_lower, welfare_terms):
        return make_business_content_result_v1(
            category="사회복지향상",
            reason="우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # C. 1차 산업 관련은 1차 산업지원 우선
    primary_industry_terms = [
        "농업", "농촌", "농가", "농산물", "농산", "농업인",
        "작물", "쌀", "마늘", "양파", "원예", "축산", "축산물",
        "가축", "한우", "수산", "수산물", "어업", "어촌",
        "임업", "산림", "귀농", "귀촌", "영농", "재배",
        "벼", "감자", "딸기", "양봉", "낙농", "병해충", "방제"
    ]
    if has_token_any(token_set_lower, primary_industry_terms) or text_has_any(text_lower, primary_industry_terms):
        return make_business_content_result_v1(
            category="1차 산업지원",
            reason="우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # D. 보건/의료 명확 키워드는 보건/의료 우선
    health_terms = [
        "보건", "의료", "병원", "건강", "치매", "감염", "질병",
        "예방접종", "정신건강", "위생", "의약", "진료", "환자", "검진"
    ]
    if has_token_any(token_set_lower, health_terms) or text_has_any(text_lower, health_terms):
        return make_business_content_result_v1(
            category="보건/의료",
            reason="우선규칙: 보건·의료·건강·감염·질병 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # E. 안전 관련은 안전 보장 우선
    safety_terms = [
        "안전", "재난", "방재", "소방", "방범", "범죄",
        "피해자", "구급", "응급", "교통질서", "모범운전자", "cctv"
    ]
    if has_token_any(token_set_lower, safety_terms) or text_has_any(text_lower, safety_terms):
        return make_business_content_result_v1(
            category="안전 보장",
            reason="우선규칙: 재난·방범·범죄피해·응급·안전 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # F. 교통/물류 명확 키워드는 교통/물류진흥 우선
    transport_terms = [
        "교통", "물류", "택시", "버스", "운수", "운송", "주차",
        "도로교통", "교통약자", "터미널", "카드결제", "단말기",
        "통신수수료", "수송"
    ]
    if has_token_any(token_set_lower, transport_terms) or text_has_any(text_lower, transport_terms):
        return make_business_content_result_v1(
            category="교통/물류진흥",
            reason="우선규칙: 교통·물류·운수·택시·버스 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # G. 주거 명확 키워드는 주거안정 우선
    housing_terms = [
        "주거", "주택", "공동주택", "빈집", "임대주택",
        "집수리", "주거환경", "마을공동시설"
    ]
    if has_token_any(token_set_lower, housing_terms) or text_has_any(text_lower, housing_terms):
        return make_business_content_result_v1(
            category="주거안정",
            reason="우선규칙: 주거·주택·공동주택·빈집 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # H. 종교활동은 종교 고유 맥락일 때 우선
    religion_terms = [
        "종교", "불교", "기독교", "천주교", "교회", "성당",
        "사찰", "법회", "목사", "스님", "신앙", "예배", "기도"
    ]
    if has_token_any(token_set_lower, religion_terms) or text_has_any(text_lower, religion_terms):
        return make_business_content_result_v1(
            category="종교활동",
            reason="우선규칙: 종교 단체·종교행사 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # I. 관광/휴양은 관광·휴양·방문객 맥락일 때 우선
    tourism_terms = [
        "관광", "휴양", "여행", "탐방", "투어", "체험관광",
        "관광객", "관광지", "둘레길", "캠핑", "해수욕장"
    ]
    if has_token_any(token_set_lower, tourism_terms) or text_has_any(text_lower, tourism_terms):
        return make_business_content_result_v1(
            category="관광/휴양활동",
            reason="우선규칙: 관광·휴양·방문객 유치 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # J. 문화활동 명확 키워드
    culture_terms = [
        "문화", "예술", "공연", "전시", "국악", "농악",
        "미술", "음악", "문학", "콘텐츠", "문화재",
        "박물관", "향교", "제례", "석전대제", "전통",
        "민속", "가요제", "가요", "예총"
    ]
    if has_token_any(token_set_lower, culture_terms) or text_has_any(text_lower, culture_terms):
        return make_business_content_result_v1(
            category="문화활동",
            reason="우선규칙: 문화·예술·전통문화·공연·전시 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # K. 행정/통일/외교 명확 키워드
    admin_terms = [
        "행정", "통일", "민주평통", "평화", "민주", "인권",
        "주민자치", "자치", "이장", "새마을", "바르게살기",
        "자유총연맹", "자원봉사", "봉사", "국제교류", "외교"
    ]
    if has_token_any(token_set_lower, admin_terms) or text_has_any(text_lower, admin_terms):
        return make_business_content_result_v1(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체·통일·평화·외교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # --------------------------------------------------------
    # 3-2) 일반 규칙
    # --------------------------------------------------------

    # strong이 1개 분야에만 잡힌 경우
    if len(strong_categories) == 1:
        main_category = strong_categories[0]
        return make_business_content_result_v1(
            category=main_category,
            reason=f"강키워드 단일 분야 매칭: {main_category}({', '.join(strong_matches[main_category])})",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # strong이 여러 분야에 잡힌 경우
    if len(strong_categories) >= 2:
        return make_business_content_result_v1(
            category="복합",
            reason=f"여러 분야 강키워드 동시 매칭: {', '.join(strong_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # weak만 잡힌 경우
    if len(strong_categories) == 0 and len(weak_categories) >= 1:
        return make_business_content_result_v1(
            category="검토필요",
            reason=f"약키워드만 매칭: {', '.join(weak_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # 아무것도 안 잡힌 경우
    return make_business_content_result_v1(
        category="기타",
        reason="매칭 키워드 없음",
        need_review=True,
        strong_matches=strong_matches,
        weak_matches=weak_matches,
        strong_categories=strong_categories,
        weak_categories=weak_categories
    )


print("✅ 사업내용분류 v1 분류 함수 정의 완료")


# ------------------------------------------------------------
# 4) 전체 데이터에 사업내용분류 v1 적용
# ------------------------------------------------------------

classification_result_v1 = df_04b.apply(classify_business_content_v1, axis=1)

for col in classification_result_v1.columns:
    df_04b[col] = classification_result_v1[col]

print("✅ 사업내용분류 v1 적용 완료")
print(f"생성된 열: {classification_result_v1.columns.tolist()}")


# ------------------------------------------------------------
# 5) 사업내용분류 대분류 분포
# ------------------------------------------------------------

business_content_classification_summary_v1 = (
    df_04b["사업내용분류_대분류_v1"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_대분류_v1")
    .reset_index(name="사업수")
)

business_content_classification_summary_v1["비율"] = (
    business_content_classification_summary_v1["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v1 대분류 분포]")
display(business_content_classification_summary_v1)


# ------------------------------------------------------------
# 6) 분석포함여부 분포
# ------------------------------------------------------------

business_content_analysis_flag_summary_v1 = (
    df_04b["사업내용분류_분석포함여부_v1"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분석포함여부_v1")
    .reset_index(name="사업수")
)

business_content_analysis_flag_summary_v1["비율"] = (
    business_content_analysis_flag_summary_v1["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v1 분석포함여부 분포]")
display(business_content_analysis_flag_summary_v1)


# ------------------------------------------------------------
# 7) 기타/검토필요/복합 요약
# ------------------------------------------------------------

review_target_values = ["기타", "검토필요", "복합"]

business_content_review_group_summary_v1 = pd.DataFrame({
    "구분": ["기타", "검토필요", "복합", "합계"],
    "사업수": [
        int((df_04b["사업내용분류_대분류_v1"] == "기타").sum()),
        int((df_04b["사업내용분류_대분류_v1"] == "검토필요").sum()),
        int((df_04b["사업내용분류_대분류_v1"] == "복합").sum()),
        int(df_04b["사업내용분류_대분류_v1"].isin(review_target_values).sum()),
    ]
})

business_content_review_group_summary_v1["비율"] = (
    business_content_review_group_summary_v1["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v1 기타/검토필요/복합 요약]")
display(business_content_review_group_summary_v1)


# ------------------------------------------------------------
# 8) rule_id별 행 수
# ------------------------------------------------------------

business_content_rule_id_summary_v1 = (
    df_04b["사업내용분류_rule_id_v1"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_rule_id_v1")
    .reset_index(name="사업수")
)

business_content_rule_id_summary_v1["비율"] = (
    business_content_rule_id_summary_v1["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v1 rule_id별 행 수]")
display(business_content_rule_id_summary_v1)


# ------------------------------------------------------------
# 9) 분류근거별 행 수 상위
# ------------------------------------------------------------

business_content_reason_summary_v1 = (
    df_04b["사업내용분류_분류근거_v1"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분류근거_v1")
    .reset_index(name="사업수")
)

business_content_reason_summary_v1["비율"] = (
    business_content_reason_summary_v1["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v1 분류근거별 행 수 상위 30개]")
display(business_content_reason_summary_v1.head(30))


# ------------------------------------------------------------
# 10) 대표 검토필요 사례 확인
# ------------------------------------------------------------

review_case_cols_v1 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류_v1",
    "사업내용분류_후보분야_v1",
    "사업내용분류_분류근거_v1",
    "사업내용분류_검토필요여부_v1",
    "사업내용분류_분석포함여부_v1",
    "사업내용분류_rule_id_v1",
]

review_case_cols_v1 = [
    col for col in review_case_cols_v1
    if col in df_04b.columns
]

business_content_review_cases_v1 = df_04b[
    (df_04b["사업내용분류_검토필요여부_v1"] == True)
    | (df_04b["사업내용분류_대분류_v1"].isin(review_target_values))
].copy()

print(f"\n[사업내용분류 v1 검토필요·기타·복합 사례 수] {len(business_content_review_cases_v1):,}행")
display(business_content_review_cases_v1[review_case_cols_v1].head(80))


# ------------------------------------------------------------
# 11) 분류별 대표 사업명 샘플
# ------------------------------------------------------------

sample_rows = []

for category in df_04b["사업내용분류_대분류_v1"].dropna().unique():
    temp = df_04b[df_04b["사업내용분류_대분류_v1"] == category].copy()
    temp_sample = temp[review_case_cols_v1].head(10)
    sample_rows.append(temp_sample)

if sample_rows:
    business_content_sample_by_category_v1 = pd.concat(sample_rows, axis=0)
else:
    business_content_sample_by_category_v1 = pd.DataFrame(columns=review_case_cols_v1)

print("\n[사업내용분류 v1 대분류별 대표 사업명 샘플]")
display(business_content_sample_by_category_v1)


# ------------------------------------------------------------
# 12) Excel 저장용 변환 함수
# ------------------------------------------------------------

def convert_cell_for_excel(x):
    """Excel 저장을 위해 list/ndarray/dict를 문자열로 변환"""

    if isinstance(x, np.ndarray):
        return ", ".join([str(v) for v in x.tolist()])

    if isinstance(x, list):
        return ", ".join([str(v) for v in x])

    if isinstance(x, tuple):
        return ", ".join([str(v) for v in list(x)])

    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False)

    return x


def convert_df_for_excel(df):
    """object 열의 복합 객체를 Excel 저장용 문자열로 변환"""
    out = df.copy()

    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].apply(convert_cell_for_excel)

    return out


# ------------------------------------------------------------
# 13) 진단 파일 저장
# ------------------------------------------------------------

classification_summary_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_classification_summary_v1_{RUN_TS}.xlsx"
)

review_cases_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_review_cases_v1_{RUN_TS}.xlsx"
)

rule_id_summary_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_rule_id_summary_v1_{RUN_TS}.xlsx"
)

reason_summary_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_reason_summary_v1_{RUN_TS}.xlsx"
)

sample_by_category_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_sample_by_category_v1_{RUN_TS}.xlsx"
)

with pd.ExcelWriter(classification_summary_path, engine="openpyxl") as writer:
    business_content_classification_summary_v1.to_excel(
        writer,
        sheet_name="classification_summary",
        index=False
    )
    business_content_analysis_flag_summary_v1.to_excel(
        writer,
        sheet_name="analysis_flag_summary",
        index=False
    )
    business_content_review_group_summary_v1.to_excel(
        writer,
        sheet_name="review_group_summary",
        index=False
    )

convert_df_for_excel(business_content_review_cases_v1[review_case_cols_v1]).to_excel(
    review_cases_path,
    index=False
)

business_content_rule_id_summary_v1.to_excel(
    rule_id_summary_path,
    index=False
)

business_content_reason_summary_v1.to_excel(
    reason_summary_path,
    index=False
)

convert_df_for_excel(business_content_sample_by_category_v1).to_excel(
    sample_by_category_path,
    index=False
)

print("\n[저장 완료]")
print(f"- v1 대분류/분석포함/검토그룹 요약: {classification_summary_path}")
print(f"- v1 검토필요·기타·복합 사례: {review_cases_path}")
print(f"- v1 rule_id별 행 수: {rule_id_summary_path}")
print(f"- v1 분류근거별 행 수: {reason_summary_path}")
print(f"- v1 대분류별 대표 사업명 샘플: {sample_by_category_path}")

print("\n✅ 04B-5 사업내용분류 v1 적용 및 기본 분포 확인 완료")

✅ 04B-5 실행 전 필수 객체 및 열 확인 완료
✅ 토큰 안전 변환 및 매칭 함수 정의 완료
✅ 사업내용분류 v1 분류 함수 정의 완료
✅ 사업내용분류 v1 적용 완료
생성된 열: ['사업내용분류_대분류_v1', '사업내용분류_세분류_v1', '사업내용분류_후보분야_v1', '사업내용분류_분류근거_v1', '사업내용분류_검토필요여부_v1', '사업내용분류_분석포함여부_v1', '사업내용분류_rule_id_v1', '사업내용분류_강키워드_v1', '사업내용분류_약키워드_v1']

[사업내용분류 v1 대분류 분포]


,사업내용분류_대분류_v1,사업수,비율
0,사회복지향상,18434,23.26
1,검토필요,16848,21.26
2,1차 산업지원,8185,10.33
3,기타,7681,9.69
4,문화활동,6410,8.09
5,행정/통일/외교,4116,5.19
6,보훈향상,3083,3.89
7,교육 보장,2867,3.62
8,안전 보장,1932,2.44
9,산업/에너지지원,1588,2.00



[사업내용분류 v1 분석포함여부 분포]


,사업내용분류_분석포함여부_v1,사업수,비율
0,주분석_포함,53786,67.86
1,해석주의,24529,30.95
2,보조분석_검토,947,1.19



[사업내용분류 v1 기타/검토필요/복합 요약]


...


[사업내용분류 v1 검토필요·기타·복합 사례 수] 25,476행


,v1_row_id,원파일명,시도,지자체명,사업명,사업명_clean_spacefix,tokens_analysis_primary,보조사업자,평가결과,평가종류,사업내용분류_대분류_v1,사업내용분류_후보분야_v1,사업내용분류_분류근거_v1,사업내용분류_검토필요여부_v1,사업내용분류_분석포함여부_v1,사업내용분류_rule_id_v1
6,6,경남_남해군.xlsx,경남,남해군,행복나눔센터 운영,행복나눔센터 운영,"[행복, 센터]",남해군지역사회보장협의 체,미흡,유지필요성평가,검토필요,"사회복지향상, 행정/통일/외교","약키워드만 매칭: 행정/통일/외교, 사회복지향상",True,해석주의,business_content_rules_v1_reference_category_o...
12,12,경남_남해군.xlsx,경남,남해군,남중권생활체육 교류 지원,남중권생활체육 교류 지원,"[남, 생활, 체육, 교류]",남 해 군 체 육 회,미흡,유지필요성평가,검토필요,"고용안정, 과학기술진흥","약키워드만 매칭: 고용안정, 과학기술진흥",True,해석주의,business_content_rules_v1_reference_category_o...
14,14,경남_남해군.xlsx,경남,남해군,레포츠 자격증 취득 지원,레포츠 자격증 취득 지원,"[레포츠, 자격증, 취득]",남 해 군 체 육 회,미흡,유지필요성평가,검토필요,"고용안정, 과학기술진흥","약키워드만 매칭: 고용안정, 과학기술진흥",True,해석주의,business_content_rules_v1_reference_category_o...
22,22,경남_남해군.xlsx,경남,남해군,2023년 보물섬 남해포럼 운영,2023년 보물섬 남해포럼 운영,"[보물섬, 포럼]",보물섬 남해포럼,보통,유지필요성평가,검토필요,행정/통일/외교,약키워드만 매칭: 행정/통일/외교,True,해석주의,business_content_rules_v1_reference_category_o...
27,27,경남_남해군.xlsx,경남,남해군,국민의식개혁 및 기초질서확립운동,국민의식개혁 및 기초질서확립운동,"[국민, 의식, 개혁, 기초, 질서, 확립, 운동]",바르게살기운동남해군협 의 회,보통,유지필요성평가,검토필요,안전 보장,약키워드만 매칭: 안전 보장,True,해석주의,business_content_rules_v1_reference_category_o...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,276,경남_산청군.xlsx,경남,산청군,산엔청희망나눔 이동목욕차량 운영,산엔청희망나눔 이동목욕차량 운영,"[산엔청, 희망, 이동, 목욕, 차량]",산청군지역사회보장협의체,미흡,성과평가,검토필요,"교통/물류진흥, 사회복지향상, 행정/통일/외교","약키워드만 매칭: 행정/통일/외교, 사회복지향상, 교통/물류진흥",True,해석주의,business_content_rules_v1_reference_category_o...
291,291,경남_산청군.xlsx,경남,산청군,어린이 그림그리기대회 행사 지원,어린이 그림그리기대회 행사 지원,"[어린이, 그림, 대회, 행사]",산청군아동위원협의회,미흡,성과평가,검토필요,"고용안정, 과학기술진흥, 문화활동","약키워드만 매칭: 문화활동, 고용안정, 과학기술진흥",True,해석주의,business_content_rules_v1_reference_category_o...
292,292,경남_산청군.xlsx,경남,산청군,"어린이날 행사 지원(1,000인)","어린이날 행사 지원(1,000인)","[어린이날, 행사, 1,000]",산청군아동위원협의회,우수,성과평가,검토필요,"고용안정, 과학기술진흥, 문화활동","약키워드만 매칭: 문화활동, 고용안정, 과학기술진흥",True,해석주의,business_content_rules_v1_reference_category_o...
293,293,경남_산청군.xlsx,경남,산청군,산청군동학농민혁명군 유해 발굴 지원,산청군동학농민혁명군 유해 발굴 지원,"[산청군, 동학, 농민, 혁명, 유해, 발굴]",산청동학농민혁명기념사업회,미흡,성과평가,검토필요,"고용안정, 과학기술진흥","약키워드만 매칭: 고용안정, 과학기술진흥",True,해석주의,business_content_rules_v1_reference_category_o...



[사업내용분류 v1 대분류별 대표 사업명 샘플]


,v1_row_id,원파일명,시도,지자체명,사업명,사업명_clean_spacefix,tokens_analysis_primary,보조사업자,평가결과,평가종류,사업내용분류_대분류_v1,사업내용분류_후보분야_v1,사업내용분류_분류근거_v1,사업내용분류_검토필요여부_v1,사업내용분류_분석포함여부_v1,사업내용분류_rule_id_v1
0,0,경남_남해군.xlsx,경남,남해군,마늘재배 영농지원단 지원,마늘재배 영농지원단 지원,"[마늘, 재배, 영농, 지원단]",농협중앙회 남해군지부,매우우수,유지필요성평가,1차 산업지원,"1차 산업지원, 고용안정, 과학기술진흥",우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,False,주분석_포함,business_content_rules_v1_reference_category_o...
15,15,경남_남해군.xlsx,경남,남해군,수산물 상생할인 지원사업,수산물 상생할인 지원사업,"[수산물, 상생, 할인]",남 해 군 수 협,미흡,유지필요성평가,1차 산업지원,"1차 산업지원, 고용안정, 과학기술진흥, 산업/에너지지원",우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,False,주분석_포함,business_content_rules_v1_reference_category_o...
19,19,경남_남해군.xlsx,경남,남해군,농촌신활력 플러스사업 (추진단사무국상근직원인건비),농촌신활력 플러스사업 (추진단사무국상근직원인건비),"[농촌신활력, 플러스사업, 추진단, 사무국, 상근, 직원]",남해군신활력플러스사업추진단,미흡,유지필요성평가,1차 산업지원,1차 산업지원,우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,False,주분석_포함,business_content_rules_v1_reference_category_o...
20,20,경남_남해군.xlsx,경남,남해군,GAP 인증농산물 포장재 지원사업,GAP 인증농산물 포장재 지원사업,"[GAP, 인증, 농산물, 포장재]",남해느루농원 외 3개소,미흡,유지필요성평가,1차 산업지원,"1차 산업지원, 고용안정, 과학기술진흥",우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,False,주분석_포함,business_content_rules_v1_reference_category_o...
90,90,경남_남해군.xlsx,경남,남해군,농업경영인회 경남도대회 참가,농업경영인회 경남도대회 참가,"[농업, 경영, 경남도, 대회]",(사)한국농업경영인남해군 연 합 회,보통,유지필요성평가,1차 산업지원,1차 산업지원,우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,False,주분석_포함,business_content_rules_v1_reference_category_o...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6975,6975,V1 후보_성과_경남_김해시.xlsx,경남,김해시,김해형 스마트팜 조성사업,김해형 스마트팜 조성사업,"[스마트, 팜, 조성]",김종우,매우우수,성과평가,방송/통신진흥,"국토개발지원, 방송/통신진흥",강키워드 단일 분야 매칭: 방송/통신진흥(스마트),False,주분석_포함,business_content_rules_v1_reference_category_o...
6976,6976,V1 후보_성과_경남_김해시.xlsx,경남,김해시,김해형 스마트팜 조성사업,김해형 스마트팜 조성사업,"[스마트, 팜, 조성]",권순남,매우우수,성과평가,방송/통신진흥,"국토개발지원, 방송/통신진흥",강키워드 단일 분야 매칭: 방송/통신진흥(스마트),False,주분석_포함,business_content_rules_v1_reference_category_o...
6977,6977,V1 후보_성과_경남_김해시.xlsx,경남,김해시,김해형 스마트팜 조성사업,김해형 스마트팜 조성사업,"[스마트, 팜, 조성]",정종대,매우우수,성과평가,방송/통신진흥,"국토개발지원, 방송/통신진흥",강키워드 단일 분야 매칭: 방송/통신진흥(스마트),False,주분석_포함,business_content_rules_v1_reference_category_o...
6978,6978,V1 후보_성과_경남_김해시.xlsx,경남,김해시,김해형 스마트팜 조성사업,김해형 스마트팜 조성사업,"[스마트, 팜, 조성]",김은규,매우우수,성과평가,방송/통신진흥,"국토개발지원, 방송/통신진흥",강키워드 단일 분야 매칭: 방송/통신진흥(스마트),False,주분석_포함,business_content_rules_v1_reference_category_o...



[저장 완료]
- v1 대분류/분석포함/검토그룹 요약: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_classification_summary_v1_20260606_1912.xlsx
- v1 검토필요·기타·복합 사례: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_review_cases_v1_20260606_1912.xlsx
- v1 rule_id별 행 수: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_rule_id_summary_v1_20260606_1912.xlsx
- v1 분류근거별 행 수: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_reason_summary_v1_20260606_1912.xlsx
- v1 대분류별 대표 사업명 샘플: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagn

04B-6. v1 결과 진단 기반 v2 보완 규칙 적용
1. 이 단계의 목적

이 단계의 목적은 04B-5의 v1 결과를 보존한 상태에서, 확인된 과민 매칭과 오분류 가능성을 보정한 v2 분류를 새로 만드는 것입니다.

주요 보정 내용은 다음입니다.

지원, 개발, 교육, 조성, 지역, 행사 같은 일반 약키워드 과민 매칭 완화
기준 파일에 별도 체육 분류가 없으므로 체육·스포츠·생활체육 관련 사업은 문화활동으로 운영화
스마트팜, 스마트 농업, 스마트 축산, 스마트 양식은 방송/통신진흥이 아니라 1차 산업지원으로 우선 처리
스마트 단독 키워드는 방송/통신 강키워드에서 제거
기존 v1 열은 유지하고, v2 열을 새로 생성

In [8]:
# 04B-6. v1 결과 진단 기반 v2 보완 규칙 적용

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 및 열 확인
# ------------------------------------------------------------

from copy import deepcopy

required_objects = [
    "df_04b",
    "business_content_rules_v1",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
    "safe_parse_tokens",
    "normalize_text_for_match",
    "get_token_set_lower",
    "has_token_any",
    "text_has_any",
    "match_keywords",
    "convert_df_for_excel",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1 → 04B-2 → 04B-3 → 04B-4 → 04B-5를 먼저 실행하세요."
    )

required_cols_04b_6 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류_v1",
    "사업내용분류_분류근거_v1",
    "사업내용분류_검토필요여부_v1",
    "사업내용분류_분석포함여부_v1",
]

missing_cols_04b_6 = [
    col for col in required_cols_04b_6
    if col not in df_04b.columns
]

if missing_cols_04b_6:
    raise KeyError(
        f"04B-6 실행에 필요한 열이 없습니다: {missing_cols_04b_6}\n"
        "04B-5가 정상 실행되었는지 확인하세요."
    )

print("✅ 04B-6 실행 전 필수 객체 및 열 확인 완료")


# ------------------------------------------------------------
# 1) v2 규칙 생성
# ------------------------------------------------------------
# v1 규칙을 복사한 뒤, 과민 매칭을 유발한 약키워드를 제거/완화한다.

BUSINESS_CONTENT_RULE_VERSION_V2 = "business_content_rules_v2_refined_general_weak_terms"

business_content_rules_v2 = deepcopy(business_content_rules_v1)


# ------------------------------------------------------------
# 1-1) 과민 매칭을 유발하는 일반 약키워드 제거
# ------------------------------------------------------------
# v1에서 '지원', '개발', '교육', '조성', '지역', '행사' 등이
# 너무 넓게 작동하여 검토필요가 과다하게 발생했다.
# 특히 고용안정/과학기술진흥 후보가 과도하게 잡혔으므로 보수적으로 제거한다.

weak_remove_map_v2 = {
    "고용안정": ["지원", "교육", "역량"],
    "과학기술진흥": ["지원", "개발", "혁신"],
    "국토개발지원": ["지역", "주민", "조성", "활성화"],
    "문화활동": ["행사", "기념", "체험"],
    "산업/에너지지원": ["시장"],
    "사회복지향상": ["시설", "센터"],
    "행정/통일/외교": ["활동", "운영", "단체", "지회", "법인", "협의회"],
}

for category, remove_terms in weak_remove_map_v2.items():
    if category in business_content_rules_v2:
        original_weak = business_content_rules_v2[category].get("weak", [])
        business_content_rules_v2[category]["weak"] = [
            w for w in original_weak
            if w not in remove_terms
        ]


# ------------------------------------------------------------
# 1-2) 체육 관련 사업을 문화활동으로 운영화
# ------------------------------------------------------------
# 사업 내용 분류.txt에는 체육 단독 분류가 없으므로,
# 체육·스포츠·생활체육 관련 사업은 문화활동에 포함한다.
# 단, '유도' 단독은 성균관유도회 오분류 위험이 있어 제외하고,
# 유도대회 등 명확한 표현만 사용한다.

sports_terms_for_culture = [
    "체육", "생활체육", "스포츠", "레포츠", "선수",
    "축구", "야구", "농구", "배구", "탁구", "배드민턴",
    "골프", "게이트볼", "파크골프", "수영", "마라톤",
    "태권도", "테니스", "볼링", "검도", "궁도",
    "전지훈련", "동계훈련", "스토브리그",
    "유도대회", "체육대회"
]

for term in sports_terms_for_culture:
    if term not in business_content_rules_v2["문화활동"]["strong"]:
        business_content_rules_v2["문화활동"]["strong"].append(term)


# ------------------------------------------------------------
# 1-3) 스마트팜 오분류 보정
# ------------------------------------------------------------
# v1에서는 '스마트' 때문에 스마트팜이 방송/통신진흥으로 갈 수 있었다.
# v2에서는 스마트팜 계열을 1차 산업지원으로 우선 처리한다.
# 또한 '스마트' 단독은 방송/통신진흥 strong에서 제거하고 weak로 둔다.

smart_agri_terms = [
    "스마트팜", "스마트 농업", "스마트농업",
    "스마트 축산", "스마트축산",
    "스마트 양식", "스마트양식"
]

for term in smart_agri_terms:
    if term not in business_content_rules_v2["1차 산업지원"]["strong"]:
        business_content_rules_v2["1차 산업지원"]["strong"].append(term)

if "스마트" in business_content_rules_v2["방송/통신진흥"]["strong"]:
    business_content_rules_v2["방송/통신진흥"]["strong"].remove("스마트")

if "스마트" not in business_content_rules_v2["방송/통신진흥"]["weak"]:
    business_content_rules_v2["방송/통신진흥"]["weak"].append("스마트")


# ------------------------------------------------------------
# 1-4) 명확한 문화·역사·지역기록 표현 보완
# ------------------------------------------------------------

culture_extra_terms_v2 = [
    "동학", "유림", "성균관유도회", "유도회",
    "향토", "면지", "군지", "읍지",
    "편찬", "발간", "역사", "전승"
]

for term in culture_extra_terms_v2:
    if term not in business_content_rules_v2["문화활동"]["strong"]:
        business_content_rules_v2["문화활동"]["strong"].append(term)


# ------------------------------------------------------------
# 1-5) 나눔·목욕 등 복지 맥락 보완
# ------------------------------------------------------------

welfare_extra_terms_v2 = [
    "행복나눔", "희망나눔", "목욕", "이동목욕",
    "무료급식", "급식지원", "푸드뱅크", "푸드마켓"
]

for term in welfare_extra_terms_v2:
    if term not in business_content_rules_v2["사회복지향상"]["strong"]:
        business_content_rules_v2["사회복지향상"]["strong"].append(term)


print("✅ 사업내용분류 v2 규칙 생성 완료")
print(f"규칙 버전: {BUSINESS_CONTENT_RULE_VERSION_V2}")


# ------------------------------------------------------------
# 2) v2 규칙 요약표 생성
# ------------------------------------------------------------

rule_rows_v2 = []

for category, rule in business_content_rules_v2.items():
    rule_rows_v2.append({
        "사업내용분류_대분류": category,
        "strong_keywords": ", ".join(rule.get("strong", [])),
        "weak_keywords": ", ".join(rule.get("weak", [])),
        "strong_keyword_count": len(rule.get("strong", [])),
        "weak_keyword_count": len(rule.get("weak", [])),
        "note": rule.get("note", ""),
        "rule_id": BUSINESS_CONTENT_RULE_VERSION_V2,
    })

business_content_rule_summary_v2 = pd.DataFrame(rule_rows_v2)

print("\n[사업내용분류 v2 규칙 요약표]")
display(business_content_rule_summary_v2)


# ------------------------------------------------------------
# 3) v2 결과 생성 함수
# ------------------------------------------------------------

def make_business_content_result_v2(
    category,
    reason,
    need_review,
    strong_matches,
    weak_matches,
    strong_categories,
    weak_categories
):
    """사업내용분류 v2 결과를 pd.Series로 반환"""

    if need_review is False:
        analysis_flag = "주분석_포함"
    elif category == "복합":
        analysis_flag = "보조분석_검토"
    else:
        analysis_flag = "해석주의"

    return pd.Series({
        "사업내용분류_대분류_v2": category,
        "사업내용분류_세분류_v2": "",
        "사업내용분류_후보분야_v2": ", ".join(sorted(set(strong_categories + weak_categories))),
        "사업내용분류_분류근거_v2": reason,
        "사업내용분류_검토필요여부_v2": bool(need_review),
        "사업내용분류_분석포함여부_v2": analysis_flag,
        "사업내용분류_rule_id_v2": BUSINESS_CONTENT_RULE_VERSION_V2,
        "사업내용분류_강키워드_v2": json.dumps(strong_matches, ensure_ascii=False),
        "사업내용분류_약키워드_v2": json.dumps(weak_matches, ensure_ascii=False),
    })


# ------------------------------------------------------------
# 4) v2 분류 함수
# ------------------------------------------------------------

def classify_business_content_v2(row):
    """
    사업내용분류 v2 분류 함수.

    v1 대비 보정:
    - 일반 약키워드 과민 매칭 완화
    - 체육 관련 사업은 문화활동으로 운영화
    - 스마트팜은 1차 산업지원 우선
    - 성균관유도회/향교/유림 맥락은 문화활동 우선
    """

    tokens = safe_parse_tokens(row.get("tokens_analysis_primary", []))
    token_set_lower = get_token_set_lower(tokens)
    text_lower = normalize_text_for_match(row.get("사업명_clean_spacefix", ""))

    strong_matches = {}
    weak_matches = {}

    for category, rule in business_content_rules_v2.items():
        strong_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("strong", [])
        )
        weak_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("weak", [])
        )

        if strong_hit:
            strong_matches[category] = strong_hit

        if weak_hit:
            weak_matches[category] = weak_hit

    strong_categories = list(strong_matches.keys())
    weak_categories = list(weak_matches.keys())


    # --------------------------------------------------------
    # 4-1) 우선순위 예외 규칙
    # --------------------------------------------------------

    # A. 보훈 관련은 보훈향상 우선
    bohun_terms = [
        "보훈", "유공자", "참전", "고엽제", "미망인", "수훈자",
        "군인회", "유족회", "전우회", "상이군경", "광복회",
        "재향군인회", "전몰군경", "무공"
    ]
    if has_token_any(token_set_lower, bohun_terms) or text_has_any(text_lower, bohun_terms):
        return make_business_content_result_v2(
            category="보훈향상",
            reason="우선규칙: 보훈·참전·유공자·유족 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # B. 사회복지 대상자 및 복지서비스 관련은 사회복지향상 우선
    welfare_terms = [
        "복지", "장애인", "노인", "노인회", "어르신", "아동", "청소년",
        "여성", "가족", "다문화", "보육", "어린이집", "경로당",
        "돌봄", "자활", "재활", "취약", "저소득", "영유아", "한부모",
        "행복나눔", "희망나눔", "목욕", "이동목욕", "무료급식", "푸드뱅크"
    ]
    if has_token_any(token_set_lower, welfare_terms) or text_has_any(text_lower, welfare_terms):
        return make_business_content_result_v2(
            category="사회복지향상",
            reason="우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # C. 스마트팜/스마트 농업은 1차 산업지원 우선
    smart_agri_priority_terms = [
        "스마트팜", "스마트 농업", "스마트농업",
        "스마트 축산", "스마트축산",
        "스마트 양식", "스마트양식"
    ]
    if text_has_any(text_lower, smart_agri_priority_terms):
        return make_business_content_result_v2(
            category="1차 산업지원",
            reason="우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # D. 1차 산업 관련은 1차 산업지원 우선
    primary_industry_terms = [
        "농업", "농촌", "농가", "농산물", "농산", "농업인",
        "작물", "쌀", "마늘", "양파", "원예", "축산", "축산물",
        "가축", "한우", "수산", "수산물", "어업", "어촌",
        "임업", "산림", "귀농", "귀촌", "영농", "재배",
        "벼", "감자", "딸기", "양봉", "낙농", "병해충", "방제"
    ]
    if has_token_any(token_set_lower, primary_industry_terms) or text_has_any(text_lower, primary_industry_terms):
        return make_business_content_result_v2(
            category="1차 산업지원",
            reason="우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # E. 보건/의료 명확 키워드는 보건/의료 우선
    health_terms = [
        "보건", "의료", "병원", "건강", "치매", "감염", "질병",
        "예방접종", "정신건강", "위생", "의약", "진료", "환자", "검진"
    ]
    if has_token_any(token_set_lower, health_terms) or text_has_any(text_lower, health_terms):
        return make_business_content_result_v2(
            category="보건/의료",
            reason="우선규칙: 보건·의료·건강·감염·질병 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # F. 안전 관련은 안전 보장 우선
    safety_terms = [
        "안전", "재난", "방재", "소방", "방범", "범죄",
        "피해자", "구급", "응급", "교통질서", "모범운전자", "cctv", "CCTV"
    ]
    if has_token_any(token_set_lower, safety_terms) or text_has_any(text_lower, safety_terms):
        return make_business_content_result_v2(
            category="안전 보장",
            reason="우선규칙: 재난·방범·범죄피해·응급·안전 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # G. 교통/물류 명확 키워드는 교통/물류진흥 우선
    transport_terms = [
        "교통", "물류", "택시", "버스", "운수", "운송", "주차",
        "도로교통", "교통약자", "터미널", "카드결제", "단말기",
        "통신수수료", "수송"
    ]
    if has_token_any(token_set_lower, transport_terms) or text_has_any(text_lower, transport_terms):
        return make_business_content_result_v2(
            category="교통/물류진흥",
            reason="우선규칙: 교통·물류·운수·택시·버스 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # H. 주거 명확 키워드는 주거안정 우선
    housing_terms = [
        "주거", "주택", "공동주택", "빈집", "임대주택",
        "집수리", "주거환경", "마을공동시설"
    ]
    if has_token_any(token_set_lower, housing_terms) or text_has_any(text_lower, housing_terms):
        return make_business_content_result_v2(
            category="주거안정",
            reason="우선규칙: 주거·주택·공동주택·빈집 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # I. 종교활동은 종교 고유 맥락일 때 우선
    religion_terms = [
        "종교", "불교", "기독교", "천주교", "교회", "성당",
        "사찰", "법회", "목사", "스님", "신앙", "예배", "기도"
    ]
    if has_token_any(token_set_lower, religion_terms) or text_has_any(text_lower, religion_terms):
        return make_business_content_result_v2(
            category="종교활동",
            reason="우선규칙: 종교 단체·종교행사 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # J. 체육 관련 사업은 문화활동으로 운영화
    # 사업 내용 분류.txt에 체육 단독 분류가 없기 때문에 문화활동으로 보낸다.
    sports_culture_terms = [
        "체육", "생활체육", "스포츠", "레포츠", "선수",
        "축구", "야구", "농구", "배구", "탁구", "배드민턴",
        "골프", "게이트볼", "파크골프", "수영", "마라톤",
        "태권도", "테니스", "볼링", "검도", "궁도",
        "전지훈련", "동계훈련", "스토브리그", "유도대회", "체육대회"
    ]
    if has_token_any(token_set_lower, sports_culture_terms) or text_has_any(text_lower, sports_culture_terms):
        return make_business_content_result_v2(
            category="문화활동",
            reason="우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로 운영화",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # K. 유림/향교/지역사 편찬 등 전통문화·역사 맥락은 문화활동 우선
    history_culture_terms = [
        "동학", "유림", "성균관유도회", "유도회",
        "향교", "향토", "면지", "군지", "읍지",
        "편찬", "발간", "역사", "전승"
    ]
    if has_token_any(token_set_lower, history_culture_terms) or text_has_any(text_lower, history_culture_terms):
        return make_business_content_result_v2(
            category="문화활동",
            reason="우선규칙: 전통문화·유림·지역사 편찬 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # L. 관광/휴양은 관광·휴양·방문객 맥락일 때 우선
    tourism_terms = [
        "관광", "휴양", "여행", "탐방", "투어", "체험관광",
        "관광객", "관광지", "둘레길", "캠핑", "해수욕장"
    ]
    if has_token_any(token_set_lower, tourism_terms) or text_has_any(text_lower, tourism_terms):
        return make_business_content_result_v2(
            category="관광/휴양활동",
            reason="우선규칙: 관광·휴양·방문객 유치 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # M. 문화활동 명확 키워드
    culture_terms = [
        "문화", "예술", "공연", "전시", "국악", "농악",
        "미술", "음악", "문학", "콘텐츠", "문화재",
        "박물관", "향교", "제례", "석전대제", "전통",
        "민속", "가요제", "가요", "예총"
    ]
    if has_token_any(token_set_lower, culture_terms) or text_has_any(text_lower, culture_terms):
        return make_business_content_result_v2(
            category="문화활동",
            reason="우선규칙: 문화·예술·전통문화·공연·전시 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # N. 행정/통일/외교 명확 키워드
    admin_terms = [
        "행정", "통일", "민주평통", "평화", "민주", "인권",
        "주민자치", "자치", "이장", "새마을", "바르게살기",
        "자유총연맹", "자원봉사", "봉사", "국제교류", "외교"
    ]
    if has_token_any(token_set_lower, admin_terms) or text_has_any(text_lower, admin_terms):
        return make_business_content_result_v2(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체·통일·평화·외교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # --------------------------------------------------------
    # 4-2) 일반 규칙
    # --------------------------------------------------------

    # strong이 1개 분야에만 잡힌 경우
    if len(strong_categories) == 1:
        main_category = strong_categories[0]
        return make_business_content_result_v2(
            category=main_category,
            reason=f"강키워드 단일 분야 매칭: {main_category}({', '.join(strong_matches[main_category])})",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # strong이 여러 분야에 잡힌 경우
    if len(strong_categories) >= 2:
        return make_business_content_result_v2(
            category="복합",
            reason=f"여러 분야 강키워드 동시 매칭: {', '.join(strong_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # weak만 잡힌 경우
    if len(strong_categories) == 0 and len(weak_categories) >= 1:
        return make_business_content_result_v2(
            category="검토필요",
            reason=f"약키워드만 매칭: {', '.join(weak_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # 아무것도 안 잡힌 경우
    return make_business_content_result_v2(
        category="기타",
        reason="매칭 키워드 없음",
        need_review=True,
        strong_matches=strong_matches,
        weak_matches=weak_matches,
        strong_categories=strong_categories,
        weak_categories=weak_categories
    )


print("✅ 사업내용분류 v2 분류 함수 정의 완료")


# ------------------------------------------------------------
# 5) 전체 데이터에 사업내용분류 v2 적용
# ------------------------------------------------------------

classification_result_v2 = df_04b.apply(classify_business_content_v2, axis=1)

for col in classification_result_v2.columns:
    df_04b[col] = classification_result_v2[col]

print("✅ 사업내용분류 v2 적용 완료")
print(f"생성된 열: {classification_result_v2.columns.tolist()}")


# ------------------------------------------------------------
# 6) v2 대분류 분포
# ------------------------------------------------------------

business_content_classification_summary_v2 = (
    df_04b["사업내용분류_대분류_v2"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_대분류_v2")
    .reset_index(name="사업수")
)

business_content_classification_summary_v2["비율"] = (
    business_content_classification_summary_v2["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v2 대분류 분포]")
display(business_content_classification_summary_v2)


# ------------------------------------------------------------
# 7) v2 분석포함여부 분포
# ------------------------------------------------------------

business_content_analysis_flag_summary_v2 = (
    df_04b["사업내용분류_분석포함여부_v2"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분석포함여부_v2")
    .reset_index(name="사업수")
)

business_content_analysis_flag_summary_v2["비율"] = (
    business_content_analysis_flag_summary_v2["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v2 분석포함여부 분포]")
display(business_content_analysis_flag_summary_v2)


# ------------------------------------------------------------
# 8) v1-v2 대분류 변화 비교
# ------------------------------------------------------------

business_content_v1_v2_compare = (
    df_04b
    .groupby(["사업내용분류_대분류_v1", "사업내용분류_대분류_v2"])
    .size()
    .reset_index(name="사업수")
    .sort_values("사업수", ascending=False)
)

print("\n[사업내용분류 v1 → v2 변화 상위 60개]")
display(business_content_v1_v2_compare.head(60))


# ------------------------------------------------------------
# 9) v1-v2 기타/검토필요/복합 비교
# ------------------------------------------------------------

review_target_values = ["기타", "검토필요", "복합"]

def count_business_content_review_group(df, col, version):
    return pd.DataFrame({
        "버전": version,
        "구분": ["기타", "검토필요", "복합", "합계"],
        "사업수": [
            int((df[col] == "기타").sum()),
            int((df[col] == "검토필요").sum()),
            int((df[col] == "복합").sum()),
            int(df[col].isin(review_target_values).sum()),
        ]
    })

business_content_review_group_compare_v1_v2 = pd.concat(
    [
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v1", "v1"),
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v2", "v2"),
    ],
    axis=0
)

business_content_review_group_compare_v1_v2["비율"] = (
    business_content_review_group_compare_v1_v2["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v1-v2 기타/검토필요/복합 비교]")
display(business_content_review_group_compare_v1_v2)


# ------------------------------------------------------------
# 10) v2 분류근거별 행 수
# ------------------------------------------------------------

business_content_reason_summary_v2 = (
    df_04b["사업내용분류_분류근거_v2"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분류근거_v2")
    .reset_index(name="사업수")
)

business_content_reason_summary_v2["비율"] = (
    business_content_reason_summary_v2["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v2 분류근거별 행 수 상위 40개]")
display(business_content_reason_summary_v2.head(40))


# ------------------------------------------------------------
# 11) v2 검토필요·기타·복합 사례
# ------------------------------------------------------------

review_case_cols_v2 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류_v1",
    "사업내용분류_대분류_v2",
    "사업내용분류_후보분야_v2",
    "사업내용분류_분류근거_v2",
    "사업내용분류_검토필요여부_v2",
    "사업내용분류_분석포함여부_v2",
    "사업내용분류_rule_id_v2",
]

review_case_cols_v2 = [
    col for col in review_case_cols_v2
    if col in df_04b.columns
]

business_content_review_cases_v2 = df_04b[
    (df_04b["사업내용분류_검토필요여부_v2"] == True)
    | (df_04b["사업내용분류_대분류_v2"].isin(review_target_values))
].copy()

print(f"\n[사업내용분류 v2 검토필요·기타·복합 사례 수] {len(business_content_review_cases_v2):,}행")
display(business_content_review_cases_v2[review_case_cols_v2].head(80))


# ------------------------------------------------------------
# 12) v1-v2 변경 사례
# ------------------------------------------------------------

business_content_changed_cases_v1_v2 = df_04b[
    df_04b["사업내용분류_대분류_v1"] != df_04b["사업내용분류_대분류_v2"]
].copy()

changed_case_cols = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "사업내용분류_대분류_v1",
    "사업내용분류_분류근거_v1",
    "사업내용분류_대분류_v2",
    "사업내용분류_분류근거_v2",
    "평가결과",
    "평가종류",
]

changed_case_cols = [
    col for col in changed_case_cols
    if col in business_content_changed_cases_v1_v2.columns
]

print(f"\n[v1-v2 대분류 변경 사례 수] {len(business_content_changed_cases_v1_v2):,}행")
display(business_content_changed_cases_v1_v2[changed_case_cols].head(100))


# ------------------------------------------------------------
# 13) 특정 보정 사례 확인
# ------------------------------------------------------------

# 스마트팜 관련
smart_farm_cases_v2 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains("스마트팜|스마트 농업|스마트농업", regex=True, na=False)
].copy()

# 체육 관련
sports_cases_v2 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "체육|생활체육|스포츠|레포츠|축구|야구|농구|배구|탁구|배드민턴|골프|게이트볼|파크골프|태권도|테니스|볼링|검도|궁도|마라톤|전지훈련",
        regex=True,
        na=False
    )
].copy()

special_check_cols = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업내용분류_대분류_v1",
    "사업내용분류_대분류_v2",
    "사업내용분류_분류근거_v2",
    "평가결과",
    "평가종류",
]

special_check_cols = [
    col for col in special_check_cols
    if col in df_04b.columns
]

print("\n[스마트팜 관련 v2 보정 사례]")
display(smart_farm_cases_v2[special_check_cols].head(50))

print("\n[체육 관련 v2 보정 사례]")
display(sports_cases_v2[special_check_cols].head(50))


# ------------------------------------------------------------
# 14) 저장
# ------------------------------------------------------------

rule_summary_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_rule_summary_v2_{RUN_TS}.xlsx"
)

classification_summary_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_classification_summary_v2_{RUN_TS}.xlsx"
)

v1_v2_compare_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_v1_v2_compare_{RUN_TS}.xlsx"
)

review_cases_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_review_cases_v2_{RUN_TS}.xlsx"
)

reason_summary_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_reason_summary_v2_{RUN_TS}.xlsx"
)

changed_cases_v1_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_changed_cases_v1_v2_{RUN_TS}.xlsx"
)

smart_farm_cases_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_smart_farm_cases_v2_{RUN_TS}.xlsx"
)

sports_cases_v2_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_sports_cases_v2_{RUN_TS}.xlsx"
)

business_content_rule_summary_v2.to_excel(rule_summary_v2_path, index=False)

with pd.ExcelWriter(classification_summary_v2_path, engine="openpyxl") as writer:
    business_content_classification_summary_v2.to_excel(
        writer,
        sheet_name="classification_summary_v2",
        index=False
    )
    business_content_analysis_flag_summary_v2.to_excel(
        writer,
        sheet_name="analysis_flag_summary_v2",
        index=False
    )
    business_content_review_group_compare_v1_v2.to_excel(
        writer,
        sheet_name="review_group_compare",
        index=False
    )

business_content_v1_v2_compare.to_excel(v1_v2_compare_path, index=False)
convert_df_for_excel(business_content_review_cases_v2[review_case_cols_v2]).to_excel(review_cases_v2_path, index=False)
business_content_reason_summary_v2.to_excel(reason_summary_v2_path, index=False)
convert_df_for_excel(business_content_changed_cases_v1_v2[changed_case_cols]).to_excel(changed_cases_v1_v2_path, index=False)
convert_df_for_excel(smart_farm_cases_v2[special_check_cols]).to_excel(smart_farm_cases_v2_path, index=False)
convert_df_for_excel(sports_cases_v2[special_check_cols]).to_excel(sports_cases_v2_path, index=False)

print("\n[저장 완료]")
print(f"- v2 규칙 요약표: {rule_summary_v2_path}")
print(f"- v2 대분류/분석포함/검토그룹 요약: {classification_summary_v2_path}")
print(f"- v1-v2 대분류 변화 비교: {v1_v2_compare_path}")
print(f"- v2 검토필요·기타·복합 사례: {review_cases_v2_path}")
print(f"- v2 분류근거별 행 수: {reason_summary_v2_path}")
print(f"- v1-v2 변경 사례: {changed_cases_v1_v2_path}")
print(f"- 스마트팜 관련 보정 사례: {smart_farm_cases_v2_path}")
print(f"- 체육 관련 보정 사례: {sports_cases_v2_path}")

print("\n✅ 04B-6 v1 결과 진단 기반 v2 보완 규칙 적용 완료")

✅ 04B-6 실행 전 필수 객체 및 열 확인 완료
✅ 사업내용분류 v2 규칙 생성 완료
규칙 버전: business_content_rules_v2_refined_general_weak_terms

[사업내용분류 v2 규칙 요약표]


,사업내용분류_대분류,strong_keywords,weak_keywords,strong_keyword_count,weak_keyword_count,note,rule_id
0,행정/통일/외교,"행정, 통일, 민주평통, 평화, 민주, 인권, 주민자치, 자치, 이장, 새마을, 바...",,16,0,행정·시민단체·통일·평화·외교 관련 사업,business_content_rules_v2_refined_general_weak...
1,안전 보장,"안전, 재난, 방재, 소방, 방범, 범죄, 피해자, 구급, 응급, 교통질서, 모범운...","예방, 보호, 순찰, 감시, 질서",13,5,재난·방범·범죄피해·응급·안전 관련 사업,business_content_rules_v2_refined_general_weak...
2,교육 보장,"교육, 학교, 대학, 인재, 장학, 평생교육, 학습, 진로, 독서, 도서관, 문해,...","역량, 연수, 프로그램, 강좌, 강사",13,5,교육·학습·장학·학교·도서관 관련 사업,business_content_rules_v2_refined_general_weak...
3,문화활동,"문화, 예술, 공연, 전시, 국악, 농악, 미술, 음악, 문학, 콘텐츠, 문화재, ...","축전, 한마당, 심포지엄",59,3,문화·예술·전통문화·공연·전시 관련 사업,business_content_rules_v2_refined_general_weak...
4,관광/휴양활동,"관광, 휴양, 축제, 여행, 탐방, 투어, 체험관광, 관광객, 관광지, 둘레길, 캠...","체험, 홍보, 마케팅, 방문, 코스",13,5,관광·휴양·축제·방문객 유치 관련 사업,business_content_rules_v2_refined_general_weak...
5,종교활동,"종교, 불교, 기독교, 천주교, 교회, 성당, 사찰, 절, 법회, 목사, 스님, 신앙","예배, 기도, 신도",12,3,종교 단체·종교행사 관련 사업,business_content_rules_v2_refined_general_weak...
6,환경향상,"환경, 생태, 기후, 탄소, 쓰레기, 폐기물, 재활용, 하천, 수질, 숲, 녹지, ...","자연, 보전, 개선, 정비",16,4,환경보전·생태·폐기물·자원순환 관련 사업,business_content_rules_v2_refined_general_weak...
7,사회복지향상,"복지, 복지관, 장애인, 노인, 노인회, 어르신, 아동, 청소년, 여성, 가족, 다...","종사자, 위문, 나눔",29,3,사회복지 대상자·돌봄·보육·취약계층 관련 사업,business_content_rules_v2_refined_general_weak...
8,보훈향상,"보훈, 유공자, 참전, 고엽제, 미망인, 수훈자, 군인회, 유족회, 전우회, 상이군...","위문, 추모, 기념",14,3,보훈·참전·유공자·유족 관련 사업,business_content_rules_v2_refined_general_weak...
9,고용안정,"고용, 일자리, 취업, 창업, 근로자, 노동, 청년창업, 직업, 훈련, 인력양성",청년,10,1,고용·취업·일자리·창업 관련 사업,business_content_rules_v2_refined_general_weak...


✅ 사업내용분류 v2 분류 함수 정의 완료
✅ 사업내용분류 v2 적용 완료
생성된 열: ['사업내용분류_대분류_v2', '사업내용분류_세분류_v2', '사업내용분류_후보분야_v2', '사업내용분류_분류근거_v2', '사업내용분류_검토필요여부_v2', '사업내용분류_분석포함여부_v2', '사업내용분류_rule_id_v2', '사업내용분류_강키워드_v2', '사업내용분류_약키워드_v2']

[사업내용분류 v2 대분류 분포]


,사업내용분류_대분류_v2,사업수,비율
0,사회복지향상,18705,23.60
1,기타,13326,16.81
2,문화활동,13178,16.63
3,1차 산업지원,8230,10.38
4,검토필요,5238,6.61
5,행정/통일/외교,3894,4.91
6,보훈향상,3083,3.89
7,교육 보장,2473,3.12
8,안전 보장,1932,2.44
9,산업/에너지지원,1547,1.95



[사업내용분류 v2 분석포함여부 분포]


...


[스마트팜 관련 v2 보정 사례]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v1,사업내용분류_대분류_v2,사업내용분류_분류근거_v2,평가결과,평가종류
2872,2872,경남_의령군.xlsx,경남,의령군,스마트팜 온실 신축사업,방송/통신진흥,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,우수,성과평가
5929,5929,경남_함양군.xlsx,경남,함양군,스마트팜 큐브 생산시설지원,방송/통신진흥,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우미흡,성과평가
6966,6966,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6967,6967,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6968,6968,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6969,6969,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6970,6970,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6971,6971,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6972,6972,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6973,6973,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가



[체육 관련 v2 보정 사례]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v1,사업내용분류_대분류_v2,사업내용분류_분류근거_v2,평가결과,평가종류
12,12,경남_남해군.xlsx,경남,남해군,남중권생활체육 교류 지원,검토필요,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,미흡,유지필요성평가
14,14,경남_남해군.xlsx,경남,남해군,레포츠 자격증 취득 지원,검토필요,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,미흡,유지필요성평가
73,73,경남_남해군.xlsx,경남,남해군,남해군수기 전국 검도대회,기타,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
74,74,경남_남해군.xlsx,경남,남해군,보물섬배 남해 전국 탁구대잔치,기타,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
75,75,경남_남해군.xlsx,경남,남해군,우수 체육단체 활동 지원,검토필요,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
77,77,경남_남해군.xlsx,경남,남해군,전문(엘리트) 체육 학교운동부 지원,교육 보장,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
78,78,경남_남해군.xlsx,경남,남해군,체육인 격려 지원(체육인의 밤 행사),검토필요,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
79,79,경남_남해군.xlsx,경남,남해군,초·중·고 체육유망주 지원,검토필요,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
80,80,경남_남해군.xlsx,경남,남해군,보물섬남해스포츠클럽 대회 참가 지원,검토필요,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
81,81,경남_남해군.xlsx,경남,남해군,장애인 생활체육 지원,사회복지향상,사회복지향상,우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,보통,유지필요성평가



[저장 완료]
- v2 규칙 요약표: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_rule_summary_v2_20260606_1912.xlsx
- v2 대분류/분석포함/검토그룹 요약: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_classification_summary_v2_20260606_1912.xlsx
- v1-v2 대분류 변화 비교: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_v1_v2_compare_20260606_1912.xlsx
- v2 검토필요·기타·복합 사례: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_review_cases_v2_20260606_1912.xlsx
- v2 분류근거별 행 수: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_busi

04B-7. v2 오분류 확인 기반 v3 소폭 보정
1. 이 단계의 목적

이 단계의 목적은 v2에서 확인된 특정 오분류 가능성을 소폭 보정하는 것입니다.

핵심 보정은 다음입니다.

| 문제                                                  | 보정 방향                                                                |
| --------------------------------------------------- | -------------------------------------------------------------------- |
| `군지부`, `군지회` 안의 `군지`가 지역사 편찬물로 오인됨                  | `군지/읍지/면지`는 단순 문자열 포함 매칭에서 제외                                        |
| `한국자유총연맹`, `바르게살기`, `새마을`, `민주평통`이 문화활동으로 잘못 갈 수 있음 | 행정/통일/외교 우선규칙을 문화활동보다 확실히 먼저 적용                                      |
| `편찬`, `발간`이 너무 넓게 문화활동으로 잡힐 수 있음                    | `문예지`, `향토지`, `읍지`, `면지`, `군지`, `백서`, `사료`, `역사` 등 맥락이 있을 때만 문화활동 처리 |


In [10]:
# 04B-7. v2 오분류 확인 기반 v3 소폭 보정

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 및 열 확인
# ------------------------------------------------------------

from copy import deepcopy

required_objects = [
    "df_04b",
    "business_content_rules_v2",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
    "safe_parse_tokens",
    "normalize_text_for_match",
    "get_token_set_lower",
    "has_token_any",
    "text_has_any",
    "match_keywords",
    "convert_df_for_excel",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1 → 04B-2 → 04B-3 → 04B-4 → 04B-5 → 04B-6을 먼저 실행하세요."
    )

required_cols_04b_7 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류_v2",
    "사업내용분류_분류근거_v2",
    "사업내용분류_검토필요여부_v2",
    "사업내용분류_분석포함여부_v2",
]

missing_cols_04b_7 = [
    col for col in required_cols_04b_7
    if col not in df_04b.columns
]

if missing_cols_04b_7:
    raise KeyError(
        f"04B-7 실행에 필요한 열이 없습니다: {missing_cols_04b_7}\n"
        "04B-6이 정상 실행되었는지 확인하세요."
    )

print("✅ 04B-7 실행 전 필수 객체 및 열 확인 완료")


# ------------------------------------------------------------
# 1) v3 규칙 생성
# ------------------------------------------------------------
# v2 규칙은 보존하고, v3 규칙을 별도로 만든다.

BUSINESS_CONTENT_RULE_VERSION_V3 = "business_content_rules_v3_fix_gunji_substring_admin_priority"

business_content_rules_v3 = deepcopy(business_content_rules_v2)


# ------------------------------------------------------------
# 1-1) 문화활동 strong에서 과민 substring 위험 키워드 제거
# ------------------------------------------------------------
# '군지', '읍지', '면지'는 지역사 편찬물일 때는 문화활동이 맞지만,
# '남해군지부', 'OO군지회' 같은 단체 지부/지회 안에서도 문자열로 잡힐 수 있다.
# 따라서 일반 strong 매칭에서는 제거하고, 아래 우선규칙에서 맥락 조건으로만 처리한다.

substring_risk_culture_terms = [
    "군지",
    "읍지",
    "면지",
    "편찬",
    "발간",
]

business_content_rules_v3["문화활동"]["strong"] = [
    term for term in business_content_rules_v3["문화활동"].get("strong", [])
    if term not in substring_risk_culture_terms
]

print("✅ v3 규칙 생성 완료")
print(f"규칙 버전: {BUSINESS_CONTENT_RULE_VERSION_V3}")


# ------------------------------------------------------------
# 2) v3 규칙 요약표 생성
# ------------------------------------------------------------

rule_rows_v3 = []

for category, rule in business_content_rules_v3.items():
    rule_rows_v3.append({
        "사업내용분류_대분류": category,
        "strong_keywords": ", ".join(rule.get("strong", [])),
        "weak_keywords": ", ".join(rule.get("weak", [])),
        "strong_keyword_count": len(rule.get("strong", [])),
        "weak_keyword_count": len(rule.get("weak", [])),
        "note": rule.get("note", ""),
        "rule_id": BUSINESS_CONTENT_RULE_VERSION_V3,
    })

business_content_rule_summary_v3 = pd.DataFrame(rule_rows_v3)

print("\n[사업내용분류 v3 규칙 요약표]")
display(business_content_rule_summary_v3)


# ------------------------------------------------------------
# 3) v3 결과 생성 함수
# ------------------------------------------------------------

def make_business_content_result_v3(
    category,
    reason,
    need_review,
    strong_matches,
    weak_matches,
    strong_categories,
    weak_categories
):
    """사업내용분류 v3 결과를 pd.Series로 반환"""

    if need_review is False:
        analysis_flag = "주분석_포함"
    elif category == "복합":
        analysis_flag = "보조분석_검토"
    else:
        analysis_flag = "해석주의"

    return pd.Series({
        "사업내용분류_대분류_v3": category,
        "사업내용분류_세분류_v3": "",
        "사업내용분류_후보분야_v3": ", ".join(sorted(set(strong_categories + weak_categories))),
        "사업내용분류_분류근거_v3": reason,
        "사업내용분류_검토필요여부_v3": bool(need_review),
        "사업내용분류_분석포함여부_v3": analysis_flag,
        "사업내용분류_rule_id_v3": BUSINESS_CONTENT_RULE_VERSION_V3,
        "사업내용분류_강키워드_v3": json.dumps(strong_matches, ensure_ascii=False),
        "사업내용분류_약키워드_v3": json.dumps(weak_matches, ensure_ascii=False),
    })


# ------------------------------------------------------------
# 4) 보조 함수: 지역사 편찬 맥락 판정
# ------------------------------------------------------------

def has_history_publication_context(token_set_lower, text_lower):
    """
    지역사·향토사·문예지·편찬/발간 맥락인지 판정한다.

    단순히 '군지', '읍지', '면지'가 문자열에 포함된다고 바로 문화활동으로 보내지 않는다.
    '군지부', '군지회', '시지부', '도지부' 등 단체 지부/지회 표현은 제외한다.
    """

    # 단체 지부/지회 표현이면 지역사 편찬 맥락으로 보지 않는다.
    org_branch_patterns = [
        "군지부", "군지회", "시지부", "시지회", "도지부", "도지회",
        "구지부", "구지회", "읍지부", "읍지회", "면지부", "면지회",
    ]

    if text_has_any(text_lower, org_branch_patterns):
        return False

    # 명확한 지역 기록·출판 맥락
    exact_history_tokens = [
        "문예지", "향토지", "읍지", "면지", "군지",
        "백서", "사료", "향토사", "지역사", "역사"
    ]

    if has_token_any(token_set_lower, exact_history_tokens):
        return True

    # 편찬/발간은 단독으로는 넓으므로 지역기록 맥락 단어와 함께 있을 때만 인정
    publication_terms = ["편찬", "발간"]
    context_terms = [
        "문예지", "향토", "향토지", "읍지", "면지", "군지",
        "백서", "사료", "역사", "지역사", "마을지", "면사", "군사", "읍사"
    ]

    has_publication = has_token_any(token_set_lower, publication_terms) or text_has_any(text_lower, publication_terms)
    has_context = has_token_any(token_set_lower, context_terms) or text_has_any(text_lower, context_terms)

    return has_publication and has_context


# ------------------------------------------------------------
# 5) v3 분류 함수
# ------------------------------------------------------------

def classify_business_content_v3(row):
    """
    사업내용분류 v3 분류 함수.

    v2 대비 보정:
    - 군지/읍지/면지 substring 오분류 방지
    - 한국자유총연맹/바르게살기/새마을 등 행정·시민단체 우선 처리
    - 지역사 편찬/발간은 맥락 조건을 충족할 때만 문화활동 처리
    """

    tokens = safe_parse_tokens(row.get("tokens_analysis_primary", []))
    token_set_lower = get_token_set_lower(tokens)
    text_lower = normalize_text_for_match(row.get("사업명_clean_spacefix", ""))

    strong_matches = {}
    weak_matches = {}

    for category, rule in business_content_rules_v3.items():
        strong_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("strong", [])
        )
        weak_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("weak", [])
        )

        if strong_hit:
            strong_matches[category] = strong_hit

        if weak_hit:
            weak_matches[category] = weak_hit

    strong_categories = list(strong_matches.keys())
    weak_categories = list(weak_matches.keys())


    # --------------------------------------------------------
    # 5-1) 우선순위 예외 규칙
    # --------------------------------------------------------

    # A. 행정/시민단체 계열은 문화활동보다 먼저 처리
    # 자유총연맹/바르게살기/새마을/민주평통/자원봉사 등은
    # 지부/지회 표현 때문에 문화활동으로 가지 않도록 우선 처리한다.
    admin_org_terms = [
        "한국자유총연맹", "자유총연맹", "바르게살기", "새마을",
        "민주평통", "평통", "자원봉사", "주민자치", "이장협의회",
        "통장협의회", "리장협의회"
    ]
    if has_token_any(token_set_lower, admin_org_terms) or text_has_any(text_lower, admin_org_terms):
        return make_business_content_result_v3(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체 조직명 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # B. 보훈 관련은 보훈향상 우선
    bohun_terms = [
        "보훈", "유공자", "참전", "고엽제", "미망인", "수훈자",
        "군인회", "유족회", "전우회", "상이군경", "광복회",
        "재향군인회", "전몰군경", "무공"
    ]
    if has_token_any(token_set_lower, bohun_terms) or text_has_any(text_lower, bohun_terms):
        return make_business_content_result_v3(
            category="보훈향상",
            reason="우선규칙: 보훈·참전·유공자·유족 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # C. 사회복지 대상자 및 복지서비스 관련은 사회복지향상 우선
    welfare_terms = [
        "복지", "장애인", "노인", "노인회", "어르신", "아동", "청소년",
        "여성", "가족", "다문화", "보육", "어린이집", "경로당",
        "돌봄", "자활", "재활", "취약", "저소득", "영유아", "한부모",
        "행복나눔", "희망나눔", "목욕", "이동목욕", "무료급식", "푸드뱅크"
    ]
    if has_token_any(token_set_lower, welfare_terms) or text_has_any(text_lower, welfare_terms):
        return make_business_content_result_v3(
            category="사회복지향상",
            reason="우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # D. 스마트팜/스마트 농업은 1차 산업지원 우선
    smart_agri_priority_terms = [
        "스마트팜", "스마트 농업", "스마트농업",
        "스마트 축산", "스마트축산",
        "스마트 양식", "스마트양식"
    ]
    if text_has_any(text_lower, smart_agri_priority_terms):
        return make_business_content_result_v3(
            category="1차 산업지원",
            reason="우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # E. 1차 산업 관련은 1차 산업지원 우선
    primary_industry_terms = [
        "농업", "농촌", "농가", "농산물", "농산", "농업인",
        "작물", "쌀", "마늘", "양파", "원예", "축산", "축산물",
        "가축", "한우", "수산", "수산물", "어업", "어촌",
        "임업", "산림", "귀농", "귀촌", "영농", "재배",
        "벼", "감자", "딸기", "양봉", "낙농", "병해충", "방제"
    ]
    if has_token_any(token_set_lower, primary_industry_terms) or text_has_any(text_lower, primary_industry_terms):
        return make_business_content_result_v3(
            category="1차 산업지원",
            reason="우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # F. 보건/의료 명확 키워드는 보건/의료 우선
    health_terms = [
        "보건", "의료", "병원", "건강", "치매", "감염", "질병",
        "예방접종", "정신건강", "위생", "의약", "진료", "환자", "검진"
    ]
    if has_token_any(token_set_lower, health_terms) or text_has_any(text_lower, health_terms):
        return make_business_content_result_v3(
            category="보건/의료",
            reason="우선규칙: 보건·의료·건강·감염·질병 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # G. 안전 관련은 안전 보장 우선
    safety_terms = [
        "안전", "재난", "방재", "소방", "방범", "범죄",
        "피해자", "구급", "응급", "교통질서", "모범운전자", "cctv", "CCTV"
    ]
    if has_token_any(token_set_lower, safety_terms) or text_has_any(text_lower, safety_terms):
        return make_business_content_result_v3(
            category="안전 보장",
            reason="우선규칙: 재난·방범·범죄피해·응급·안전 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # H. 교통/물류 명확 키워드는 교통/물류진흥 우선
    transport_terms = [
        "교통", "물류", "택시", "버스", "운수", "운송", "주차",
        "도로교통", "교통약자", "터미널", "카드결제", "단말기",
        "통신수수료", "수송"
    ]
    if has_token_any(token_set_lower, transport_terms) or text_has_any(text_lower, transport_terms):
        return make_business_content_result_v3(
            category="교통/물류진흥",
            reason="우선규칙: 교통·물류·운수·택시·버스 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # I. 주거 명확 키워드는 주거안정 우선
    housing_terms = [
        "주거", "주택", "공동주택", "빈집", "임대주택",
        "집수리", "주거환경", "마을공동시설"
    ]
    if has_token_any(token_set_lower, housing_terms) or text_has_any(text_lower, housing_terms):
        return make_business_content_result_v3(
            category="주거안정",
            reason="우선규칙: 주거·주택·공동주택·빈집 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # J. 종교활동은 종교 고유 맥락일 때 우선
    religion_terms = [
        "종교", "불교", "기독교", "천주교", "교회", "성당",
        "사찰", "법회", "목사", "스님", "신앙", "예배", "기도"
    ]
    if has_token_any(token_set_lower, religion_terms) or text_has_any(text_lower, religion_terms):
        return make_business_content_result_v3(
            category="종교활동",
            reason="우선규칙: 종교 단체·종교행사 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # K. 체육 관련 사업은 문화활동으로 운영화
    sports_culture_terms = [
        "체육", "생활체육", "스포츠", "레포츠", "선수",
        "축구", "야구", "농구", "배구", "탁구", "배드민턴",
        "골프", "게이트볼", "파크골프", "수영", "마라톤",
        "태권도", "테니스", "볼링", "검도", "궁도",
        "전지훈련", "동계훈련", "스토브리그", "유도대회", "체육대회"
    ]
    if has_token_any(token_set_lower, sports_culture_terms) or text_has_any(text_lower, sports_culture_terms):
        return make_business_content_result_v3(
            category="문화활동",
            reason="우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로 운영화",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # L. 지역사 편찬/문예지 등은 맥락 조건을 충족할 때만 문화활동
    if has_history_publication_context(token_set_lower, text_lower):
        return make_business_content_result_v3(
            category="문화활동",
            reason="우선규칙: 지역사·향토사·문예지 편찬/발간 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # M. 유림/향교/전통문화 맥락은 문화활동 우선
    traditional_culture_terms = [
        "동학", "유림", "성균관유도회", "유도회",
        "향교", "전승", "전통", "민속"
    ]
    if has_token_any(token_set_lower, traditional_culture_terms) or text_has_any(text_lower, traditional_culture_terms):
        return make_business_content_result_v3(
            category="문화활동",
            reason="우선규칙: 전통문화·유림·향교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # N. 관광/휴양은 관광·휴양·방문객 맥락일 때 우선
    tourism_terms = [
        "관광", "휴양", "여행", "탐방", "투어", "체험관광",
        "관광객", "관광지", "둘레길", "캠핑", "해수욕장"
    ]
    if has_token_any(token_set_lower, tourism_terms) or text_has_any(text_lower, tourism_terms):
        return make_business_content_result_v3(
            category="관광/휴양활동",
            reason="우선규칙: 관광·휴양·방문객 유치 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # O. 문화활동 명확 키워드
    culture_terms = [
        "문화", "예술", "공연", "전시", "국악", "농악",
        "미술", "음악", "문학", "콘텐츠", "문화재",
        "박물관", "향교", "제례", "석전대제",
        "민속", "가요제", "가요", "예총"
    ]
    if has_token_any(token_set_lower, culture_terms) or text_has_any(text_lower, culture_terms):
        return make_business_content_result_v3(
            category="문화활동",
            reason="우선규칙: 문화·예술·공연·전시 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # P. 행정/통일/외교 명확 키워드
    admin_terms = [
        "행정", "통일", "민주평통", "평화", "민주", "인권",
        "주민자치", "자치", "이장", "새마을", "바르게살기",
        "자유총연맹", "자원봉사", "봉사", "국제교류", "외교"
    ]
    if has_token_any(token_set_lower, admin_terms) or text_has_any(text_lower, admin_terms):
        return make_business_content_result_v3(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체·통일·평화·외교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # --------------------------------------------------------
    # 5-2) 일반 규칙
    # --------------------------------------------------------

    # strong이 1개 분야에만 잡힌 경우
    if len(strong_categories) == 1:
        main_category = strong_categories[0]
        return make_business_content_result_v3(
            category=main_category,
            reason=f"강키워드 단일 분야 매칭: {main_category}({', '.join(strong_matches[main_category])})",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # strong이 여러 분야에 잡힌 경우
    if len(strong_categories) >= 2:
        return make_business_content_result_v3(
            category="복합",
            reason=f"여러 분야 강키워드 동시 매칭: {', '.join(strong_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # weak만 잡힌 경우
    if len(strong_categories) == 0 and len(weak_categories) >= 1:
        return make_business_content_result_v3(
            category="검토필요",
            reason=f"약키워드만 매칭: {', '.join(weak_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # 아무것도 안 잡힌 경우
    return make_business_content_result_v3(
        category="기타",
        reason="매칭 키워드 없음",
        need_review=True,
        strong_matches=strong_matches,
        weak_matches=weak_matches,
        strong_categories=strong_categories,
        weak_categories=weak_categories
    )


print("✅ 사업내용분류 v3 분류 함수 정의 완료")


# ------------------------------------------------------------
# 6) 전체 데이터에 사업내용분류 v3 적용
# ------------------------------------------------------------

classification_result_v3 = df_04b.apply(classify_business_content_v3, axis=1)

for col in classification_result_v3.columns:
    df_04b[col] = classification_result_v3[col]

print("✅ 사업내용분류 v3 적용 완료")
print(f"생성된 열: {classification_result_v3.columns.tolist()}")


# ------------------------------------------------------------
# 7) v3 대분류 분포
# ------------------------------------------------------------

business_content_classification_summary_v3 = (
    df_04b["사업내용분류_대분류_v3"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_대분류_v3")
    .reset_index(name="사업수")
)

business_content_classification_summary_v3["비율"] = (
    business_content_classification_summary_v3["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v3 대분류 분포]")
display(business_content_classification_summary_v3)


# ------------------------------------------------------------
# 8) v3 분석포함여부 분포
# ------------------------------------------------------------

business_content_analysis_flag_summary_v3 = (
    df_04b["사업내용분류_분석포함여부_v3"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분석포함여부_v3")
    .reset_index(name="사업수")
)

business_content_analysis_flag_summary_v3["비율"] = (
    business_content_analysis_flag_summary_v3["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v3 분석포함여부 분포]")
display(business_content_analysis_flag_summary_v3)


# ------------------------------------------------------------
# 9) v2-v3 변화 비교
# ------------------------------------------------------------

business_content_v2_v3_compare = (
    df_04b
    .groupby(["사업내용분류_대분류_v2", "사업내용분류_대분류_v3"])
    .size()
    .reset_index(name="사업수")
    .sort_values("사업수", ascending=False)
)

print("\n[사업내용분류 v2 → v3 변화 상위 60개]")
display(business_content_v2_v3_compare.head(60))


# ------------------------------------------------------------
# 10) v2-v3 기타/검토필요/복합 비교
# ------------------------------------------------------------

review_target_values = ["기타", "검토필요", "복합"]

def count_business_content_review_group(df, col, version):
    return pd.DataFrame({
        "버전": version,
        "구분": ["기타", "검토필요", "복합", "합계"],
        "사업수": [
            int((df[col] == "기타").sum()),
            int((df[col] == "검토필요").sum()),
            int((df[col] == "복합").sum()),
            int(df[col].isin(review_target_values).sum()),
        ]
    })

business_content_review_group_compare_v2_v3 = pd.concat(
    [
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v2", "v2"),
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v3", "v3"),
    ],
    axis=0
)

business_content_review_group_compare_v2_v3["비율"] = (
    business_content_review_group_compare_v2_v3["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v2-v3 기타/검토필요/복합 비교]")
display(business_content_review_group_compare_v2_v3)


# ------------------------------------------------------------
# 11) v3 분류근거별 행 수
# ------------------------------------------------------------

business_content_reason_summary_v3 = (
    df_04b["사업내용분류_분류근거_v3"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분류근거_v3")
    .reset_index(name="사업수")
)

business_content_reason_summary_v3["비율"] = (
    business_content_reason_summary_v3["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v3 분류근거별 행 수 상위 40개]")
display(business_content_reason_summary_v3.head(40))


# ------------------------------------------------------------
# 12) v2-v3 변경 사례
# ------------------------------------------------------------

business_content_changed_cases_v2_v3 = df_04b[
    df_04b["사업내용분류_대분류_v2"] != df_04b["사업내용분류_대분류_v3"]
].copy()

changed_case_cols_v2_v3 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "사업내용분류_대분류_v2",
    "사업내용분류_분류근거_v2",
    "사업내용분류_대분류_v3",
    "사업내용분류_분류근거_v3",
    "평가결과",
    "평가종류",
]

changed_case_cols_v2_v3 = [
    col for col in changed_case_cols_v2_v3
    if col in business_content_changed_cases_v2_v3.columns
]

print(f"\n[v2-v3 대분류 변경 사례 수] {len(business_content_changed_cases_v2_v3):,}행")
display(business_content_changed_cases_v2_v3[changed_case_cols_v2_v3].head(100))


# ------------------------------------------------------------
# 13) 행정단체 계열 점검
# ------------------------------------------------------------

admin_org_pattern = (
    "자유총연맹|한국자유총연맹|바르게살기|새마을|민주평통|평통|자원봉사|주민자치"
)

admin_org_check_v3 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(admin_org_pattern, regex=True, na=False)
].copy()

admin_org_check_cols = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업내용분류_대분류_v2",
    "사업내용분류_대분류_v3",
    "사업내용분류_분류근거_v3",
    "평가결과",
    "평가종류",
]

admin_org_check_cols = [
    col for col in admin_org_check_cols
    if col in admin_org_check_v3.columns
]

print("\n[행정단체 계열 v3 보정 점검]")
display(admin_org_check_v3[admin_org_check_cols].head(80))


# ------------------------------------------------------------
# 14) 지역사 편찬/발간 계열 점검
# ------------------------------------------------------------

history_publication_pattern = (
    "문예지|향토지|읍지|면지|군지|백서|사료|향토사|지역사|편찬|발간"
)

history_publication_check_v3 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(history_publication_pattern, regex=True, na=False)
].copy()

history_publication_check_cols = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업내용분류_대분류_v2",
    "사업내용분류_대분류_v3",
    "사업내용분류_분류근거_v3",
    "평가결과",
    "평가종류",
]

history_publication_check_cols = [
    col for col in history_publication_check_cols
    if col in history_publication_check_v3.columns
]

print("\n[지역사 편찬/발간 계열 v3 보정 점검]")
display(history_publication_check_v3[history_publication_check_cols].head(100))


# ------------------------------------------------------------
# 15) 저장
# ------------------------------------------------------------

rule_summary_v3_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_rule_summary_v3_{RUN_TS}.xlsx"
)

classification_summary_v3_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_classification_summary_v3_{RUN_TS}.xlsx"
)

v2_v3_compare_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_v2_v3_compare_{RUN_TS}.xlsx"
)

changed_cases_v2_v3_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_changed_cases_v2_v3_{RUN_TS}.xlsx"
)

reason_summary_v3_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_reason_summary_v3_{RUN_TS}.xlsx"
)

admin_org_check_v3_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_admin_org_check_v3_{RUN_TS}.xlsx"
)

history_publication_check_v3_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_history_publication_check_v3_{RUN_TS}.xlsx"
)

business_content_rule_summary_v3.to_excel(rule_summary_v3_path, index=False)

with pd.ExcelWriter(classification_summary_v3_path, engine="openpyxl") as writer:
    business_content_classification_summary_v3.to_excel(
        writer,
        sheet_name="classification_summary_v3",
        index=False
    )
    business_content_analysis_flag_summary_v3.to_excel(
        writer,
        sheet_name="analysis_flag_summary_v3",
        index=False
    )
    business_content_review_group_compare_v2_v3.to_excel(
        writer,
        sheet_name="review_group_compare",
        index=False
    )

business_content_v2_v3_compare.to_excel(v2_v3_compare_path, index=False)
convert_df_for_excel(business_content_changed_cases_v2_v3[changed_case_cols_v2_v3]).to_excel(changed_cases_v2_v3_path, index=False)
business_content_reason_summary_v3.to_excel(reason_summary_v3_path, index=False)
convert_df_for_excel(admin_org_check_v3[admin_org_check_cols]).to_excel(admin_org_check_v3_path, index=False)
convert_df_for_excel(history_publication_check_v3[history_publication_check_cols]).to_excel(history_publication_check_v3_path, index=False)

print("\n[저장 완료]")
print(f"- v3 규칙 요약표: {rule_summary_v3_path}")
print(f"- v3 대분류/분석포함/검토그룹 요약: {classification_summary_v3_path}")
print(f"- v2-v3 대분류 변화 비교: {v2_v3_compare_path}")
print(f"- v2-v3 변경 사례: {changed_cases_v2_v3_path}")
print(f"- v3 분류근거별 행 수: {reason_summary_v3_path}")
print(f"- 행정단체 계열 점검: {admin_org_check_v3_path}")
print(f"- 지역사 편찬/발간 계열 점검: {history_publication_check_v3_path}")

print("\n✅ 04B-7 v2 오분류 확인 기반 v3 소폭 보정 완료")

✅ 04B-7 실행 전 필수 객체 및 열 확인 완료
✅ v3 규칙 생성 완료
규칙 버전: business_content_rules_v3_fix_gunji_substring_admin_priority

[사업내용분류 v3 규칙 요약표]


,사업내용분류_대분류,strong_keywords,weak_keywords,strong_keyword_count,weak_keyword_count,note,rule_id
0,행정/통일/외교,"행정, 통일, 민주평통, 평화, 민주, 인권, 주민자치, 자치, 이장, 새마을, 바...",,16,0,행정·시민단체·통일·평화·외교 관련 사업,business_content_rules_v3_fix_gunji_substring_...
1,안전 보장,"안전, 재난, 방재, 소방, 방범, 범죄, 피해자, 구급, 응급, 교통질서, 모범운...","예방, 보호, 순찰, 감시, 질서",13,5,재난·방범·범죄피해·응급·안전 관련 사업,business_content_rules_v3_fix_gunji_substring_...
2,교육 보장,"교육, 학교, 대학, 인재, 장학, 평생교육, 학습, 진로, 독서, 도서관, 문해,...","역량, 연수, 프로그램, 강좌, 강사",13,5,교육·학습·장학·학교·도서관 관련 사업,business_content_rules_v3_fix_gunji_substring_...
3,문화활동,"문화, 예술, 공연, 전시, 국악, 농악, 미술, 음악, 문학, 콘텐츠, 문화재, ...","축전, 한마당, 심포지엄",54,3,문화·예술·전통문화·공연·전시 관련 사업,business_content_rules_v3_fix_gunji_substring_...
4,관광/휴양활동,"관광, 휴양, 축제, 여행, 탐방, 투어, 체험관광, 관광객, 관광지, 둘레길, 캠...","체험, 홍보, 마케팅, 방문, 코스",13,5,관광·휴양·축제·방문객 유치 관련 사업,business_content_rules_v3_fix_gunji_substring_...
5,종교활동,"종교, 불교, 기독교, 천주교, 교회, 성당, 사찰, 절, 법회, 목사, 스님, 신앙","예배, 기도, 신도",12,3,종교 단체·종교행사 관련 사업,business_content_rules_v3_fix_gunji_substring_...
6,환경향상,"환경, 생태, 기후, 탄소, 쓰레기, 폐기물, 재활용, 하천, 수질, 숲, 녹지, ...","자연, 보전, 개선, 정비",16,4,환경보전·생태·폐기물·자원순환 관련 사업,business_content_rules_v3_fix_gunji_substring_...
7,사회복지향상,"복지, 복지관, 장애인, 노인, 노인회, 어르신, 아동, 청소년, 여성, 가족, 다...","종사자, 위문, 나눔",29,3,사회복지 대상자·돌봄·보육·취약계층 관련 사업,business_content_rules_v3_fix_gunji_substring_...
8,보훈향상,"보훈, 유공자, 참전, 고엽제, 미망인, 수훈자, 군인회, 유족회, 전우회, 상이군...","위문, 추모, 기념",14,3,보훈·참전·유공자·유족 관련 사업,business_content_rules_v3_fix_gunji_substring_...
9,고용안정,"고용, 일자리, 취업, 창업, 근로자, 노동, 청년창업, 직업, 훈련, 인력양성",청년,10,1,고용·취업·일자리·창업 관련 사업,business_content_rules_v3_fix_gunji_substring_...


✅ 사업내용분류 v3 분류 함수 정의 완료
✅ 사업내용분류 v3 적용 완료
생성된 열: ['사업내용분류_대분류_v3', '사업내용분류_세분류_v3', '사업내용분류_후보분야_v3', '사업내용분류_분류근거_v3', '사업내용분류_검토필요여부_v3', '사업내용분류_분석포함여부_v3', '사업내용분류_rule_id_v3', '사업내용분류_강키워드_v3', '사업내용분류_약키워드_v3']

[사업내용분류 v3 대분류 분포]


,사업내용분류_대분류_v3,사업수,비율
0,사회복지향상,18575,23.43
1,기타,13395,16.90
2,문화활동,12902,16.28
3,1차 산업지원,8220,10.37
4,검토필요,5226,6.59
5,행정/통일/외교,4262,5.38
6,보훈향상,3080,3.89
7,교육 보장,2478,3.13
8,안전 보장,1918,2.42
9,산업/에너지지원,1554,1.96



[사업내용분류 v3 분석포함여부 분포]


...


[행정단체 계열 v3 보정 점검]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v2,사업내용분류_대분류_v3,사업내용분류_분류근거_v3,평가결과,평가종류
24,24,경남_남해군.xlsx,경남,남해군,제13회 새마을의 날 기념식,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,보통,유지필요성평가
25,25,경남_남해군.xlsx,경남,남해군,2023년 남해군 새마을지도자 대회,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,보통,유지필요성평가
26,26,경남_남해군.xlsx,경남,남해군,새마을지도자 화합한마음대회,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,보통,유지필요성평가
29,29,경남_남해군.xlsx,경남,남해군,바르게살기운동남해군협의회 운영비 지원,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,보통,유지필요성평가
30,30,경남_남해군.xlsx,경남,남해군,한국자유총연맹남해군지부 운영비 지원,문화활동,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,보통,유지필요성평가
...,...,...,...,...,...,...,...,...,...,...
1545,1545,경남_양산시.xlsx,경남,양산시,바르게살기운동 한마음 활성화대회,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,매우우수,성과평가
1548,1548,경남_양산시.xlsx,경남,양산시,한국자유총연맹양산시지회 운영비,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,우수,성과평가
1549,1549,경남_양산시.xlsx,경남,양산시,자유총연맹 안보강연회,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,우수,성과평가
1550,1550,경남_양산시.xlsx,경남,양산시,자유총연맹 도민통합 한마음대회,행정/통일/외교,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,우수,성과평가



[지역사 편찬/발간 계열 v3 보정 점검]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v2,사업내용분류_대분류_v3,사업내용분류_분류근거_v3,평가결과,평가종류
30,30,경남_남해군.xlsx,경남,남해군,한국자유총연맹남해군지부 운영비 지원,문화활동,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,보통,유지필요성평가
42,42,경남_남해군.xlsx,경남,남해군,지역사회보장협의체 운영 활성화 지원,문화활동,문화활동,강키워드 단일 분야 매칭: 문화활동(역사),보통,유지필요성평가
62,62,경남_남해군.xlsx,경남,남해군,향토사발굴수집조사연구 및 책자발간,문화활동,문화활동,우선규칙: 지역사·향토사·문예지 편찬/발간 맥락,보통,유지필요성평가
111,111,경남_남해군.xlsx,경남,남해군,새마을운동남해군지회 운영비 지원,문화활동,행정/통일/외교,우선규칙: 행정·시민단체 조직명 관련 키워드,우수,유지필요성평가
158,158,경남_남해군.xlsx,경남,남해군,"미조하나, 에덴선교어린이집 운영비 지원(면지역)",사회복지향상,사회복지향상,우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,보통,성과평가
...,...,...,...,...,...,...,...,...,...,...
3618,3618,경남_거창군.xlsx,경남,거창군,조사료 사일리지 절단기 지원,기타,기타,매칭 키워드 없음,보통,성과평가
3619,3619,경남_거창군.xlsx,경남,거창군,조사료 생산용 종자구입 지원(자체재원),검토필요,검토필요,약키워드만 매칭: 1차 산업지원,보통,성과평가
3667,3667,경남_합천군.xlsx,경남,합천군,친환경 축산물 농가 사료 운송 비 지원,1차 산업지원,1차 산업지원,우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,매우미흡,유지필요성평가
3704,3704,경남_합천군.xlsx,경남,합천군,향토사 연구소,문화활동,문화활동,우선규칙: 지역사·향토사·문예지 편찬/발간 맥락,미흡,유지필요성평가



[저장 완료]
- v3 규칙 요약표: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_rule_summary_v3_20260606_1912.xlsx
- v3 대분류/분석포함/검토그룹 요약: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_classification_summary_v3_20260606_1912.xlsx
- v2-v3 대분류 변화 비교: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_v2_v3_compare_20260606_1912.xlsx
- v2-v3 변경 사례: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_changed_cases_v2_v3_20260606_1912.xlsx
- v3 분류근거별 행 수: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_busin

04B-8. v3 점검 기반 v4 최종 후보 생성
1. 이 단계의 목적

이 단계의 목적은 v3 결과를 보존한 상태에서, 최종 handoff 직전 사용할 v4 최종 후보 분류를 생성하는 것입니다.

핵심 보정은 다음입니다.

1. 지역사회 안의 역사 substring 오분류 제거
2. 역사는 단독 문자열 포함 매칭에서 제외
3. 편찬, 발간, 문예지, 향토사, 총람 등 명확한 지역기록 맥락만 문화활동 처리
4. 새마을, 바르게살기, 자유총연맹 등 행정단체명은 운영비·지부·지회 등 조직운영 맥락일 때만 우선적으로 행정/통일/외교 처리
5. 행정단체명이 있어도 노인, 장애인, 독서, 문화, 체육 등 실질 사업내용이 명확하면 해당 내용 분야 우선

In [11]:
# 04B-8. v3 점검 기반 v4 최종 후보 생성

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 및 열 확인
# ------------------------------------------------------------

from copy import deepcopy

required_objects = [
    "df_04b",
    "business_content_rules_v3",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
    "safe_parse_tokens",
    "normalize_text_for_match",
    "get_token_set_lower",
    "has_token_any",
    "text_has_any",
    "match_keywords",
    "convert_df_for_excel",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1 → 04B-2 → 04B-3 → 04B-4 → 04B-5 → 04B-6 → 04B-7을 먼저 실행하세요."
    )

required_cols_04b_8 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류_v3",
    "사업내용분류_분류근거_v3",
    "사업내용분류_검토필요여부_v3",
    "사업내용분류_분석포함여부_v3",
]

missing_cols_04b_8 = [
    col for col in required_cols_04b_8
    if col not in df_04b.columns
]

if missing_cols_04b_8:
    raise KeyError(
        f"04B-8 실행에 필요한 열이 없습니다: {missing_cols_04b_8}\n"
        "04B-7이 정상 실행되었는지 확인하세요."
    )

print("✅ 04B-8 실행 전 필수 객체 및 열 확인 완료")


# ------------------------------------------------------------
# 1) v4 규칙 생성
# ------------------------------------------------------------
# v3 규칙은 보존하고, v4 규칙을 별도로 만든다.

BUSINESS_CONTENT_RULE_VERSION_V4 = "business_content_rules_v4_final_candidate_fix_history_substring_admin_content_priority"

business_content_rules_v4 = deepcopy(business_content_rules_v3)


# ------------------------------------------------------------
# 1-1) 문화활동 strong에서 '역사' 제거
# ------------------------------------------------------------
# '지역사회' 안의 '역사'가 문화활동으로 잡히는 substring 오분류를 막기 위함.
# 역사 관련 사업은 아래 has_history_publication_context_v4()에서 맥락 조건으로만 처리한다.

for risky_term in ["역사"]:
    if risky_term in business_content_rules_v4["문화활동"]["strong"]:
        business_content_rules_v4["문화활동"]["strong"].remove(risky_term)

print("✅ v4 규칙 생성 완료")
print(f"규칙 버전: {BUSINESS_CONTENT_RULE_VERSION_V4}")


# ------------------------------------------------------------
# 2) v4 규칙 요약표 생성
# ------------------------------------------------------------

rule_rows_v4 = []

for category, rule in business_content_rules_v4.items():
    rule_rows_v4.append({
        "사업내용분류_대분류": category,
        "strong_keywords": ", ".join(rule.get("strong", [])),
        "weak_keywords": ", ".join(rule.get("weak", [])),
        "strong_keyword_count": len(rule.get("strong", [])),
        "weak_keyword_count": len(rule.get("weak", [])),
        "note": rule.get("note", ""),
        "rule_id": BUSINESS_CONTENT_RULE_VERSION_V4,
    })

business_content_rule_summary_v4 = pd.DataFrame(rule_rows_v4)

print("\n[사업내용분류 v4 규칙 요약표]")
display(business_content_rule_summary_v4)


# ------------------------------------------------------------
# 3) v4 결과 생성 함수
# ------------------------------------------------------------

def make_business_content_result_v4(
    category,
    reason,
    need_review,
    strong_matches,
    weak_matches,
    strong_categories,
    weak_categories
):
    """사업내용분류 v4 결과를 pd.Series로 반환"""

    if need_review is False:
        analysis_flag = "주분석_포함"
    elif category == "복합":
        analysis_flag = "보조분석_검토"
    else:
        analysis_flag = "해석주의"

    return pd.Series({
        "사업내용분류_대분류_v4": category,
        "사업내용분류_세분류_v4": "",
        "사업내용분류_후보분야_v4": ", ".join(sorted(set(strong_categories + weak_categories))),
        "사업내용분류_분류근거_v4": reason,
        "사업내용분류_검토필요여부_v4": bool(need_review),
        "사업내용분류_분석포함여부_v4": analysis_flag,
        "사업내용분류_rule_id_v4": BUSINESS_CONTENT_RULE_VERSION_V4,
        "사업내용분류_강키워드_v4": json.dumps(strong_matches, ensure_ascii=False),
        "사업내용분류_약키워드_v4": json.dumps(weak_matches, ensure_ascii=False),
    })


# ------------------------------------------------------------
# 4) 보조 함수: 행정단체 운영 맥락 판정
# ------------------------------------------------------------

def has_admin_org(text_lower, token_set_lower):
    """새마을·바르게살기·자유총연맹 등 행정/시민단체 조직명 존재 여부"""

    admin_org_terms = [
        "한국자유총연맹", "자유총연맹",
        "바르게살기",
        "새마을",
        "민주평통", "평통",
        "자원봉사",
        "주민자치",
        "이장협의회", "통장협의회", "리장협의회",
    ]

    return has_token_any(token_set_lower, admin_org_terms) or text_has_any(text_lower, admin_org_terms)


def has_admin_operation_context(text_lower, token_set_lower):
    """
    행정단체 조직 운영 맥락인지 판정한다.
    '운영비', '지부', '지회', '사무실', '집기' 등은 단체 운영 성격이 강하다.
    """

    admin_operation_terms = [
        "운영비", "사무실", "사무국", "사무처", "집기",
        "지부", "지회", "협의회 운영", "단체운영", "조직운영",
        "기념식", "총회", "회관", "보수 및 집기", "운영 지원",
    ]

    return has_token_any(token_set_lower, admin_operation_terms) or text_has_any(text_lower, admin_operation_terms)


def has_substantive_policy_content(text_lower, token_set_lower):
    """
    행정단체명이 있더라도 실질 사업내용이 명확한지 판정한다.
    예: 노인/장애인/복지/독서/문화/체육/농업/안전 등.
    """

    substantive_terms = [
        # 복지/보훈
        "복지", "장애인", "노인", "어르신", "아동", "청소년",
        "여성", "다문화", "보육", "어린이집", "경로당", "돌봄",
        "무료급식", "목욕", "이동목욕", "취약", "저소득",
        "보훈", "유공자", "참전",

        # 교육/문화/체육
        "교육", "학교", "장학", "학습", "독서", "도서관", "문해",
        "문화", "예술", "공연", "전시", "음악", "미술", "국악",
        "체육", "생활체육", "스포츠", "레포츠", "축구", "야구",
        "배드민턴", "탁구", "골프", "태권도", "검도",

        # 농림축산/수산
        "농업", "농촌", "농가", "농산물", "축산", "수산", "어업",
        "산림", "영농", "스마트팜",

        # 보건/안전/환경/주거/교통/산업
        "보건", "의료", "건강", "치매", "감염", "질병",
        "안전", "재난", "방범", "소방", "범죄",
        "환경", "생태", "쓰레기", "재활용",
        "주거", "주택", "교통", "택시", "버스",
        "일자리", "취업", "창업", "기업", "소상공인", "전통시장",
    ]

    return has_token_any(token_set_lower, substantive_terms) or text_has_any(text_lower, substantive_terms)


# ------------------------------------------------------------
# 5) 보조 함수: 지역사 편찬/발간 맥락 판정 v4
# ------------------------------------------------------------

def has_history_publication_context_v4(token_set_lower, text_lower):
    """
    지역사·향토사·문예지·총람·편찬/발간 맥락인지 판정한다.

    v4 보정:
    - '지역사회' 안의 '역사'는 역사로 보지 않는다.
    - '군지부', '군지회' 안의 '군지'는 지역사 편찬물로 보지 않는다.
    - '사료'는 조사료 같은 농업 단어와 섞일 수 있으므로 token exact일 때만 인정한다.
    """

    # 1) 명시적 제외 패턴
    false_positive_patterns = [
        "지역사회", "지역사회보장", "지역사회보장협의체",
        "군지부", "군지회", "시지부", "시지회", "도지부", "도지회",
        "구지부", "구지회", "읍지부", "읍지회", "면지부", "면지회",
        "조사료",
    ]

    if text_has_any(text_lower, false_positive_patterns):
        return False

    # 2) 정확 토큰으로 인정할 지역기록/출판 관련 표현
    exact_history_tokens = [
        "문예지", "향토지", "읍지", "면지", "군지",
        "백서", "사료", "향토사", "지역사", "역사",
        "마을지", "군사", "읍사", "면사", "총람", "금석문총람"
    ]

    if has_token_any(token_set_lower, exact_history_tokens):
        return True

    # 3) 편찬/발간은 지역기록 맥락 단어와 함께 있을 때만 인정
    publication_terms = ["편찬", "발간", "재발간"]
    context_terms = [
        "문예지", "향토", "향토지", "읍지", "면지", "군지",
        "백서", "사료", "지역사", "향토사", "마을지", "총람",
        "금석문", "자료집", "문화원"
    ]

    has_publication = has_token_any(token_set_lower, publication_terms) or text_has_any(text_lower, publication_terms)
    has_context = has_token_any(token_set_lower, context_terms) or text_has_any(text_lower, context_terms)

    return has_publication and has_context


# ------------------------------------------------------------
# 6) v4 분류 함수
# ------------------------------------------------------------

def classify_business_content_v4(row):
    """
    사업내용분류 v4 분류 함수.

    v3 대비 보정:
    - 지역사회 안의 '역사' substring 오분류 제거
    - 행정단체명은 조직 운영 맥락일 때만 우선 적용
    - 행정단체명이 있어도 실질 사업내용이 명확하면 내용 분야 우선
    """

    tokens = safe_parse_tokens(row.get("tokens_analysis_primary", []))
    token_set_lower = get_token_set_lower(tokens)
    text_lower = normalize_text_for_match(row.get("사업명_clean_spacefix", ""))

    strong_matches = {}
    weak_matches = {}

    for category, rule in business_content_rules_v4.items():
        strong_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("strong", [])
        )
        weak_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("weak", [])
        )

        if strong_hit:
            strong_matches[category] = strong_hit

        if weak_hit:
            weak_matches[category] = weak_hit

    strong_categories = list(strong_matches.keys())
    weak_categories = list(weak_matches.keys())

    admin_org_present = has_admin_org(text_lower, token_set_lower)
    admin_operation_present = has_admin_operation_context(text_lower, token_set_lower)
    substantive_content_present = has_substantive_policy_content(text_lower, token_set_lower)


    # --------------------------------------------------------
    # 6-1) 행정단체 조직 운영 맥락
    # --------------------------------------------------------
    # 단체명 + 운영비/지부/지회/사무실/집기 등은 행정/통일/외교로 우선 처리한다.
    # 단, 명확한 실질 정책내용이 있고 운영 맥락이 아니면 아래 내용 분야 규칙으로 넘긴다.

    if admin_org_present and (admin_operation_present or not substantive_content_present):
        return make_business_content_result_v4(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체 조직 운영 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )


    # --------------------------------------------------------
    # 6-2) 실질 사업내용 우선 규칙
    # --------------------------------------------------------

    # A. 보훈 관련은 보훈향상 우선
    bohun_terms = [
        "보훈", "유공자", "참전", "고엽제", "미망인", "수훈자",
        "군인회", "유족회", "전우회", "상이군경", "광복회",
        "재향군인회", "전몰군경", "무공"
    ]
    if has_token_any(token_set_lower, bohun_terms) or text_has_any(text_lower, bohun_terms):
        return make_business_content_result_v4(
            category="보훈향상",
            reason="우선규칙: 보훈·참전·유공자·유족 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # B. 사회복지 대상자 및 복지서비스 관련은 사회복지향상 우선
    welfare_terms = [
        "복지", "장애인", "노인", "노인회", "어르신", "아동", "청소년",
        "여성", "가족", "다문화", "보육", "어린이집", "경로당",
        "돌봄", "자활", "재활", "취약", "저소득", "영유아", "한부모",
        "행복나눔", "희망나눔", "목욕", "이동목욕", "무료급식", "푸드뱅크"
    ]
    if has_token_any(token_set_lower, welfare_terms) or text_has_any(text_lower, welfare_terms):
        return make_business_content_result_v4(
            category="사회복지향상",
            reason="우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # C. 스마트팜/스마트 농업은 1차 산업지원 우선
    smart_agri_priority_terms = [
        "스마트팜", "스마트 농업", "스마트농업",
        "스마트 축산", "스마트축산",
        "스마트 양식", "스마트양식"
    ]
    if text_has_any(text_lower, smart_agri_priority_terms):
        return make_business_content_result_v4(
            category="1차 산업지원",
            reason="우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # D. 1차 산업 관련은 1차 산업지원 우선
    primary_industry_terms = [
        "농업", "농촌", "농가", "농산물", "농산", "농업인",
        "작물", "쌀", "마늘", "양파", "원예", "축산", "축산물",
        "가축", "한우", "수산", "수산물", "어업", "어촌",
        "임업", "산림", "귀농", "귀촌", "영농", "재배",
        "벼", "감자", "딸기", "양봉", "낙농", "병해충", "방제"
    ]
    if has_token_any(token_set_lower, primary_industry_terms) or text_has_any(text_lower, primary_industry_terms):
        return make_business_content_result_v4(
            category="1차 산업지원",
            reason="우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # E. 보건/의료 명확 키워드는 보건/의료 우선
    health_terms = [
        "보건", "의료", "병원", "건강", "치매", "감염", "질병",
        "예방접종", "정신건강", "위생", "의약", "진료", "환자", "검진"
    ]
    if has_token_any(token_set_lower, health_terms) or text_has_any(text_lower, health_terms):
        return make_business_content_result_v4(
            category="보건/의료",
            reason="우선규칙: 보건·의료·건강·감염·질병 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # F. 안전 관련은 안전 보장 우선
    safety_terms = [
        "안전", "재난", "방재", "소방", "방범", "범죄",
        "피해자", "구급", "응급", "교통질서", "모범운전자", "cctv", "CCTV"
    ]
    if has_token_any(token_set_lower, safety_terms) or text_has_any(text_lower, safety_terms):
        return make_business_content_result_v4(
            category="안전 보장",
            reason="우선규칙: 재난·방범·범죄피해·응급·안전 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # G. 교통/물류 명확 키워드는 교통/물류진흥 우선
    transport_terms = [
        "교통", "물류", "택시", "버스", "운수", "운송", "주차",
        "도로교통", "교통약자", "터미널", "카드결제", "단말기",
        "통신수수료", "수송"
    ]
    if has_token_any(token_set_lower, transport_terms) or text_has_any(text_lower, transport_terms):
        return make_business_content_result_v4(
            category="교통/물류진흥",
            reason="우선규칙: 교통·물류·운수·택시·버스 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # H. 주거 명확 키워드는 주거안정 우선
    housing_terms = [
        "주거", "주택", "공동주택", "빈집", "임대주택",
        "집수리", "주거환경", "마을공동시설"
    ]
    if has_token_any(token_set_lower, housing_terms) or text_has_any(text_lower, housing_terms):
        return make_business_content_result_v4(
            category="주거안정",
            reason="우선규칙: 주거·주택·공동주택·빈집 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # I. 종교활동은 종교 고유 맥락일 때 우선
    religion_terms = [
        "종교", "불교", "기독교", "천주교", "교회", "성당",
        "사찰", "법회", "목사", "스님", "신앙", "예배", "기도"
    ]
    if has_token_any(token_set_lower, religion_terms) or text_has_any(text_lower, religion_terms):
        return make_business_content_result_v4(
            category="종교활동",
            reason="우선규칙: 종교 단체·종교행사 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # J. 교육 관련 명확 키워드는 교육 보장 우선
    education_terms = [
        "교육", "학교", "대학", "인재", "장학", "평생교육",
        "학습", "진로", "독서", "도서관", "문해", "교실", "아카데미"
    ]
    if has_token_any(token_set_lower, education_terms) or text_has_any(text_lower, education_terms):
        return make_business_content_result_v4(
            category="교육 보장",
            reason="우선규칙: 교육·학습·장학·학교·도서관 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # K. 체육 관련 사업은 문화활동으로 운영화
    sports_culture_terms = [
        "체육", "생활체육", "스포츠", "레포츠", "선수",
        "축구", "야구", "농구", "배구", "탁구", "배드민턴",
        "골프", "게이트볼", "파크골프", "수영", "마라톤",
        "태권도", "테니스", "볼링", "검도", "궁도",
        "전지훈련", "동계훈련", "스토브리그", "유도대회", "체육대회"
    ]
    if has_token_any(token_set_lower, sports_culture_terms) or text_has_any(text_lower, sports_culture_terms):
        return make_business_content_result_v4(
            category="문화활동",
            reason="우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로 운영화",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # L. 지역사 편찬/문예지 등은 맥락 조건을 충족할 때만 문화활동
    if has_history_publication_context_v4(token_set_lower, text_lower):
        return make_business_content_result_v4(
            category="문화활동",
            reason="우선규칙: 지역사·향토사·문예지 편찬/발간 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # M. 유림/향교/전통문화 맥락은 문화활동 우선
    traditional_culture_terms = [
        "동학", "유림", "성균관유도회", "유도회",
        "향교", "전승", "전통", "민속"
    ]
    if has_token_any(token_set_lower, traditional_culture_terms) or text_has_any(text_lower, traditional_culture_terms):
        return make_business_content_result_v4(
            category="문화활동",
            reason="우선규칙: 전통문화·유림·향교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # N. 관광/휴양은 관광·휴양·방문객 맥락일 때 우선
    tourism_terms = [
        "관광", "휴양", "여행", "탐방", "투어", "체험관광",
        "관광객", "관광지", "둘레길", "캠핑", "해수욕장"
    ]
    if has_token_any(token_set_lower, tourism_terms) or text_has_any(text_lower, tourism_terms):
        return make_business_content_result_v4(
            category="관광/휴양활동",
            reason="우선규칙: 관광·휴양·방문객 유치 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # O. 문화활동 명확 키워드
    culture_terms = [
        "문화", "예술", "공연", "전시", "국악", "농악",
        "미술", "음악", "문학", "콘텐츠", "문화재",
        "박물관", "향교", "제례", "석전대제",
        "민속", "가요제", "가요", "예총"
    ]
    if has_token_any(token_set_lower, culture_terms) or text_has_any(text_lower, culture_terms):
        return make_business_content_result_v4(
            category="문화활동",
            reason="우선규칙: 문화·예술·공연·전시 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # P. 행정/통일/외교 명확 키워드
    admin_terms = [
        "행정", "통일", "민주평통", "평화", "민주", "인권",
        "주민자치", "자치", "이장", "새마을", "바르게살기",
        "자유총연맹", "자원봉사", "봉사", "국제교류", "외교"
    ]
    if has_token_any(token_set_lower, admin_terms) or text_has_any(text_lower, admin_terms):
        return make_business_content_result_v4(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체·통일·평화·외교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # --------------------------------------------------------
    # 6-3) 일반 규칙
    # --------------------------------------------------------

    if len(strong_categories) == 1:
        main_category = strong_categories[0]
        return make_business_content_result_v4(
            category=main_category,
            reason=f"강키워드 단일 분야 매칭: {main_category}({', '.join(strong_matches[main_category])})",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    if len(strong_categories) >= 2:
        return make_business_content_result_v4(
            category="복합",
            reason=f"여러 분야 강키워드 동시 매칭: {', '.join(strong_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    if len(strong_categories) == 0 and len(weak_categories) >= 1:
        return make_business_content_result_v4(
            category="검토필요",
            reason=f"약키워드만 매칭: {', '.join(weak_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    return make_business_content_result_v4(
        category="기타",
        reason="매칭 키워드 없음",
        need_review=True,
        strong_matches=strong_matches,
        weak_matches=weak_matches,
        strong_categories=strong_categories,
        weak_categories=weak_categories
    )


print("✅ 사업내용분류 v4 분류 함수 정의 완료")


# ------------------------------------------------------------
# 7) 전체 데이터에 사업내용분류 v4 적용
# ------------------------------------------------------------

classification_result_v4 = df_04b.apply(classify_business_content_v4, axis=1)

for col in classification_result_v4.columns:
    df_04b[col] = classification_result_v4[col]

print("✅ 사업내용분류 v4 적용 완료")
print(f"생성된 열: {classification_result_v4.columns.tolist()}")


# ------------------------------------------------------------
# 8) v4 대분류 분포
# ------------------------------------------------------------

business_content_classification_summary_v4 = (
    df_04b["사업내용분류_대분류_v4"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_대분류_v4")
    .reset_index(name="사업수")
)

business_content_classification_summary_v4["비율"] = (
    business_content_classification_summary_v4["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v4 대분류 분포]")
display(business_content_classification_summary_v4)


# ------------------------------------------------------------
# 9) v4 분석포함여부 분포
# ------------------------------------------------------------

business_content_analysis_flag_summary_v4 = (
    df_04b["사업내용분류_분석포함여부_v4"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분석포함여부_v4")
    .reset_index(name="사업수")
)

business_content_analysis_flag_summary_v4["비율"] = (
    business_content_analysis_flag_summary_v4["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v4 분석포함여부 분포]")
display(business_content_analysis_flag_summary_v4)


# ------------------------------------------------------------
# 10) v3-v4 변화 비교
# ------------------------------------------------------------

business_content_v3_v4_compare = (
    df_04b
    .groupby(["사업내용분류_대분류_v3", "사업내용분류_대분류_v4"])
    .size()
    .reset_index(name="사업수")
    .sort_values("사업수", ascending=False)
)

print("\n[사업내용분류 v3 → v4 변화 상위 60개]")
display(business_content_v3_v4_compare.head(60))


# ------------------------------------------------------------
# 11) v3-v4 기타/검토필요/복합 비교
# ------------------------------------------------------------

review_target_values = ["기타", "검토필요", "복합"]

def count_business_content_review_group(df, col, version):
    return pd.DataFrame({
        "버전": version,
        "구분": ["기타", "검토필요", "복합", "합계"],
        "사업수": [
            int((df[col] == "기타").sum()),
            int((df[col] == "검토필요").sum()),
            int((df[col] == "복합").sum()),
            int(df[col].isin(review_target_values).sum()),
        ]
    })

business_content_review_group_compare_v3_v4 = pd.concat(
    [
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v3", "v3"),
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v4", "v4"),
    ],
    axis=0
)

business_content_review_group_compare_v3_v4["비율"] = (
    business_content_review_group_compare_v3_v4["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v3-v4 기타/검토필요/복합 비교]")
display(business_content_review_group_compare_v3_v4)


# ------------------------------------------------------------
# 12) v4 분류근거별 행 수
# ------------------------------------------------------------

business_content_reason_summary_v4 = (
    df_04b["사업내용분류_분류근거_v4"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분류근거_v4")
    .reset_index(name="사업수")
)

business_content_reason_summary_v4["비율"] = (
    business_content_reason_summary_v4["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v4 분류근거별 행 수 상위 40개]")
display(business_content_reason_summary_v4.head(40))


# ------------------------------------------------------------
# 13) v3-v4 변경 사례
# ------------------------------------------------------------

business_content_changed_cases_v3_v4 = df_04b[
    df_04b["사업내용분류_대분류_v3"] != df_04b["사업내용분류_대분류_v4"]
].copy()

changed_case_cols_v3_v4 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "사업내용분류_대분류_v3",
    "사업내용분류_분류근거_v3",
    "사업내용분류_대분류_v4",
    "사업내용분류_분류근거_v4",
    "평가결과",
    "평가종류",
]

changed_case_cols_v3_v4 = [
    col for col in changed_case_cols_v3_v4
    if col in business_content_changed_cases_v3_v4.columns
]

print(f"\n[v3-v4 대분류 변경 사례 수] {len(business_content_changed_cases_v3_v4):,}행")
display(business_content_changed_cases_v3_v4[changed_case_cols_v3_v4].head(100))


# ------------------------------------------------------------
# 14) 문제 패턴 점검
# ------------------------------------------------------------

# 14-1. 지역사회/역사 substring 점검
history_substring_check_v4 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "지역사회|지역사회보장|역사|지역사|향토사|문예지|편찬|발간|총람",
        regex=True,
        na=False
    )
].copy()

# 14-2. 행정단체 + 실질 내용 점검
admin_content_check_v4 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "새마을|바르게살기|자유총연맹|한국자유총연맹|민주평통|자원봉사",
        regex=True,
        na=False
    )
].copy()

# 14-3. 체육 관련 점검
sports_check_v4 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "체육|생활체육|스포츠|레포츠|축구|야구|농구|배구|탁구|배드민턴|골프|게이트볼|파크골프|태권도|테니스|볼링|검도|궁도|마라톤|전지훈련",
        regex=True,
        na=False
    )
].copy()

# 14-4. 스마트팜 관련 점검
smart_farm_check_v4 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "스마트팜|스마트 농업|스마트농업|스마트 축산|스마트축산|스마트 양식|스마트양식",
        regex=True,
        na=False
    )
].copy()

check_cols_v4 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업내용분류_대분류_v3",
    "사업내용분류_대분류_v4",
    "사업내용분류_분류근거_v4",
    "평가결과",
    "평가종류",
]

check_cols_v4 = [col for col in check_cols_v4 if col in df_04b.columns]

print("\n[지역사회/역사 substring v4 점검]")
display(history_substring_check_v4[check_cols_v4].head(100))

print("\n[행정단체 + 실질 내용 v4 점검]")
display(admin_content_check_v4[check_cols_v4].head(100))

print("\n[체육 관련 v4 점검]")
display(sports_check_v4[check_cols_v4].head(80))

print("\n[스마트팜 관련 v4 점검]")
display(smart_farm_check_v4[check_cols_v4].head(80))


# ------------------------------------------------------------
# 15) 저장
# ------------------------------------------------------------

rule_summary_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_rule_summary_v4_{RUN_TS}.xlsx"
)

classification_summary_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_classification_summary_v4_{RUN_TS}.xlsx"
)

v3_v4_compare_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_v3_v4_compare_{RUN_TS}.xlsx"
)

changed_cases_v3_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_changed_cases_v3_v4_{RUN_TS}.xlsx"
)

reason_summary_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_reason_summary_v4_{RUN_TS}.xlsx"
)

history_substring_check_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_history_substring_check_v4_{RUN_TS}.xlsx"
)

admin_content_check_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_admin_content_check_v4_{RUN_TS}.xlsx"
)

sports_check_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_sports_check_v4_{RUN_TS}.xlsx"
)

smart_farm_check_v4_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_smart_farm_check_v4_{RUN_TS}.xlsx"
)

business_content_rule_summary_v4.to_excel(rule_summary_v4_path, index=False)

with pd.ExcelWriter(classification_summary_v4_path, engine="openpyxl") as writer:
    business_content_classification_summary_v4.to_excel(
        writer,
        sheet_name="classification_summary_v4",
        index=False
    )
    business_content_analysis_flag_summary_v4.to_excel(
        writer,
        sheet_name="analysis_flag_summary_v4",
        index=False
    )
    business_content_review_group_compare_v3_v4.to_excel(
        writer,
        sheet_name="review_group_compare",
        index=False
    )

business_content_v3_v4_compare.to_excel(v3_v4_compare_path, index=False)
convert_df_for_excel(business_content_changed_cases_v3_v4[changed_case_cols_v3_v4]).to_excel(changed_cases_v3_v4_path, index=False)
business_content_reason_summary_v4.to_excel(reason_summary_v4_path, index=False)

convert_df_for_excel(history_substring_check_v4[check_cols_v4]).to_excel(history_substring_check_v4_path, index=False)
convert_df_for_excel(admin_content_check_v4[check_cols_v4]).to_excel(admin_content_check_v4_path, index=False)
convert_df_for_excel(sports_check_v4[check_cols_v4]).to_excel(sports_check_v4_path, index=False)
convert_df_for_excel(smart_farm_check_v4[check_cols_v4]).to_excel(smart_farm_check_v4_path, index=False)

print("\n[저장 완료]")
print(f"- v4 규칙 요약표: {rule_summary_v4_path}")
print(f"- v4 대분류/분석포함/검토그룹 요약: {classification_summary_v4_path}")
print(f"- v3-v4 대분류 변화 비교: {v3_v4_compare_path}")
print(f"- v3-v4 변경 사례: {changed_cases_v3_v4_path}")
print(f"- v4 분류근거별 행 수: {reason_summary_v4_path}")
print(f"- 지역사회/역사 substring 점검: {history_substring_check_v4_path}")
print(f"- 행정단체 + 실질 내용 점검: {admin_content_check_v4_path}")
print(f"- 체육 관련 점검: {sports_check_v4_path}")
print(f"- 스마트팜 관련 점검: {smart_farm_check_v4_path}")

print("\n✅ 04B-8 v3 점검 기반 v4 최종 후보 생성 완료")

✅ 04B-8 실행 전 필수 객체 및 열 확인 완료
✅ v4 규칙 생성 완료
규칙 버전: business_content_rules_v4_final_candidate_fix_history_substring_admin_content_priority

[사업내용분류 v4 규칙 요약표]


,사업내용분류_대분류,strong_keywords,weak_keywords,strong_keyword_count,weak_keyword_count,note,rule_id
0,행정/통일/외교,"행정, 통일, 민주평통, 평화, 민주, 인권, 주민자치, 자치, 이장, 새마을, 바...",,16,0,행정·시민단체·통일·평화·외교 관련 사업,business_content_rules_v4_final_candidate_fix_...
1,안전 보장,"안전, 재난, 방재, 소방, 방범, 범죄, 피해자, 구급, 응급, 교통질서, 모범운...","예방, 보호, 순찰, 감시, 질서",13,5,재난·방범·범죄피해·응급·안전 관련 사업,business_content_rules_v4_final_candidate_fix_...
2,교육 보장,"교육, 학교, 대학, 인재, 장학, 평생교육, 학습, 진로, 독서, 도서관, 문해,...","역량, 연수, 프로그램, 강좌, 강사",13,5,교육·학습·장학·학교·도서관 관련 사업,business_content_rules_v4_final_candidate_fix_...
3,문화활동,"문화, 예술, 공연, 전시, 국악, 농악, 미술, 음악, 문학, 콘텐츠, 문화재, ...","축전, 한마당, 심포지엄",53,3,문화·예술·전통문화·공연·전시 관련 사업,business_content_rules_v4_final_candidate_fix_...
4,관광/휴양활동,"관광, 휴양, 축제, 여행, 탐방, 투어, 체험관광, 관광객, 관광지, 둘레길, 캠...","체험, 홍보, 마케팅, 방문, 코스",13,5,관광·휴양·축제·방문객 유치 관련 사업,business_content_rules_v4_final_candidate_fix_...
5,종교활동,"종교, 불교, 기독교, 천주교, 교회, 성당, 사찰, 절, 법회, 목사, 스님, 신앙","예배, 기도, 신도",12,3,종교 단체·종교행사 관련 사업,business_content_rules_v4_final_candidate_fix_...
6,환경향상,"환경, 생태, 기후, 탄소, 쓰레기, 폐기물, 재활용, 하천, 수질, 숲, 녹지, ...","자연, 보전, 개선, 정비",16,4,환경보전·생태·폐기물·자원순환 관련 사업,business_content_rules_v4_final_candidate_fix_...
7,사회복지향상,"복지, 복지관, 장애인, 노인, 노인회, 어르신, 아동, 청소년, 여성, 가족, 다...","종사자, 위문, 나눔",29,3,사회복지 대상자·돌봄·보육·취약계층 관련 사업,business_content_rules_v4_final_candidate_fix_...
8,보훈향상,"보훈, 유공자, 참전, 고엽제, 미망인, 수훈자, 군인회, 유족회, 전우회, 상이군...","위문, 추모, 기념",14,3,보훈·참전·유공자·유족 관련 사업,business_content_rules_v4_final_candidate_fix_...
9,고용안정,"고용, 일자리, 취업, 창업, 근로자, 노동, 청년창업, 직업, 훈련, 인력양성",청년,10,1,고용·취업·일자리·창업 관련 사업,business_content_rules_v4_final_candidate_fix_...


✅ 사업내용분류 v4 분류 함수 정의 완료
✅ 사업내용분류 v4 적용 완료
생성된 열: ['사업내용분류_대분류_v4', '사업내용분류_세분류_v4', '사업내용분류_후보분야_v4', '사업내용분류_분류근거_v4', '사업내용분류_검토필요여부_v4', '사업내용분류_분석포함여부_v4', '사업내용분류_rule_id_v4', '사업내용분류_강키워드_v4', '사업내용분류_약키워드_v4']

[사업내용분류 v4 대분류 분포]


,사업내용분류_대분류_v4,사업수,비율
0,사회복지향상,18689,23.58
1,기타,13683,17.26
2,문화활동,11782,14.86
3,1차 산업지원,8225,10.38
4,검토필요,5248,6.62
5,교육 보장,3994,5.04
6,행정/통일/외교,3819,4.82
7,보훈향상,3081,3.89
8,안전 보장,1932,2.44
9,산업/에너지지원,1557,1.96



[사업내용분류 v4 분석포함여부 분포]


...


[체육 관련 v4 점검]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v3,사업내용분류_대분류_v4,사업내용분류_분류근거_v4,평가결과,평가종류
12,12,경남_남해군.xlsx,경남,남해군,남중권생활체육 교류 지원,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,미흡,유지필요성평가
14,14,경남_남해군.xlsx,경남,남해군,레포츠 자격증 취득 지원,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,미흡,유지필요성평가
73,73,경남_남해군.xlsx,경남,남해군,남해군수기 전국 검도대회,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
74,74,경남_남해군.xlsx,경남,남해군,보물섬배 남해 전국 탁구대잔치,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
75,75,경남_남해군.xlsx,경남,남해군,우수 체육단체 활동 지원,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,유지필요성평가
...,...,...,...,...,...,...,...,...,...,...
915,915,경남_진주시.xlsx,경남,진주시,진주리틀야구단 운영지원,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,우수,성과평가
917,917,경남_진주시.xlsx,경남,진주시,읍면동 체육대회 행사비 지원,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,우수,성과평가
919,919,경남_진주시.xlsx,경남,진주시,진주남강마라톤대회,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,우수,성과평가
920,920,경남_진주시.xlsx,경남,진주시,진주마라톤대회,문화활동,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,우수,성과평가



[스마트팜 관련 v4 점검]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v3,사업내용분류_대분류_v4,사업내용분류_분류근거_v4,평가결과,평가종류
2872,2872,경남_의령군.xlsx,경남,의령군,스마트팜 온실 신축사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,우수,성과평가
5929,5929,경남_함양군.xlsx,경남,함양군,스마트팜 큐브 생산시설지원,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우미흡,성과평가
6966,6966,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6967,6967,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
6968,6968,V1 후보_성과_경남_김해시.xlsx,경남,김해시,원예 스마트팜 현대화 사업,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우우수,성과평가
...,...,...,...,...,...,...,...,...,...,...
67480,67494,전북_본청.xlsx,전북,본청,친환경 스마트 양식기반 구축,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,보통,성과평가
68350,68364,전북_전주시.xlsx,전북,전주시,농업인스마트팜시설지원,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,보통,성과평가
69611,69625,전남_해남군.xlsx,전남,해남군,스마트팜 기술 적용 고구마 우량종순 증식 시범,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,우수,성과평가
69614,69628,전남_해남군.xlsx,전남,해남군,시설원예 스마트팜 보급 시범,1차 산업지원,1차 산업지원,우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락,매우 우수,성과평가



[저장 완료]
- v4 규칙 요약표: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_rule_summary_v4_20260606_1912.xlsx
- v4 대분류/분석포함/검토그룹 요약: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_classification_summary_v4_20260606_1912.xlsx
- v3-v4 대분류 변화 비교: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_v3_v4_compare_20260606_1912.xlsx
- v3-v4 변경 사례: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_changed_cases_v3_v4_20260606_1912.xlsx
- v4 분류근거별 행 수: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_busin

04B-9. v4 과보정 점검 기반 v5 최종 후보 생성
1. 이 단계의 목적

이 단계의 목적은 v4 결과를 보존한 상태에서, 최종 handoff 직전 사용할 v5 최종 후보 분류를 생성하는 것입니다.

v5에서는 다음 문제만 소폭 보정합니다.

1. 지역사회보장협의체, 사회보장협의체, 지역사회보장 → 사회복지향상
2. 체육·스포츠 키워드가 있으면 학교, 교실이 함께 있어도 → 문화활동
3. 문화학교, 문화교실, 예술학교, 예술교육, 문화예술교육 → 문화활동
4. 통일, 안보, 민주평통, 평화교육, 민주시민교육 → 행정/통일/외교
5. 청소년문화축제, 청소년예술제, 청소년체육대회처럼 청소년+문화/축제/예술/체육 맥락 → 문화활동

In [12]:
# 04B-9. v4 과보정 점검 기반 v5 최종 후보 생성

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 및 열 확인
# ------------------------------------------------------------

from copy import deepcopy

required_objects = [
    "df_04b",
    "business_content_rules_v4",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
    "safe_parse_tokens",
    "normalize_text_for_match",
    "get_token_set_lower",
    "has_token_any",
    "text_has_any",
    "match_keywords",
    "convert_df_for_excel",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1 → 04B-2 → 04B-3 → 04B-4 → 04B-5 → 04B-6 → 04B-7 → 04B-8을 먼저 실행하세요."
    )

required_cols_04b_9 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류_v4",
    "사업내용분류_분류근거_v4",
    "사업내용분류_검토필요여부_v4",
    "사업내용분류_분석포함여부_v4",
]

missing_cols_04b_9 = [
    col for col in required_cols_04b_9
    if col not in df_04b.columns
]

if missing_cols_04b_9:
    raise KeyError(
        f"04B-9 실행에 필요한 열이 없습니다: {missing_cols_04b_9}\n"
        "04B-8이 정상 실행되었는지 확인하세요."
    )

print("✅ 04B-9 실행 전 필수 객체 및 열 확인 완료")


# ------------------------------------------------------------
# 1) v5 규칙 생성
# ------------------------------------------------------------
# v4 규칙은 보존하고, v5 규칙을 별도로 만든다.

BUSINESS_CONTENT_RULE_VERSION_V5 = "business_content_rules_v5_final_candidate_fix_education_overcapture"

business_content_rules_v5 = deepcopy(business_content_rules_v4)

print("✅ v5 규칙 생성 완료")
print(f"규칙 버전: {BUSINESS_CONTENT_RULE_VERSION_V5}")


# ------------------------------------------------------------
# 2) v5 규칙 요약표 생성
# ------------------------------------------------------------

rule_rows_v5 = []

for category, rule in business_content_rules_v5.items():
    rule_rows_v5.append({
        "사업내용분류_대분류": category,
        "strong_keywords": ", ".join(rule.get("strong", [])),
        "weak_keywords": ", ".join(rule.get("weak", [])),
        "strong_keyword_count": len(rule.get("strong", [])),
        "weak_keyword_count": len(rule.get("weak", [])),
        "note": rule.get("note", ""),
        "rule_id": BUSINESS_CONTENT_RULE_VERSION_V5,
    })

business_content_rule_summary_v5 = pd.DataFrame(rule_rows_v5)

print("\n[사업내용분류 v5 규칙 요약표]")
display(business_content_rule_summary_v5)


# ------------------------------------------------------------
# 3) v5 결과 생성 함수
# ------------------------------------------------------------

def make_business_content_result_v5(
    category,
    reason,
    need_review,
    strong_matches,
    weak_matches,
    strong_categories,
    weak_categories
):
    """사업내용분류 v5 결과를 pd.Series로 반환"""

    if need_review is False:
        analysis_flag = "주분석_포함"
    elif category == "복합":
        analysis_flag = "보조분석_검토"
    else:
        analysis_flag = "해석주의"

    return pd.Series({
        "사업내용분류_대분류_v5": category,
        "사업내용분류_세분류_v5": "",
        "사업내용분류_후보분야_v5": ", ".join(sorted(set(strong_categories + weak_categories))),
        "사업내용분류_분류근거_v5": reason,
        "사업내용분류_검토필요여부_v5": bool(need_review),
        "사업내용분류_분석포함여부_v5": analysis_flag,
        "사업내용분류_rule_id_v5": BUSINESS_CONTENT_RULE_VERSION_V5,
        "사업내용분류_강키워드_v5": json.dumps(strong_matches, ensure_ascii=False),
        "사업내용분류_약키워드_v5": json.dumps(weak_matches, ensure_ascii=False),
    })


# ------------------------------------------------------------
# 4) v5 보조 함수
# ------------------------------------------------------------

def has_social_security_council_context(text_lower, token_set_lower):
    """지역사회보장협의체 등 사회복지 전달체계 맥락 판정"""

    terms = [
        "지역사회보장협의체",
        "사회보장협의체",
        "지역사회보장",
        "지역 사회 보장 협의체",
        "사회 보장 협의체",
    ]

    return has_token_any(token_set_lower, terms) or text_has_any(text_lower, terms)


def has_unification_security_context(text_lower, token_set_lower):
    """
    통일·안보·민주시민교육·평화교육 맥락 판정.
    교육이라는 단어가 있어도 통일/안보/평화 시민교육이면 행정/통일/외교로 처리한다.
    """

    terms = [
        "통일",
        "안보",
        "민주평통",
        "평통",
        "민주시민교육",
        "민주시민 교육",
        "평화교육",
        "평화 교육",
        "통일교육",
        "통일 교육",
        "안보교육",
        "안보 교육",
        "안보견학",
        "안보 견학",
        "평화학교",
    ]

    return has_token_any(token_set_lower, terms) or text_has_any(text_lower, terms)


def has_culture_education_context(text_lower, token_set_lower):
    """
    문화+교육 혼합 표현 중 문화활동으로 보는 것이 자연스러운 맥락.
    예: 문화학교, 문화교실, 문화예술교육, 예술학교 등
    """

    terms = [
        "문화학교",
        "문화 교실",
        "문화교실",
        "문화강좌",
        "문화 강좌",
        "문화예술교육",
        "문화 예술 교육",
        "예술교육",
        "예술 교육",
        "예술학교",
        "예술 학교",
        "국악교실",
        "국악 교실",
        "미술교실",
        "미술 교실",
        "음악교실",
        "음악 교실",
        "문학강좌",
        "문학 강좌",
    ]

    return has_token_any(token_set_lower, terms) or text_has_any(text_lower, terms)


def has_youth_culture_event_context(text_lower, token_set_lower):
    """
    청소년 + 문화/예술/축제/체육 맥락.
    청소년이 들어가더라도 복지 일반이 아니라 문화·예술·체육 행사 성격이면 문화활동으로 처리한다.
    """

    youth_terms = ["청소년", "청년"]
    culture_event_terms = [
        "문화축제", "문화 축제",
        "예술제", "예술 제",
        "문화제",
        "축제",
        "가요제",
        "음악회",
        "공연",
        "전시",
        "체육대회",
        "체육 대회",
        "스포츠",
        "생활체육",
    ]

    has_youth = has_token_any(token_set_lower, youth_terms) or text_has_any(text_lower, youth_terms)
    has_event = has_token_any(token_set_lower, culture_event_terms) or text_has_any(text_lower, culture_event_terms)

    return has_youth and has_event


def has_sports_context(text_lower, token_set_lower):
    """
    체육·스포츠 맥락 판정.
    학교/교실이 함께 있어도 체육 종목이나 생활체육 맥락이 강하면 문화활동으로 처리한다.
    """

    sports_terms = [
        "체육", "생활체육", "스포츠", "레포츠", "선수",
        "축구", "야구", "농구", "배구", "탁구", "배드민턴",
        "골프", "게이트볼", "파크골프", "수영", "마라톤",
        "태권도", "테니스", "볼링", "검도", "궁도",
        "전지훈련", "동계훈련", "스토브리그",
        "유도대회", "체육대회", "운동부", "탁구교실",
        "축구대회", "야구대회", "마라톤대회", "테니스대회"
    ]

    return has_token_any(token_set_lower, sports_terms) or text_has_any(text_lower, sports_terms)


def has_admin_org_v5(text_lower, token_set_lower):
    """행정/시민단체 조직명 존재 여부"""

    admin_org_terms = [
        "한국자유총연맹", "자유총연맹",
        "바르게살기",
        "새마을",
        "민주평통", "평통",
        "자원봉사",
        "주민자치",
        "이장협의회", "통장협의회", "리장협의회",
    ]

    return has_token_any(token_set_lower, admin_org_terms) or text_has_any(text_lower, admin_org_terms)


def has_admin_operation_context_v5(text_lower, token_set_lower):
    """행정단체 조직 운영 맥락 판정"""

    admin_operation_terms = [
        "운영비", "사무실", "사무국", "사무처", "집기",
        "지부", "지회", "협의회 운영", "단체운영", "조직운영",
        "기념식", "총회", "회관", "보수 및 집기", "운영 지원",
        "지도자대회", "지도자 대회", "한마음다짐대회", "화합한마음대회"
    ]

    return has_token_any(token_set_lower, admin_operation_terms) or text_has_any(text_lower, admin_operation_terms)


def has_history_publication_context_v5(token_set_lower, text_lower):
    """
    지역사·향토사·문예지·총람·편찬/발간 맥락 판정.
    v4 기준을 유지한다.
    """

    false_positive_patterns = [
        "지역사회", "지역사회보장", "지역사회보장협의체",
        "군지부", "군지회", "시지부", "시지회", "도지부", "도지회",
        "구지부", "구지회", "읍지부", "읍지회", "면지부", "면지회",
        "조사료",
    ]

    if text_has_any(text_lower, false_positive_patterns):
        return False

    exact_history_tokens = [
        "문예지", "향토지", "읍지", "면지", "군지",
        "백서", "사료", "향토사", "지역사", "역사",
        "마을지", "군사", "읍사", "면사", "총람", "금석문총람"
    ]

    if has_token_any(token_set_lower, exact_history_tokens):
        return True

    publication_terms = ["편찬", "발간", "재발간"]
    context_terms = [
        "문예지", "향토", "향토지", "읍지", "면지", "군지",
        "백서", "사료", "지역사", "향토사", "마을지", "총람",
        "금석문", "자료집", "문화원"
    ]

    has_publication = has_token_any(token_set_lower, publication_terms) or text_has_any(text_lower, publication_terms)
    has_context = has_token_any(token_set_lower, context_terms) or text_has_any(text_lower, context_terms)

    return has_publication and has_context


print("✅ v5 보조 함수 정의 완료")


# ------------------------------------------------------------
# 5) v5 분류 함수
# ------------------------------------------------------------

def classify_business_content_v5(row):
    """
    사업내용분류 v5 분류 함수.

    v4 대비 보정:
    - 지역사회보장협의체는 사회복지향상
    - 체육/스포츠 맥락은 학교/교실이 있어도 문화활동
    - 문화학교/문화예술교육 등은 문화활동
    - 통일/안보/평화/민주시민교육은 행정/통일/외교
    - 청소년+문화/예술/축제/체육 행사는 문화활동
    """

    tokens = safe_parse_tokens(row.get("tokens_analysis_primary", []))
    token_set_lower = get_token_set_lower(tokens)
    text_lower = normalize_text_for_match(row.get("사업명_clean_spacefix", ""))

    strong_matches = {}
    weak_matches = {}

    for category, rule in business_content_rules_v5.items():
        strong_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("strong", [])
        )
        weak_hit = match_keywords(
            token_set_lower,
            text_lower,
            rule.get("weak", [])
        )

        if strong_hit:
            strong_matches[category] = strong_hit

        if weak_hit:
            weak_matches[category] = weak_hit

    strong_categories = list(strong_matches.keys())
    weak_categories = list(weak_matches.keys())


    # --------------------------------------------------------
    # 5-1) v5 최우선 보정 규칙
    # --------------------------------------------------------

    # A. 지역사회보장협의체는 사회복지향상
    if has_social_security_council_context(text_lower, token_set_lower):
        return make_business_content_result_v5(
            category="사회복지향상",
            reason="우선규칙: 지역사회보장협의체·사회보장협의체 관련 사회복지 전달체계 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # B. 통일·안보·민주시민교육·평화교육은 행정/통일/외교
    if has_unification_security_context(text_lower, token_set_lower):
        return make_business_content_result_v5(
            category="행정/통일/외교",
            reason="우선규칙: 통일·안보·평화·민주시민교육 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # C. 체육·스포츠 맥락은 문화활동
    if has_sports_context(text_lower, token_set_lower):
        return make_business_content_result_v5(
            category="문화활동",
            reason="우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로 운영화",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # D. 문화+교육 혼합 표현은 문화활동
    if has_culture_education_context(text_lower, token_set_lower):
        return make_business_content_result_v5(
            category="문화활동",
            reason="우선규칙: 문화학교·문화교실·문화예술교육 등 문화교육 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # E. 청소년+문화/예술/축제/체육 행사는 문화활동
    if has_youth_culture_event_context(text_lower, token_set_lower):
        return make_business_content_result_v5(
            category="문화활동",
            reason="우선규칙: 청소년 문화·예술·축제·체육 행사 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )


    # --------------------------------------------------------
    # 5-2) 행정단체 조직 운영 맥락
    # --------------------------------------------------------

    admin_org_present = has_admin_org_v5(text_lower, token_set_lower)
    admin_operation_present = has_admin_operation_context_v5(text_lower, token_set_lower)

    if admin_org_present and admin_operation_present:
        return make_business_content_result_v5(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체 조직 운영 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )


    # --------------------------------------------------------
    # 5-3) 실질 사업내용 우선 규칙
    # --------------------------------------------------------

    # A. 보훈 관련은 보훈향상 우선
    bohun_terms = [
        "보훈", "유공자", "참전", "고엽제", "미망인", "수훈자",
        "군인회", "유족회", "전우회", "상이군경", "광복회",
        "재향군인회", "전몰군경", "무공"
    ]
    if has_token_any(token_set_lower, bohun_terms) or text_has_any(text_lower, bohun_terms):
        return make_business_content_result_v5(
            category="보훈향상",
            reason="우선규칙: 보훈·참전·유공자·유족 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # B. 사회복지 대상자 및 복지서비스 관련은 사회복지향상 우선
    welfare_terms = [
        "복지", "장애인", "노인", "노인회", "어르신", "아동", "청소년",
        "여성", "가족", "다문화", "보육", "어린이집", "경로당",
        "돌봄", "자활", "재활", "취약", "저소득", "영유아", "한부모",
        "행복나눔", "희망나눔", "목욕", "이동목욕", "무료급식", "푸드뱅크"
    ]
    if has_token_any(token_set_lower, welfare_terms) or text_has_any(text_lower, welfare_terms):
        return make_business_content_result_v5(
            category="사회복지향상",
            reason="우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # C. 스마트팜/스마트 농업은 1차 산업지원 우선
    smart_agri_priority_terms = [
        "스마트팜", "스마트 농업", "스마트농업",
        "스마트 축산", "스마트축산",
        "스마트 양식", "스마트양식"
    ]
    if text_has_any(text_lower, smart_agri_priority_terms):
        return make_business_content_result_v5(
            category="1차 산업지원",
            reason="우선규칙: 스마트팜·스마트농업 등 1차 산업 디지털화 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # D. 1차 산업 관련은 1차 산업지원 우선
    primary_industry_terms = [
        "농업", "농촌", "농가", "농산물", "농산", "농업인",
        "작물", "쌀", "마늘", "양파", "원예", "축산", "축산물",
        "가축", "한우", "수산", "수산물", "어업", "어촌",
        "임업", "산림", "귀농", "귀촌", "영농", "재배",
        "벼", "감자", "딸기", "양봉", "낙농", "병해충", "방제"
    ]
    if has_token_any(token_set_lower, primary_industry_terms) or text_has_any(text_lower, primary_industry_terms):
        return make_business_content_result_v5(
            category="1차 산업지원",
            reason="우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # E. 보건/의료 명확 키워드는 보건/의료 우선
    health_terms = [
        "보건", "의료", "병원", "건강", "치매", "감염", "질병",
        "예방접종", "정신건강", "위생", "의약", "진료", "환자", "검진"
    ]
    if has_token_any(token_set_lower, health_terms) or text_has_any(text_lower, health_terms):
        return make_business_content_result_v5(
            category="보건/의료",
            reason="우선규칙: 보건·의료·건강·감염·질병 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # F. 안전 관련은 안전 보장 우선
    safety_terms = [
        "안전", "재난", "방재", "소방", "방범", "범죄",
        "피해자", "구급", "응급", "교통질서", "모범운전자", "cctv", "CCTV"
    ]
    if has_token_any(token_set_lower, safety_terms) or text_has_any(text_lower, safety_terms):
        return make_business_content_result_v5(
            category="안전 보장",
            reason="우선규칙: 재난·방범·범죄피해·응급·안전 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # G. 교통/물류 명확 키워드는 교통/물류진흥 우선
    transport_terms = [
        "교통", "물류", "택시", "버스", "운수", "운송", "주차",
        "도로교통", "교통약자", "터미널", "카드결제", "단말기",
        "통신수수료", "수송"
    ]
    if has_token_any(token_set_lower, transport_terms) or text_has_any(text_lower, transport_terms):
        return make_business_content_result_v5(
            category="교통/물류진흥",
            reason="우선규칙: 교통·물류·운수·택시·버스 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # H. 주거 명확 키워드는 주거안정 우선
    housing_terms = [
        "주거", "주택", "공동주택", "빈집", "임대주택",
        "집수리", "주거환경", "마을공동시설"
    ]
    if has_token_any(token_set_lower, housing_terms) or text_has_any(text_lower, housing_terms):
        return make_business_content_result_v5(
            category="주거안정",
            reason="우선규칙: 주거·주택·공동주택·빈집 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # I. 종교활동은 종교 고유 맥락일 때 우선
    religion_terms = [
        "종교", "불교", "기독교", "천주교", "교회", "성당",
        "사찰", "법회", "목사", "스님", "신앙", "예배", "기도"
    ]
    if has_token_any(token_set_lower, religion_terms) or text_has_any(text_lower, religion_terms):
        return make_business_content_result_v5(
            category="종교활동",
            reason="우선규칙: 종교 단체·종교행사 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # J. 교육 관련 명확 키워드는 교육 보장
    education_terms = [
        "교육", "학교", "대학", "인재", "장학", "평생교육",
        "학습", "진로", "독서", "도서관", "문해", "교실", "아카데미"
    ]
    if has_token_any(token_set_lower, education_terms) or text_has_any(text_lower, education_terms):
        return make_business_content_result_v5(
            category="교육 보장",
            reason="우선규칙: 교육·학습·장학·학교·도서관 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # K. 지역사 편찬/문예지 등은 맥락 조건을 충족할 때만 문화활동
    if has_history_publication_context_v5(token_set_lower, text_lower):
        return make_business_content_result_v5(
            category="문화활동",
            reason="우선규칙: 지역사·향토사·문예지 편찬/발간 맥락",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # L. 유림/향교/전통문화 맥락은 문화활동
    traditional_culture_terms = [
        "동학", "유림", "성균관유도회", "유도회",
        "향교", "전승", "전통", "민속"
    ]
    if has_token_any(token_set_lower, traditional_culture_terms) or text_has_any(text_lower, traditional_culture_terms):
        return make_business_content_result_v5(
            category="문화활동",
            reason="우선규칙: 전통문화·유림·향교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # M. 관광/휴양은 관광·휴양·방문객 맥락일 때 우선
    tourism_terms = [
        "관광", "휴양", "여행", "탐방", "투어", "체험관광",
        "관광객", "관광지", "둘레길", "캠핑", "해수욕장"
    ]
    if has_token_any(token_set_lower, tourism_terms) or text_has_any(text_lower, tourism_terms):
        return make_business_content_result_v5(
            category="관광/휴양활동",
            reason="우선규칙: 관광·휴양·방문객 유치 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # N. 문화활동 명확 키워드
    culture_terms = [
        "문화", "예술", "공연", "전시", "국악", "농악",
        "미술", "음악", "문학", "콘텐츠", "문화재",
        "박물관", "향교", "제례", "석전대제",
        "민속", "가요제", "가요", "예총"
    ]
    if has_token_any(token_set_lower, culture_terms) or text_has_any(text_lower, culture_terms):
        return make_business_content_result_v5(
            category="문화활동",
            reason="우선규칙: 문화·예술·공연·전시 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    # O. 행정/통일/외교 명확 키워드
    admin_terms = [
        "행정", "통일", "민주평통", "평화", "민주", "인권",
        "주민자치", "자치", "이장", "새마을", "바르게살기",
        "자유총연맹", "자원봉사", "봉사", "국제교류", "외교"
    ]
    if has_token_any(token_set_lower, admin_terms) or text_has_any(text_lower, admin_terms):
        return make_business_content_result_v5(
            category="행정/통일/외교",
            reason="우선규칙: 행정·시민단체·통일·평화·외교 관련 키워드",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )


    # --------------------------------------------------------
    # 5-4) 일반 규칙
    # --------------------------------------------------------

    if len(strong_categories) == 1:
        main_category = strong_categories[0]
        return make_business_content_result_v5(
            category=main_category,
            reason=f"강키워드 단일 분야 매칭: {main_category}({', '.join(strong_matches[main_category])})",
            need_review=False,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    if len(strong_categories) >= 2:
        return make_business_content_result_v5(
            category="복합",
            reason=f"여러 분야 강키워드 동시 매칭: {', '.join(strong_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    if len(strong_categories) == 0 and len(weak_categories) >= 1:
        return make_business_content_result_v5(
            category="검토필요",
            reason=f"약키워드만 매칭: {', '.join(weak_categories)}",
            need_review=True,
            strong_matches=strong_matches,
            weak_matches=weak_matches,
            strong_categories=strong_categories,
            weak_categories=weak_categories
        )

    return make_business_content_result_v5(
        category="기타",
        reason="매칭 키워드 없음",
        need_review=True,
        strong_matches=strong_matches,
        weak_matches=weak_matches,
        strong_categories=strong_categories,
        weak_categories=weak_categories
    )


print("✅ 사업내용분류 v5 분류 함수 정의 완료")


# ------------------------------------------------------------
# 6) 전체 데이터에 사업내용분류 v5 적용
# ------------------------------------------------------------

classification_result_v5 = df_04b.apply(classify_business_content_v5, axis=1)

for col in classification_result_v5.columns:
    df_04b[col] = classification_result_v5[col]

print("✅ 사업내용분류 v5 적용 완료")
print(f"생성된 열: {classification_result_v5.columns.tolist()}")


# ------------------------------------------------------------
# 7) v5 대분류 분포
# ------------------------------------------------------------

business_content_classification_summary_v5 = (
    df_04b["사업내용분류_대분류_v5"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_대분류_v5")
    .reset_index(name="사업수")
)

business_content_classification_summary_v5["비율"] = (
    business_content_classification_summary_v5["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v5 대분류 분포]")
display(business_content_classification_summary_v5)


# ------------------------------------------------------------
# 8) v5 분석포함여부 분포
# ------------------------------------------------------------

business_content_analysis_flag_summary_v5 = (
    df_04b["사업내용분류_분석포함여부_v5"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분석포함여부_v5")
    .reset_index(name="사업수")
)

business_content_analysis_flag_summary_v5["비율"] = (
    business_content_analysis_flag_summary_v5["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v5 분석포함여부 분포]")
display(business_content_analysis_flag_summary_v5)


# ------------------------------------------------------------
# 9) v4-v5 변화 비교
# ------------------------------------------------------------

business_content_v4_v5_compare = (
    df_04b
    .groupby(["사업내용분류_대분류_v4", "사업내용분류_대분류_v5"])
    .size()
    .reset_index(name="사업수")
    .sort_values("사업수", ascending=False)
)

print("\n[사업내용분류 v4 → v5 변화 상위 60개]")
display(business_content_v4_v5_compare.head(60))


# ------------------------------------------------------------
# 10) v4-v5 기타/검토필요/복합 비교
# ------------------------------------------------------------

review_target_values = ["기타", "검토필요", "복합"]

def count_business_content_review_group(df, col, version):
    return pd.DataFrame({
        "버전": version,
        "구분": ["기타", "검토필요", "복합", "합계"],
        "사업수": [
            int((df[col] == "기타").sum()),
            int((df[col] == "검토필요").sum()),
            int((df[col] == "복합").sum()),
            int(df[col].isin(review_target_values).sum()),
        ]
    })

business_content_review_group_compare_v4_v5 = pd.concat(
    [
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v4", "v4"),
        count_business_content_review_group(df_04b, "사업내용분류_대분류_v5", "v5"),
    ],
    axis=0
)

business_content_review_group_compare_v4_v5["비율"] = (
    business_content_review_group_compare_v4_v5["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v4-v5 기타/검토필요/복합 비교]")
display(business_content_review_group_compare_v4_v5)


# ------------------------------------------------------------
# 11) v5 분류근거별 행 수
# ------------------------------------------------------------

business_content_reason_summary_v5 = (
    df_04b["사업내용분류_분류근거_v5"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분류근거_v5")
    .reset_index(name="사업수")
)

business_content_reason_summary_v5["비율"] = (
    business_content_reason_summary_v5["사업수"] / len(df_04b) * 100
).round(2)

print("\n[사업내용분류 v5 분류근거별 행 수 상위 40개]")
display(business_content_reason_summary_v5.head(40))


# ------------------------------------------------------------
# 12) v4-v5 변경 사례
# ------------------------------------------------------------

business_content_changed_cases_v4_v5 = df_04b[
    df_04b["사업내용분류_대분류_v4"] != df_04b["사업내용분류_대분류_v5"]
].copy()

changed_case_cols_v4_v5 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "사업내용분류_대분류_v4",
    "사업내용분류_분류근거_v4",
    "사업내용분류_대분류_v5",
    "사업내용분류_분류근거_v5",
    "평가결과",
    "평가종류",
]

changed_case_cols_v4_v5 = [
    col for col in changed_case_cols_v4_v5
    if col in business_content_changed_cases_v4_v5.columns
]

print(f"\n[v4-v5 대분류 변경 사례 수] {len(business_content_changed_cases_v4_v5):,}행")
display(business_content_changed_cases_v4_v5[changed_case_cols_v4_v5].head(120))


# ------------------------------------------------------------
# 13) 핵심 보정 패턴 점검
# ------------------------------------------------------------

# 13-1. 지역사회보장협의체
social_security_council_check_v5 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "지역사회보장|사회보장협의체|지역사회보장협의체",
        regex=True,
        na=False
    )
].copy()

# 13-2. 체육 + 학교/교실
sports_education_check_v5 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "체육|생활체육|스포츠|레포츠|축구|야구|농구|배구|탁구|배드민턴|골프|게이트볼|파크골프|태권도|테니스|볼링|검도|궁도|마라톤|운동부|탁구교실|축구대회",
        regex=True,
        na=False
    )
].copy()

# 13-3. 문화+교육 혼합 표현
culture_education_check_v5 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "문화학교|문화교실|문화예술교육|예술교육|예술학교|국악교실|미술교실|음악교실",
        regex=True,
        na=False
    )
].copy()

# 13-4. 통일/안보/평화교육
unification_security_check_v5 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "통일|안보|민주평통|평통|민주시민교육|평화교육|통일교육|안보교육|안보견학|평화학교",
        regex=True,
        na=False
    )
].copy()

# 13-5. 청소년+문화/예술/체육 행사
youth_culture_check_v5 = df_04b[
    df_04b["사업명_clean_spacefix"].astype(str).str.contains(
        "청소년.*(문화|축제|예술|체육|스포츠|공연|음악)|청년.*(문화|축제|예술|체육|스포츠|공연|음악)",
        regex=True,
        na=False
    )
].copy()

check_cols_v5 = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업내용분류_대분류_v4",
    "사업내용분류_대분류_v5",
    "사업내용분류_분류근거_v5",
    "평가결과",
    "평가종류",
]

check_cols_v5 = [col for col in check_cols_v5 if col in df_04b.columns]

print("\n[지역사회보장협의체 v5 점검]")
display(social_security_council_check_v5[check_cols_v5].head(80))

print("\n[체육+학교/교실 v5 점검]")
display(sports_education_check_v5[check_cols_v5].head(80))

print("\n[문화+교육 혼합 표현 v5 점검]")
display(culture_education_check_v5[check_cols_v5].head(80))

print("\n[통일/안보/평화교육 v5 점검]")
display(unification_security_check_v5[check_cols_v5].head(80))

print("\n[청소년+문화/예술/체육 행사 v5 점검]")
display(youth_culture_check_v5[check_cols_v5].head(80))


# ------------------------------------------------------------
# 14) 저장
# ------------------------------------------------------------

rule_summary_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_rule_summary_v5_{RUN_TS}.xlsx"
)

classification_summary_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_classification_summary_v5_{RUN_TS}.xlsx"
)

v4_v5_compare_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_v4_v5_compare_{RUN_TS}.xlsx"
)

changed_cases_v4_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_changed_cases_v4_v5_{RUN_TS}.xlsx"
)

reason_summary_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_reason_summary_v5_{RUN_TS}.xlsx"
)

social_security_council_check_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_social_security_council_check_v5_{RUN_TS}.xlsx"
)

sports_education_check_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_sports_education_check_v5_{RUN_TS}.xlsx"
)

culture_education_check_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_culture_education_check_v5_{RUN_TS}.xlsx"
)

unification_security_check_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_unification_security_check_v5_{RUN_TS}.xlsx"
)

youth_culture_check_v5_path = (
    DIAGNOSTICS_DIR
    / f"04B_business_content_youth_culture_check_v5_{RUN_TS}.xlsx"
)

business_content_rule_summary_v5.to_excel(rule_summary_v5_path, index=False)

with pd.ExcelWriter(classification_summary_v5_path, engine="openpyxl") as writer:
    business_content_classification_summary_v5.to_excel(
        writer,
        sheet_name="classification_summary_v5",
        index=False
    )
    business_content_analysis_flag_summary_v5.to_excel(
        writer,
        sheet_name="analysis_flag_summary_v5",
        index=False
    )
    business_content_review_group_compare_v4_v5.to_excel(
        writer,
        sheet_name="review_group_compare",
        index=False
    )

business_content_v4_v5_compare.to_excel(v4_v5_compare_path, index=False)
convert_df_for_excel(business_content_changed_cases_v4_v5[changed_case_cols_v4_v5]).to_excel(changed_cases_v4_v5_path, index=False)
business_content_reason_summary_v5.to_excel(reason_summary_v5_path, index=False)

convert_df_for_excel(social_security_council_check_v5[check_cols_v5]).to_excel(social_security_council_check_v5_path, index=False)
convert_df_for_excel(sports_education_check_v5[check_cols_v5]).to_excel(sports_education_check_v5_path, index=False)
convert_df_for_excel(culture_education_check_v5[check_cols_v5]).to_excel(culture_education_check_v5_path, index=False)
convert_df_for_excel(unification_security_check_v5[check_cols_v5]).to_excel(unification_security_check_v5_path, index=False)
convert_df_for_excel(youth_culture_check_v5[check_cols_v5]).to_excel(youth_culture_check_v5_path, index=False)

print("\n[저장 완료]")
print(f"- v5 규칙 요약표: {rule_summary_v5_path}")
print(f"- v5 대분류/분석포함/검토그룹 요약: {classification_summary_v5_path}")
print(f"- v4-v5 대분류 변화 비교: {v4_v5_compare_path}")
print(f"- v4-v5 변경 사례: {changed_cases_v4_v5_path}")
print(f"- v5 분류근거별 행 수: {reason_summary_v5_path}")
print(f"- 지역사회보장협의체 점검: {social_security_council_check_v5_path}")
print(f"- 체육+학교/교실 점검: {sports_education_check_v5_path}")
print(f"- 문화+교육 혼합 표현 점검: {culture_education_check_v5_path}")
print(f"- 통일/안보/평화교육 점검: {unification_security_check_v5_path}")
print(f"- 청소년+문화/예술/체육 행사 점검: {youth_culture_check_v5_path}")

print("\n✅ 04B-9 v4 과보정 점검 기반 v5 최종 후보 생성 완료")

✅ 04B-9 실행 전 필수 객체 및 열 확인 완료
✅ v5 규칙 생성 완료
규칙 버전: business_content_rules_v5_final_candidate_fix_education_overcapture

[사업내용분류 v5 규칙 요약표]


,사업내용분류_대분류,strong_keywords,weak_keywords,strong_keyword_count,weak_keyword_count,note,rule_id
0,행정/통일/외교,"행정, 통일, 민주평통, 평화, 민주, 인권, 주민자치, 자치, 이장, 새마을, 바...",,16,0,행정·시민단체·통일·평화·외교 관련 사업,business_content_rules_v5_final_candidate_fix_...
1,안전 보장,"안전, 재난, 방재, 소방, 방범, 범죄, 피해자, 구급, 응급, 교통질서, 모범운...","예방, 보호, 순찰, 감시, 질서",13,5,재난·방범·범죄피해·응급·안전 관련 사업,business_content_rules_v5_final_candidate_fix_...
2,교육 보장,"교육, 학교, 대학, 인재, 장학, 평생교육, 학습, 진로, 독서, 도서관, 문해,...","역량, 연수, 프로그램, 강좌, 강사",13,5,교육·학습·장학·학교·도서관 관련 사업,business_content_rules_v5_final_candidate_fix_...
3,문화활동,"문화, 예술, 공연, 전시, 국악, 농악, 미술, 음악, 문학, 콘텐츠, 문화재, ...","축전, 한마당, 심포지엄",53,3,문화·예술·전통문화·공연·전시 관련 사업,business_content_rules_v5_final_candidate_fix_...
4,관광/휴양활동,"관광, 휴양, 축제, 여행, 탐방, 투어, 체험관광, 관광객, 관광지, 둘레길, 캠...","체험, 홍보, 마케팅, 방문, 코스",13,5,관광·휴양·축제·방문객 유치 관련 사업,business_content_rules_v5_final_candidate_fix_...
5,종교활동,"종교, 불교, 기독교, 천주교, 교회, 성당, 사찰, 절, 법회, 목사, 스님, 신앙","예배, 기도, 신도",12,3,종교 단체·종교행사 관련 사업,business_content_rules_v5_final_candidate_fix_...
6,환경향상,"환경, 생태, 기후, 탄소, 쓰레기, 폐기물, 재활용, 하천, 수질, 숲, 녹지, ...","자연, 보전, 개선, 정비",16,4,환경보전·생태·폐기물·자원순환 관련 사업,business_content_rules_v5_final_candidate_fix_...
7,사회복지향상,"복지, 복지관, 장애인, 노인, 노인회, 어르신, 아동, 청소년, 여성, 가족, 다...","종사자, 위문, 나눔",29,3,사회복지 대상자·돌봄·보육·취약계층 관련 사업,business_content_rules_v5_final_candidate_fix_...
8,보훈향상,"보훈, 유공자, 참전, 고엽제, 미망인, 수훈자, 군인회, 유족회, 전우회, 상이군...","위문, 추모, 기념",14,3,보훈·참전·유공자·유족 관련 사업,business_content_rules_v5_final_candidate_fix_...
9,고용안정,"고용, 일자리, 취업, 창업, 근로자, 노동, 청년창업, 직업, 훈련, 인력양성",청년,10,1,고용·취업·일자리·창업 관련 사업,business_content_rules_v5_final_candidate_fix_...


✅ v5 보조 함수 정의 완료
✅ 사업내용분류 v5 분류 함수 정의 완료
✅ 사업내용분류 v5 적용 완료
생성된 열: ['사업내용분류_대분류_v5', '사업내용분류_세분류_v5', '사업내용분류_후보분야_v5', '사업내용분류_분류근거_v5', '사업내용분류_검토필요여부_v5', '사업내용분류_분석포함여부_v5', '사업내용분류_rule_id_v5', '사업내용분류_강키워드_v5', '사업내용분류_약키워드_v5']

[사업내용분류 v5 대분류 분포]


,사업내용분류_대분류_v5,사업수,비율
0,사회복지향상,16777,21.17
1,문화활동,14632,18.46
2,기타,13323,16.81
3,1차 산업지원,8182,10.32
4,검토필요,5212,6.58
5,행정/통일/외교,4319,5.45
6,교육 보장,3367,4.25
7,보훈향상,2898,3.66
8,안전 보장,1891,2.39
9,산업/에너지지원,1558,1.97



[사업내용분류 v5 분석포함여부 분포]


...


[통일/안보/평화교육 v5 점검]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v4,사업내용분류_대분류_v5,사업내용분류_분류근거_v5,평가결과,평가종류
35,35,경남_남해군.xlsx,경남,남해군,안보의식 고취 및 교육,교육 보장,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,보통,유지필요성평가
110,110,경남_남해군.xlsx,경남,남해군,평화통일 홍보활동,행정/통일/외교,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,우수,유지필요성평가
112,112,경남_남해군.xlsx,경남,남해군,민주평통자문회의남해군협의회 운영비 지원,행정/통일/외교,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,우수,유지필요성평가
270,270,경남_산청군.xlsx,경남,산청군,자유총연맹 안보현장견학 및 교육,교육 보장,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,우수,성과평가
332,332,경남_산청군.xlsx,경남,산청군,안보현장 견학사업,기타,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,보통,성과평가
...,...,...,...,...,...,...,...,...,...,...
5284,5284,경남_함안군.xlsx,경남,함안군,통일현장 체험학습,교육 보장,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,미흡,성과평가/유지필요성평가
5285,5285,경남_함안군.xlsx,경남,함안군,민주평통함안군협의회,행정/통일/외교,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,우수,성과평가/유지필요성평가
5297,5297,경남_함안군.xlsx,경남,함안군,청소년 및 향군회원 안보교육,사회복지향상,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,보통,성과평가/유지필요성평가
5299,5299,경남_함안군.xlsx,경남,함안군,한국전쟁 경찰 승전기념 안보결의 및 환경정화활동,복합,행정/통일/외교,우선규칙: 통일·안보·평화·민주시민교육 관련 키워드,매우 미흡,성과평가/유지필요성평가



[청소년+문화/예술/체육 행사 v5 점검]


,v1_row_id,원파일명,시도,지자체명,사업명,사업내용분류_대분류_v4,사업내용분류_대분류_v5,사업내용분류_분류근거_v5,평가결과,평가종류
216,216,경남_남해군.xlsx,경남,남해군,장애 청소년 체육활동 지원,사회복지향상,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,성과평가
605,605,경남_진주시.xlsx,경남,진주시,청소년 평화 음악회 개최,사회복지향상,문화활동,우선규칙: 청소년 문화·예술·축제·체육 행사 맥락,보통,유지필요성평가
803,803,경남_진주시.xlsx,경남,진주시,"청소년, 형평 음악회 지원",사회복지향상,문화활동,우선규칙: 청소년 문화·예술·축제·체육 행사 맥락,매우 미흡,성과평가
854,854,경남_진주시.xlsx,경남,진주시,이야기가 있는 청소년 클래식 음악회,사회복지향상,문화활동,우선규칙: 청소년 문화·예술·축제·체육 행사 맥락,보통,성과평가
1306,1306,경남_양산시.xlsx,경남,양산시,청소년 체육교실,사회복지향상,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,보통,성과평가
...,...,...,...,...,...,...,...,...,...,...
21984,21998,울산_남구.xlsx,울산,남구,청소년 체육행사 지원,사회복지향상,문화활동,우선규칙: 기준 파일에 체육 단독 분류가 없어 체육·스포츠 관련 사업을 문화활동으로...,매우미흡,유지필요성평가
22560,22574,울산_울주군.xlsx,울산,울주군,청소년 예술제,사회복지향상,문화활동,우선규칙: 청소년 문화·예술·축제·체육 행사 맥락,보통,성과평가
22756,22770,울산_울주군.xlsx,울산,울주군,청소년 문화유적 탐방,사회복지향상,사회복지향상,우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,보통,유지필요성
22890,22904,충남_논산시.xlsx,충남,논산시,청소년 안전문화정착 캠페인,사회복지향상,사회복지향상,우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,보통,성과평가



[저장 완료]
- v5 규칙 요약표: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_rule_summary_v5_20260606_1912.xlsx
- v5 대분류/분석포함/검토그룹 요약: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_classification_summary_v5_20260606_1912.xlsx
- v4-v5 대분류 변화 비교: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_v4_v5_compare_20260606_1912.xlsx
- v4-v5 변경 사례: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_business_content_changed_cases_v4_v5_20260606_1912.xlsx
- v5 분류근거별 행 수: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/diagnostics/04B_busin

04B-10. v5 최종 확정 및 최종 산출물 저장
1. 이 단계의 목적

이 단계의 목적은 사업내용분류_*_v5 열을 접미사 없는 최종 열로 복사하고, 05번 분석에 바로 사용할 수 있는 handoff 파일을 저장하는 것입니다.

즉, 아래처럼 확정합니다.

---
사업내용분류_대분류 = 사업내용분류_대분류_v5
사업내용분류_세분류 = 사업내용분류_세분류_v5
사업내용분류_후보분야 = 사업내용분류_후보분야_v5
사업내용분류_분류근거 = 사업내용분류_분류근거_v5
사업내용분류_검토필요여부 = 사업내용분류_검토필요여부_v5
사업내용분류_분석포함여부 = 사업내용분류_분석포함여부_v5
사업내용분류_rule_id = 사업내용분류_rule_id_v5


In [13]:
# 04B-10. v5 최종 확정 및 master/handoff/summary/review 파일 저장

# ------------------------------------------------------------
# 0) 실행 전 필수 객체 및 열 확인
# ------------------------------------------------------------

required_objects = [
    "df_04b",
    "FINAL_DIR",
    "DIAGNOSTICS_DIR",
    "RUN_TS",
    "convert_df_for_excel",
    "safe_parse_tokens",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"다음 객체가 없습니다: {missing_objects}\n"
        "04B-1부터 04B-9까지 순서대로 실행했는지 확인하세요."
    )

required_cols_04b_10 = [
    # 식별·병합용
    "v1_row_id",
    "시도",
    "지자체명",
    "원파일명",
    "사업명",
    "보조사업자",
    "평가결과",
    "평가종류",

    # 참고용
    "사업명_clean",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "tokens_analysis_second_filter",

    # v5 최종 후보
    "사업내용분류_대분류_v5",
    "사업내용분류_세분류_v5",
    "사업내용분류_후보분야_v5",
    "사업내용분류_분류근거_v5",
    "사업내용분류_검토필요여부_v5",
    "사업내용분류_분석포함여부_v5",
    "사업내용분류_rule_id_v5",
]

missing_cols_04b_10 = [
    col for col in required_cols_04b_10
    if col not in df_04b.columns
]

if missing_cols_04b_10:
    raise KeyError(
        f"04B-10 실행에 필요한 열이 없습니다: {missing_cols_04b_10}\n"
        "04B-9가 정상 완료되었는지 확인하세요."
    )

FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("✅ 04B-10 실행 전 필수 객체 및 열 확인 완료")
print(f"FINAL_DIR: {FINAL_DIR}")


# ------------------------------------------------------------
# 1) v5 열을 접미사 없는 최종 열로 복사
# ------------------------------------------------------------

final_col_map = {
    "사업내용분류_대분류_v5": "사업내용분류_대분류",
    "사업내용분류_세분류_v5": "사업내용분류_세분류",
    "사업내용분류_후보분야_v5": "사업내용분류_후보분야",
    "사업내용분류_분류근거_v5": "사업내용분류_분류근거",
    "사업내용분류_검토필요여부_v5": "사업내용분류_검토필요여부",
    "사업내용분류_분석포함여부_v5": "사업내용분류_분석포함여부",
    "사업내용분류_rule_id_v5": "사업내용분류_rule_id",
}

for src_col, dst_col in final_col_map.items():
    df_04b[dst_col] = df_04b[src_col]

df_04b["사업내용분류_최종버전"] = "v5"

print("✅ v5 열을 최종 사업내용분류 열로 복사 완료")


# ------------------------------------------------------------
# 2) 최종 열 품질 점검
# ------------------------------------------------------------

final_check_summary = pd.DataFrame({
    "항목": [
        "전체 행 수",
        "전체 열 수",
        "v1_row_id 결측 수",
        "v1_row_id 중복 수",
        "원파일명 결측 수",
        "사업명 결측 수",
        "사업내용분류_대분류 결측 수",
        "사업내용분류_분석포함여부 결측 수",
        "사업내용분류_rule_id 결측 수",
    ],
    "값": [
        len(df_04b),
        df_04b.shape[1],
        df_04b["v1_row_id"].isna().sum(),
        df_04b["v1_row_id"].duplicated().sum(),
        df_04b["원파일명"].isna().sum(),
        df_04b["사업명"].isna().sum(),
        df_04b["사업내용분류_대분류"].isna().sum(),
        df_04b["사업내용분류_분석포함여부"].isna().sum(),
        df_04b["사업내용분류_rule_id"].isna().sum(),
    ]
})

print("\n[최종 열 품질 점검]")
display(final_check_summary)


# ------------------------------------------------------------
# 3) 최종 대분류/분석포함여부/검토그룹 요약 생성
# ------------------------------------------------------------

business_content_final_category_summary = (
    df_04b["사업내용분류_대분류"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_대분류")
    .reset_index(name="사업수")
)

business_content_final_category_summary["비율"] = (
    business_content_final_category_summary["사업수"] / len(df_04b) * 100
).round(2)


business_content_final_analysis_flag_summary = (
    df_04b["사업내용분류_분석포함여부"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분석포함여부")
    .reset_index(name="사업수")
)

business_content_final_analysis_flag_summary["비율"] = (
    business_content_final_analysis_flag_summary["사업수"] / len(df_04b) * 100
).round(2)


review_target_values = ["기타", "검토필요", "복합"]

business_content_final_review_group_summary = pd.DataFrame({
    "구분": ["기타", "검토필요", "복합", "합계"],
    "사업수": [
        int((df_04b["사업내용분류_대분류"] == "기타").sum()),
        int((df_04b["사업내용분류_대분류"] == "검토필요").sum()),
        int((df_04b["사업내용분류_대분류"] == "복합").sum()),
        int(df_04b["사업내용분류_대분류"].isin(review_target_values).sum()),
    ]
})

business_content_final_review_group_summary["비율"] = (
    business_content_final_review_group_summary["사업수"] / len(df_04b) * 100
).round(2)


business_content_final_rule_id_summary = (
    df_04b["사업내용분류_rule_id"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_rule_id")
    .reset_index(name="사업수")
)

business_content_final_rule_id_summary["비율"] = (
    business_content_final_rule_id_summary["사업수"] / len(df_04b) * 100
).round(2)


business_content_final_reason_summary = (
    df_04b["사업내용분류_분류근거"]
    .value_counts(dropna=False)
    .rename_axis("사업내용분류_분류근거")
    .reset_index(name="사업수")
)

business_content_final_reason_summary["비율"] = (
    business_content_final_reason_summary["사업수"] / len(df_04b) * 100
).round(2)

print("\n[최종 사업내용분류 대분류 분포]")
display(business_content_final_category_summary)

print("\n[최종 사업내용분류 분석포함여부 분포]")
display(business_content_final_analysis_flag_summary)

print("\n[최종 기타/검토필요/복합 요약]")
display(business_content_final_review_group_summary)

print("\n[최종 rule_id별 행 수]")
display(business_content_final_rule_id_summary)

print("\n[최종 분류근거별 행 수 상위 40개]")
display(business_content_final_reason_summary.head(40))


# ------------------------------------------------------------
# 4) 검토용 사례 생성
# ------------------------------------------------------------

review_case_cols_final = [
    "v1_row_id",
    "원파일명",
    "시도",
    "지자체명",
    "사업명",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "보조사업자",
    "평가결과",
    "평가종류",
    "사업내용분류_대분류",
    "사업내용분류_세분류",
    "사업내용분류_후보분야",
    "사업내용분류_분류근거",
    "사업내용분류_검토필요여부",
    "사업내용분류_분석포함여부",
    "사업내용분류_rule_id",
    "사업내용분류_최종버전",
]

review_case_cols_final = [
    col for col in review_case_cols_final
    if col in df_04b.columns
]

business_content_final_review_cases = df_04b[
    (df_04b["사업내용분류_검토필요여부"] == True)
    | (df_04b["사업내용분류_대분류"].isin(review_target_values))
].copy()

print(f"\n[최종 검토필요·기타·복합 사례 수] {len(business_content_final_review_cases):,}행")
display(business_content_final_review_cases[review_case_cols_final].head(100))


# ------------------------------------------------------------
# 5) rule_id별 대표 사업명 생성
# ------------------------------------------------------------

rule_sample_rows = []

for rule_id, temp in df_04b.groupby("사업내용분류_rule_id", dropna=False):
    temp_sample = temp[review_case_cols_final].head(30).copy()
    temp_sample.insert(0, "sample_group", f"rule_id={rule_id}")
    rule_sample_rows.append(temp_sample)

if rule_sample_rows:
    business_content_final_rule_samples = pd.concat(rule_sample_rows, axis=0)
else:
    business_content_final_rule_samples = pd.DataFrame(columns=["sample_group"] + review_case_cols_final)


# ------------------------------------------------------------
# 6) 분류근거별 대표 사업명 생성
# ------------------------------------------------------------

reason_sample_rows = []

top_reasons = business_content_final_reason_summary["사업내용분류_분류근거"].head(50).tolist()

for reason in top_reasons:
    temp = df_04b[df_04b["사업내용분류_분류근거"] == reason].copy()
    temp_sample = temp[review_case_cols_final].head(20).copy()
    temp_sample.insert(0, "sample_group", f"reason={reason}")
    reason_sample_rows.append(temp_sample)

if reason_sample_rows:
    business_content_final_reason_samples = pd.concat(reason_sample_rows, axis=0)
else:
    business_content_final_reason_samples = pd.DataFrame(columns=["sample_group"] + review_case_cols_final)


# ------------------------------------------------------------
# 7) 05번 handoff 파일 생성
# ------------------------------------------------------------
# 한 행 = 한 사업
# 05번 병합과 분석에 필요한 식별자/원자료/최종 사업내용분류 열만 포함한다.

handoff_cols = [
    # 식별·병합용 열
    "v1_row_id",
    "시도",
    "지자체명",
    "원파일명",
    "사업명",
    "보조사업자",
    "평가결과",
    "평가종류",

    # 참고용 사업명 정제/토큰 열
    "사업명_clean",
    "사업명_clean_spacefix",
    "tokens_analysis_primary",
    "tokens_analysis_second_filter",

    # 최종 분석용 사업내용분류 변수
    "사업내용분류_대분류",
    "사업내용분류_세분류",
    "사업내용분류_후보분야",
    "사업내용분류_분류근거",
    "사업내용분류_검토필요여부",
    "사업내용분류_분석포함여부",
    "사업내용분류_rule_id",
    "사업내용분류_최종버전",
]

handoff_cols = [col for col in handoff_cols if col in df_04b.columns]

df_04b_handoff = df_04b[handoff_cols].copy()

print("\n[handoff 파일 기본 구조]")
print(f"df_04b_handoff shape: {df_04b_handoff.shape}")
display(df_04b_handoff.head(10))


# ------------------------------------------------------------
# 8) parquet 저장용 복사본 생성
# ------------------------------------------------------------
# parquet 저장 시 numpy.ndarray가 섞여 있으면 오류가 날 수 있으므로
# 토큰 열은 list[str] 형태로 변환한다.

def convert_for_parquet_safe(df):
    out = df.copy()

    token_like_cols = [
        col for col in out.columns
        if "tokens" in col
    ]

    for col in token_like_cols:
        out[col] = out[col].apply(safe_parse_tokens)

    return out

df_04b_master_for_parquet = convert_for_parquet_safe(df_04b)
df_04b_handoff_for_parquet = convert_for_parquet_safe(df_04b_handoff)


# ------------------------------------------------------------
# 9) 최종 파일 경로 설정
# ------------------------------------------------------------

master_parquet_path = (
    FINAL_DIR
    / f"04B_business_content_classification_master_v1_{RUN_TS}.parquet"
)

handoff_parquet_path = (
    FINAL_DIR
    / f"04B_business_content_classification_handoff_v1_{RUN_TS}.parquet"
)

handoff_excel_path = (
    FINAL_DIR
    / f"04B_business_content_classification_handoff_v1_{RUN_TS}.xlsx"
)

summary_excel_path = (
    FINAL_DIR
    / f"04B_business_content_classification_summary_v1_{RUN_TS}.xlsx"
)

review_cases_excel_path = (
    FINAL_DIR
    / f"04B_business_content_classification_review_cases_v1_{RUN_TS}.xlsx"
)


# ------------------------------------------------------------
# 10) 최종 파일 저장
# ------------------------------------------------------------

# master: 전체 df_04b 보존용
df_04b_master_for_parquet.to_parquet(master_parquet_path, index=False)

# handoff: 05번 전달용
df_04b_handoff_for_parquet.to_parquet(handoff_parquet_path, index=False)

# handoff Excel: 사람이 확인하기 쉬운 버전
convert_df_for_excel(df_04b_handoff).to_excel(
    handoff_excel_path,
    index=False
)

# summary Excel
with pd.ExcelWriter(summary_excel_path, engine="openpyxl") as writer:
    final_check_summary.to_excel(
        writer,
        sheet_name="quality_check",
        index=False
    )
    business_content_final_category_summary.to_excel(
        writer,
        sheet_name="category_summary",
        index=False
    )
    business_content_final_analysis_flag_summary.to_excel(
        writer,
        sheet_name="analysis_flag_summary",
        index=False
    )
    business_content_final_review_group_summary.to_excel(
        writer,
        sheet_name="review_group_summary",
        index=False
    )
    business_content_final_rule_id_summary.to_excel(
        writer,
        sheet_name="rule_id_summary",
        index=False
    )
    business_content_final_reason_summary.to_excel(
        writer,
        sheet_name="reason_summary",
        index=False
    )

# review cases Excel
with pd.ExcelWriter(review_cases_excel_path, engine="openpyxl") as writer:
    convert_df_for_excel(
        business_content_final_review_cases[review_case_cols_final]
    ).to_excel(
        writer,
        sheet_name="review_cases",
        index=False
    )

    convert_df_for_excel(
        business_content_final_rule_samples
    ).to_excel(
        writer,
        sheet_name="rule_id_samples",
        index=False
    )

    convert_df_for_excel(
        business_content_final_reason_samples
    ).to_excel(
        writer,
        sheet_name="reason_samples",
        index=False
    )

print("\n[최종 저장 완료]")
print(f"- master parquet: {master_parquet_path}")
print(f"- handoff parquet: {handoff_parquet_path}")
print(f"- handoff Excel: {handoff_excel_path}")
print(f"- summary Excel: {summary_excel_path}")
print(f"- review cases Excel: {review_cases_excel_path}")


# ------------------------------------------------------------
# 11) 저장 파일 존재 여부 및 최종 출력 확인
# ------------------------------------------------------------

saved_files = [
    master_parquet_path,
    handoff_parquet_path,
    handoff_excel_path,
    summary_excel_path,
    review_cases_excel_path,
]

saved_file_check = pd.DataFrame({
    "파일": [p.name for p in saved_files],
    "경로": [str(p) for p in saved_files],
    "존재여부": [p.exists() for p in saved_files],
    "크기_MB": [
        round(p.stat().st_size / (1024 * 1024), 2) if p.exists() else None
        for p in saved_files
    ],
})

print("\n[저장 파일 존재 여부 확인]")
display(saved_file_check)

print("\n[04B 최종 요약]")
print(f"전체 행 수: {len(df_04b):,}")
print(f"전체 열 수(master): {df_04b.shape[1]:,}")
print(f"handoff 행 수: {len(df_04b_handoff):,}")
print(f"handoff 열 수: {df_04b_handoff.shape[1]:,}")
print(f"v1_row_id 결측 수: {df_04b['v1_row_id'].isna().sum():,}")
print(f"v1_row_id 중복 수: {df_04b['v1_row_id'].duplicated().sum():,}")
print(f"원파일명 결측 수: {df_04b['원파일명'].isna().sum():,}")

기타_n = int((df_04b["사업내용분류_대분류"] == "기타").sum())
검토필요_n = int((df_04b["사업내용분류_대분류"] == "검토필요").sum())
복합_n = int((df_04b["사업내용분류_대분류"] == "복합").sum())
review_total_n = int(df_04b["사업내용분류_대분류"].isin(review_target_values).sum())

print(f"기타 행 수: {기타_n:,} ({기타_n / len(df_04b) * 100:.2f}%)")
print(f"검토필요 행 수: {검토필요_n:,} ({검토필요_n / len(df_04b) * 100:.2f}%)")
print(f"복합 행 수: {복합_n:,} ({복합_n / len(df_04b) * 100:.2f}%)")
print(f"기타+검토필요+복합 합계: {review_total_n:,} ({review_total_n / len(df_04b) * 100:.2f}%)")

print("\n✅ 04B-10 v5 최종 확정 및 최종 산출물 저장 완료")

✅ 04B-10 실행 전 필수 객체 및 열 확인 완료
FINAL_DIR: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/final
✅ v5 열을 최종 사업내용분류 열로 복사 완료

[최종 열 품질 점검]


,항목,값
0,전체 행 수,79262
1,전체 열 수,96
2,v1_row_id 결측 수,0
3,v1_row_id 중복 수,0
4,원파일명 결측 수,0
5,사업명 결측 수,0
6,사업내용분류_대분류 결측 수,0
7,사업내용분류_분석포함여부 결측 수,0
8,사업내용분류_rule_id 결측 수,0



[최종 사업내용분류 대분류 분포]


,사업내용분류_대분류,사업수,비율
0,사회복지향상,16777,21.17
1,문화활동,14632,18.46
2,기타,13323,16.81
3,1차 산업지원,8182,10.32
4,검토필요,5212,6.58
5,행정/통일/외교,4319,5.45
6,교육 보장,3367,4.25
7,보훈향상,2898,3.66
8,안전 보장,1891,2.39
9,산업/에너지지원,1558,1.97



[최종 사업내용분류 분석포함여부 분포]


...


[handoff 파일 기본 구조]
df_04b_handoff shape: (79262, 20)


,v1_row_id,시도,지자체명,원파일명,사업명,보조사업자,평가결과,평가종류,사업명_clean,사업명_clean_spacefix,tokens_analysis_primary,tokens_analysis_second_filter,사업내용분류_대분류,사업내용분류_세분류,사업내용분류_후보분야,사업내용분류_분류근거,사업내용분류_검토필요여부,사업내용분류_분석포함여부,사업내용분류_rule_id,사업내용분류_최종버전
0,0,경남,남해군,경남_남해군.xlsx,마늘재배 영농지원단 지원,농협중앙회 남해군지부,매우우수,유지필요성평가,마늘재배 영농지원단 지원,마늘재배 영농지원단 지원,"[마늘, 재배, 영농, 지원단]","[마늘, 재배, 영농, 지원단]",1차 산업지원,,1차 산업지원,우선규칙: 농림축산·수산·임업 등 1차 산업 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
1,1,경남,남해군,경남_남해군.xlsx,2023년 사)진주지역범죄피해자지원센터‘등불 ’남해지부 운영,사단법인진주지역범죄피 해자지원센터남해지부,미흡,유지필요성평가,2023년 사)진주지역범죄피해자지원센터'등불 '남해지부 운영,2023년 사)진주지역범죄피해자지원센터'등불 '남해지부 운영,"[지역, 범죄, 피해자, 센터, 등불, 지부]","[지역, 범죄, 피해자, 센터, 등불, 지부]",안전 보장,,안전 보장,우선규칙: 재난·방범·범죄피해·응급·안전 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
2,2,경남,남해군,경남_남해군.xlsx,자유민주주의신장 및 자유선진의식 고취,한국자유총연맹남해군지 회,미흡,유지필요성평가,자유민주주의신장 및 자유선진의식 고취,자유민주주의신장 및 자유선진의식 고취,"[자유, 민주주의, 신장, 자유선진, 의식, 고취]","[자유, 민주주의, 신장, 자유선진, 의식, 고취]",행정/통일/외교,,행정/통일/외교,우선규칙: 행정·시민단체·통일·평화·외교 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
3,3,경남,남해군,경남_남해군.xlsx,보물섬 독서학교 지원,보물섬남해독서학교,미흡,유지필요성평가,보물섬 독서학교 지원,보물섬 독서학교 지원,"[보물섬, 독서, 학교]","[보물섬, 독서, 학교]",교육 보장,,교육 보장,우선규칙: 교육·학습·장학·학교·도서관 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
4,4,경남,남해군,경남_남해군.xlsx,참전경찰유공자 보훈활동,한국참전경찰유공자회,미흡,유지필요성평가,참전경찰유공자 보훈활동,참전경찰유공자 보훈활동,"[참전, 경찰, 유공자, 보훈, 활동]","[참전, 경찰, 유공자, 보훈, 활동]",보훈향상,,보훈향상,우선규칙: 보훈·참전·유공자·유족 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
5,5,경남,남해군,경남_남해군.xlsx,무공수훈자회 운영비 보조,대한민국무공수훈자회남 해 군 지 회,미흡,유지필요성평가,무공수훈자회 운영비 보조,무공수훈자회 운영비 보조,"[무공, 수훈자]","[무공, 수훈자]",보훈향상,,보훈향상,우선규칙: 보훈·참전·유공자·유족 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
6,6,경남,남해군,경남_남해군.xlsx,행복나눔센터 운영,남해군지역사회보장협의 체,미흡,유지필요성평가,행복나눔센터 운영,행복나눔센터 운영,"[행복, 센터]","[행복, 센터]",사회복지향상,,사회복지향상,우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
7,7,경남,남해군,경남_남해군.xlsx,경로당 양곡구입비 지원,봉내경로당 외 10개소,미흡,유지필요성평가,경로당 양곡구입비 지원,경로당 양곡구입비 지원,"[경로당, 양곡, 구입비]","[경로당, 양곡, 구입비]",사회복지향상,,사회복지향상,우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
8,8,경남,남해군,경남_남해군.xlsx,장애인직업재활 자활교육,남해장애인종합복지관,미흡,유지필요성평가,장애인직업재활 자활교육,장애인직업재활 자활교육,"[장애인, 직업, 재활, 자활, 교육]","[장애인, 직업, 재활, 자활, 교육]",사회복지향상,,"고용안정, 교육 보장, 사회복지향상",우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5
9,9,경남,남해군,경남_남해군.xlsx,어린이집 운전기사 인건비 (미조지역영유아수송),모 모 어 린 이 집,미흡,유지필요성평가,어린이집 운전기사 인건비 (미조지역영유아수송),어린이집 운전기사 인건비 (미조지역영유아수송),"[어린이집, 운전기사, 미조, 지역, 영유아, 수송]","[어린이집, 운전기사, 미조, 지역, 영유아, 수송]",사회복지향상,,"교통/물류진흥, 사회복지향상, 산업/에너지지원",우선규칙: 복지 대상자·돌봄·보육·취약계층 관련 키워드,False,주분석_포함,business_content_rules_v5_final_candidate_fix_...,v5



[최종 저장 완료]
- master parquet: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/final/04B_business_content_classification_master_v1_20260606_1912.parquet
- handoff parquet: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/final/04B_business_content_classification_handoff_v1_20260606_1912.parquet
- handoff Excel: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/final/04B_business_content_classification_handoff_v1_20260606_1912.xlsx
- summary Excel: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/final/04B_business_content_classification_summary_v1_20260606_1912.xlsx
- review cases Excel: /content/drive/MyDrive/파공장 (PA Factory)/2026 KAPA/[guide & output] 분석 협업 매뉴얼 및 Figure 후보/03_중간결과표/04_v2_사업내용분류분석/tables/fi

,파일,경로,존재여부,크기_MB
0,04B_business_content_classification_master_v1_...,/content/drive/MyDrive/파공장 (PA Factory)/2026 K...,False,None
1,04B_business_content_classification_handoff_v1...,/content/drive/MyDrive/파공장 (PA Factory)/2026 K...,False,None
2,04B_business_content_classification_handoff_v1...,/content/drive/MyDrive/파공장 (PA Factory)/2026 K...,False,None
3,04B_business_content_classification_summary_v1...,/content/drive/MyDrive/파공장 (PA Factory)/2026 K...,False,None
4,04B_business_content_classification_review_cas...,/content/drive/MyDrive/파공장 (PA Factory)/2026 K...,False,None



[04B 최종 요약]
전체 행 수: 79,262
전체 열 수(master): 96
handoff 행 수: 79,262
handoff 열 수: 20
v1_row_id 결측 수: 0
v1_row_id 중복 수: 0
원파일명 결측 수: 0
기타 행 수: 13,323 (16.81%)
검토필요 행 수: 5,212 (6.58%)
복합 행 수: 517 (0.65%)
기타+검토필요+복합 합계: 19,052 (24.04%)

✅ 04B-10 v5 최종 확정 및 최종 산출물 저장 완료


기존 04번 v3c는 사업명 텍스트 기반 탐색적 자동분류 초안으로 보존했고,
최종 05번 분석용 정책분야 변수는 04B에서 보조금24 사업주제 기준인
사업 내용 분류.txt의 18개 분류명을 기준으로 새로 생성했다.
최종 변수명은 사업내용분류_*이며, v5를 최종 확정 버전으로 사용한다.